# TokaMark `task_1-3` Sensor-Robustness Study (Gupta arXiv 2607.11915) on Kaggle

**Goal:** Run the complete sensor-failure robustness evaluation on a *trained, frozen* task_1-3 model across 6 failure scenarios (random dropout, channel ablation, category importance, temporal gap, correlated group failure, disruption-proximate) and 3 imputation modes (zero, train-mean, forward-fill).

**Requirements on Kaggle:**
- **Accelerator:** GPU T4 (x1 or x2).
- **Internet:** ON (for downloading test shots from Hugging Face).
- **Input Data Attached:** Add the output of your training notebook (`Fusion_recon` or `run_40gb`) to `/kaggle/input/`.
- **Execution:** Click **"Save Version" -> "Save & Run All (Commit)"** to run headlessly in the background (~25-35 minutes total).


In [ ]:
import os, sys, time, shutil, subprocess, json, glob

# ================= CONFIG =================
DATA_SPLIT    = "random"  # "random" or "temporal"
MODEL_TYPE    = "lstm"    # trained failure-aware CNN+LSTM
FAILURE_AWARE = 1        # must match the trained model: 1 = Module 2 Failure-Aware + PINN checkpoint, 0 = baseline
BATCH         = 512       # batch size for test inference
SEED          = 42
DL_WORKERS    = 0         # 0 = main process (hang-proof on Kaggle)
ROB_SCENARIOS = "all"     # or comma-separated list of scenarios
ROB_IMPUTE    = "zero,mean,ffill"
# ==========================================

T0 = time.time()
os.environ["TM_T0"] = str(T0)

def run(cmd, check=True):
    print("$", cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=os.environ)
    for line in p.stdout: print(line, end="", flush=True)
    p.wait()
    if check and p.returncode: raise RuntimeError(f"command failed ({p.returncode}): {cmd}")
    return p.returncode

print("=== System Resources ===")
run("nvidia-smi -L; nproc; free -g | head -2; df -h /kaggle/working /tmp 2>/dev/null", check=False)
run("mount -o remount,size=8G /dev/shm 2>/dev/null || sudo mount -o remount,size=8G /dev/shm", check=False)
run("df -h /dev/shm", check=False)

W = "/kaggle/working"
cands = [d for d in ("/kaggle/temp", "/tmp", "/kaggle/working") if os.path.isdir(d)]
DATA_ROOT = max(cands, key=lambda d: shutil.disk_usage(d).free) + "/tokamark_data"
print(f"data root: {DATA_ROOT} | free GB: {shutil.disk_usage(os.path.dirname(DATA_ROOT)).free/1e9:.2f}")

# Install necessary runtime dependencies
run("python -m pip install -q zarr==3.1.5 xarray 'fsspec==2025.3.0' 's3fs==2025.3.0' "
    "huggingface_hub pyyaml scipy requests filelock torchinfo==1.8.0", check=True)
run("python -c \"import zarr, xarray, s3fs, huggingface_hub, torchinfo, requests, filelock; print('dependency imports OK')\"", check=True)

# ==============================================================================
# RESTORE TRAINED ARTIFACTS FROM /kaggle/input (Handles DATASETS and NOTEBOOKS)
# ==============================================================================
print("\n=== Checking /kaggle/input for trained model artifacts ===")
input_dirs = glob.glob("/kaggle/input/*")
print("Input folders detected:", [os.path.basename(d) for d in input_dirs])

restored = False

# Priority 1: Direct match in dataset or notebook folders (e.g. /kaggle/input/results/out)
for cand in input_dirs:
    cand_out = os.path.join(cand, "out")
    if os.path.isdir(cand_out):
        print(f"Found artifacts in input directory: {cand}")
        run(f"mkdir -p '{W}/out' && cp -r '{cand_out}/'* '{W}/out/'")
        for r in ("tokamark", "tokamark_baseline"):
            r_src = os.path.join(cand, r)
            if os.path.isdir(r_src) and not os.path.isdir(f"{W}/{r}"):
                print(f"Restoring repo {r} from {cand}...")
                run(f"cp -r '{r_src}' '{W}/'")
        restored = True
        break

# Priority 2: Check for any results.zip archive in /kaggle/input
if not restored:
    for root, dirs, files in os.walk("/kaggle/input"):
        for f in files:
            if f.endswith(".zip") and any(k in f.lower() for k in ("result", "fusion", "run", "40gb")):
                fpath = os.path.join(root, f)
                print(f"Extracting zip archive: {fpath} -> {W}/")
                run(f"unzip -q -o '{fpath}' -d {W}/")
                restored = True
                break
        if restored:
            break

# Priority 3: Deep recursive search for best_model.pt
if not restored:
    for root, dirs, files in os.walk("/kaggle/input"):
        if "best_model.pt" in files:
            cur = root
            while cur and cur != "/kaggle/input":
                if os.path.basename(cur) == "out":
                    print(f"Found out directory at {cur}, copying to {W}/out...")
                    run(f"mkdir -p '{W}/out' && cp -r '{cur}/'* '{W}/out/'")
                    restored = True
                    break
                cur = os.path.dirname(cur)
            if restored:
                break

# Verify and position the restored model checkpoint
expected_pt = f"{W}/out/{DATA_SPLIT}/model/best_model.pt"
if not os.path.exists(expected_pt):
    for root, dirs, files in os.walk(W):
        if "best_model.pt" in files:
            src_pt = os.path.join(root, "best_model.pt")
            run(f"mkdir -p '{W}/out/{DATA_SPLIT}/model' && cp '{src_pt}' '{expected_pt}'")
            print(f"[OK] Located and positioned model checkpoint to: {expected_pt}")
            break

if not os.path.exists(expected_pt):
    raise RuntimeError("Missing best_model.pt! Please attach your trained model results archive to /kaggle/input.")

print(f"[OK] Model successfully verified at: {expected_pt}")
expected_sel = f"{W}/out/{DATA_SPLIT}/selection.json"
if os.path.exists(expected_sel):
    print(f"[OK] Selection file verified at: {expected_sel}")
else:
    print(f"[INFO] selection.json not in archive - Step 1 (prep) will download official {DATA_SPLIT} test split automatically.")

# Ensure repos exist (clone if not copied from input)
REPOS = (("tokamark", "1a200f4"), ("tokamark_baseline", "1e7c489"))
for name, sha in REPOS:
    if not os.path.isdir(f"{W}/{name}"):
        run(f"git clone -q https://github.com/UKAEA-IBM-STFC-Fusion-FMs/{name}.git {W}/{name}")
    run(f"cd {W}/{name} && git checkout -q {sha} && git log -1 --format='{name} @ %h'")

env = dict(
    TM_SEED=SEED,
    TM_DATA_ROOT=DATA_ROOT,
    TM_WORK=W,
    TM_TOKAMARK=f"{W}/tokamark",
    TM_BASELINE=f"{W}/tokamark_baseline",
    TM_DL_WORKERS=DL_WORKERS,
    TM_GIT_REVS=",".join(f"{n}@{s}" for n, s in REPOS),
    TM_BATCH=BATCH,
    TM_MODEL=MODEL_TYPE, TM_FAILURE_AWARE=("1" if FAILURE_AWARE else "0"),
    TM_ROB_SCENARIOS=ROB_SCENARIOS,
    TM_ROB_IMPUTE=ROB_IMPUTE,
    TM_ROB_SEED=SEED,
    TM_ROB_BATCH=BATCH,
)
os.environ.update({k: str(v) for k, v in env.items()})
print("Environment configured successfully.")


In [ ]:
%%writefile /kaggle/working/pipeline.py
#!/usr/bin/env python
"""TokaMark task_1-3 CNN baseline on Kaggle: prep | train | eval | plot | summary.

Usage: python pipeline.py {prep,train,eval,plot,summary} [--split random|temporal]
All settings come from environment variables (see CFG below), set by the notebook.

Protocol notes (deviations from the official run_training.py / run_evaluation.py):
- Custom step-based training loop mirroring src/trainer.py:BatchStepTrainer (same
  Adam(lr, wd=1e-4) + MultiOutputMSELoss, validate_every, patience, drop_last=True),
  extended with: wall-clock deadline, periodic + atomic checkpointing, resume from
  last.pt, CUDA-OOM reporting, and non-finite-loss skipping.
- Shots are a random subset OF THE OFFICIAL SPLIT LISTS (membership never mixed),
  chosen under a byte budget; official stats/outlier/split files always match the
  split protocol. test uses shuffle_windows=False (official eval behaviour).
"""
import os, sys, json, time, math, random, shutil, zipfile, argparse, subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed

T0 = float(os.environ.get("TM_T0", time.time()))  # session start (set by notebook)


def env(k, d):
    return type(d)(os.environ.get(k, d))


# Tuned hyper-parameters (written by `tune`): explicit env var > TM_TUNED file > default.
_TUNED = json.load(open(os.environ["TM_TUNED"]))["chosen"] if os.environ.get("TM_TUNED") else {}


def hp(var, key, d):
    return type(d)(os.environ[var]) if var in os.environ else type(d)(_TUNED.get(key, d))


# Ablation variant: plain | mask (failure-aware mask + augmentation, no PINN) | mask_pinn (+ flux-loop loss).
# TM_VARIANT is a convenience over TM_FAILURE_AWARE / TM_PINN_LAMBDA; unset -> the legacy two variables decide.
_VAR = os.environ.get("TM_VARIANT", "").lower().strip()
if _VAR not in ("", "plain", "mask", "mask_pinn"):
    raise ValueError(f"TM_VARIANT must be plain|mask|mask_pinn, got {_VAR!r}")
_FA = (_VAR != "plain") if _VAR else env("TM_FAILURE_AWARE", "0") == "1"
_PINN = {"plain": 0.0, "mask": 0.0}.get(_VAR, env("TM_PINN_LAMBDA", 0.1))
if _VAR == "mask_pinn" and _PINN <= 0:
    raise ValueError("TM_VARIANT=mask_pinn needs TM_PINN_LAMBDA > 0")

CFG = dict(
    task=env("TM_TASK", "task_1-3"),
    model=env("TM_MODEL", "mlp"),                         # mlp (CNN+MLP) | lstm (CNN+LSTM) | transformer (one token per sensor)
    seed=env("TM_SEED", 23),                              # init, augmentation and data order (NOT the shot selection)
    select_seed=env("TM_SELECT_SEED", 23),                # shot selection only; selection.json is reused when present
    run_tag=env("TM_RUN_TAG", ""),                        # "" legacy out/<split>/; "auto" -> <model>_<variant>_s<seed>; else literal
    repo_id="UKAEA-IBM-STFC/tokamark-v1",
    work=env("TM_WORK", "/kaggle/working"),
    data_root=env("TM_DATA_ROOT", "/tmp/tokamark_data"),
    tokamark_dir=env("TM_TOKAMARK", "/kaggle/working/tokamark"),
    baseline_dir=env("TM_BASELINE", "/kaggle/working/tokamark_baseline"),
    data_budget_gb=env("TM_DATA_GB", 40.0),
    prep_deadline_h=env("TM_PREP_DEADLINE_H", 1.75),      # hours since T0: stop downloading
    train_deadline_h=env("TM_TRAIN_DEADLINE_H", 8.0),     # hours since T0: stop training
    batch_size=env("TM_BATCH", 512),
    validate_every=env("TM_VALIDATE_EVERY", 100),
    patience=env("TM_PATIENCE", 10),
    lr=hp("TM_LR", "lr", 1e-4),
    weight_decay=hp("TM_WEIGHT_DECAY", "weight_decay", 1e-4),   # official Adam weight decay
    dropout=hp("TM_DROPOUT", "dropout", 0.0),             # latent dropout before the psi decoder (all models); also inside the transformer
    d_model=hp("TM_D_MODEL", "d_model", 64),              # transformer token width
    tx_layers=env("TM_TX_LAYERS", 3),
    tx_heads=env("TM_TX_HEADS", 4),
    tune_trials=env("TM_TUNE_TRIALS", 6),                 # `tune`: random-search trials per model
    tune_steps=env("TM_TUNE_STEPS", 300),                 # `tune`: training steps per trial (validation split only)
    tune_seed=env("TM_TUNE_SEED", 7),                     # `tune`: same sampled lr/wd/dropout list for every model
    train_from_scratch=env("TM_TRAIN_FROM_SCRATCH", "0") == "1",
    max_steps=env("TM_MAX_STEPS", 10_000_000),
    dl_workers=env("TM_DL_WORKERS", -1),                  # -1 -> auto; 0 -> main process only; N -> N workers
    use_dp=env("TM_USE_DP", "auto"),                      # auto|0|1
    ckpt_every_s=env("TM_CKPT_EVERY_S", 600),
    force_reselect=env("TM_FORCE_RESELECT", "0") == "1",
    plot_windows=env("TM_PLOT_WINDOWS", 3),
    git_revs=env("TM_GIT_REVS", "unknown"),
    index_threads=env("TM_INDEX_THREADS", 12),            # concurrency; global req/s throttle (_RL_RPS) is the real limiter
    index_stop_gb=env("TM_INDEX_STOP_GB", 0.0),           # 0 -> auto: max(3, 1.6 x data budget)
    index_min_tars=env("TM_INDEX_MIN_TARS", 6),           # spread subset across >= this many archives
    failure_aware=_FA,                                    # True -> Module 2 failure-aware masked architecture + augmentation
    fail_aug_p=env("TM_FAIL_AUG_P", 0.5),                 # probability of failure augmentation during training
    pinn_lambda=_PINN,                                    # weight of flux-loop consistency loss (0 -> term not computed at all)
    pinn_mode=env("TM_PINN_MODE", "measured").lower().strip(),   # measured: real loop positions + measured flux | legacy: old pixels vs true psi
    variant=_VAR or ("plain" if not _FA else ("mask" if _PINN <= 0 else "mask_pinn")),
)
if CFG["pinn_mode"] not in ("measured", "legacy"):
    raise ValueError(f"TM_PINN_MODE must be measured|legacy, got {CFG['pinn_mode']!r}")
if CFG["run_tag"].lower() == "auto":   # legacy-PINN runs get their own dir so they never collide with measured ones
    CFG["run_tag"] = (f"{CFG['model'].lower().strip()}_{CFG['variant']}"
                      f"{'_legacy' if CFG['variant'] == 'mask_pinn' and CFG['pinn_mode'] == 'legacy' else ''}_s{CFG['seed']}")


def log(*a):
    print(f"[{(time.time() - T0) / 60:7.1f}m]", *a, flush=True)


def paths(split):
    base = os.path.join(CFG["work"], "out", split)
    out = os.path.join(base, CFG["run_tag"]) if CFG["run_tag"] else base   # per-run dir; selection stays shared in base
    return dict(
        base=base,
        out=out,
        data=os.path.join(CFG["data_root"], split),
        index=os.path.join(CFG["work"], "out", "tar_index.json"),
        sel=os.path.join(base, "selection.json"),
        model_dir=os.path.join(out, "model") + "/",
        results=os.path.join(out, "results") + "/",
        figures=os.path.join(out, "figures") + "/",
    )


def setup_imports():
    for p in (os.path.join(CFG["tokamark_dir"], "src"), CFG["baseline_dir"]):
        if p not in sys.path:
            sys.path.insert(0, p)
    os.chdir(CFG["baseline_dir"])


def split_files(split):
    """Official split CSV + stats + outlier file; ALWAYS matched per protocol."""
    setup_imports()
    from tokamark.tools.path import (RANDOM_SPLIT_TOKAMARK_DATA_SPLITS_FILE as RS, RANDOM_SPLIT_SIGNALS_STATS_FILE as RT,
                                     TEMPORAL_SPLIT_TOKAMARK_DATA_SPLITS_FILE as TS, TEMPORAL_SPLIT_SIGNALS_STATS_FILE as TT)
    from MAST_tools.utils.path_utils import RANDOM_SPLIT_OUTLIER_METADATA_FILE as RO, TEMPORAL_SPLIT_OUTLIER_METADATA_FILE as TO
    if split == "random":
        return RS, RT, RO
    if split == "temporal":
        return TS, TT, TO
    raise ValueError(f"unknown split {split}")


def patch_stats_module(stats):
    """Point get_signals_metadata at the stats file of the CURRENT split protocol.

    Upstream get_task_metadata() always reads the RANDOM-split stats file, and
    tokamark.evaluator binds get_signals_metadata at import time; patch both module
    references so metadata and NRMSE normalisation always match the protocol.
    """
    setup_imports()
    import tokamark.tasks as _tt
    import tokamark.evaluator as _te
    if getattr(_tt, "_tm_patched", None) == stats:
        return
    orig = getattr(_tt, "_tm_orig", None) or _tt.get_signals_metadata
    _tt._tm_orig = orig
    patched = lambda file_path=stats, _o=orig: _o(file_path=file_path)  # noqa: E731
    _tt.get_signals_metadata = patched
    _te.get_signals_metadata = patched
    _tt._tm_patched = stats
    log(f"patched get_signals_metadata -> {os.path.basename(stats)}")


def first_batch(loader, tries=100):
    """Return the first non-empty batch (collate_fn may yield None for empty batches)."""
    it = iter(loader)
    for _ in range(tries):
        b = next(it, None)
        if b is not None:
            return b
    raise RuntimeError("loader produced no non-empty batch")


def first_batch_timed(loader, tag="first batch", tries=100, timeout_s=1800):
    """first_batch with a heartbeat + hard wall-clock cap.

    A silent hang here (workers stuck on IPC / fork-after-CUDA / slow zarr reads) used
    to look like training forever; now it is visible every 30 s and fails loudly.
    """
    import threading
    res, done = {}, threading.Event()

    def _work():
        try:
            res["b"] = first_batch(loader, tries)
        except BaseException as e:  # noqa
            res["e"] = e
        finally:
            done.set()

    threading.Thread(target=_work, daemon=True).start()
    t0 = time.time()
    while not done.wait(30):
        log(f"  waiting for {tag}... {time.time() - t0:.0f}s (workers are still producing it)")
        if time.time() - t0 > timeout_s:
            raise RuntimeError(
                f"{tag} did not arrive within {timeout_s}s. This is a dataloader hang, not training. "
                f"Try TM_DL_WORKERS=0 (main-process loading, no workers/IPC) or a smaller TM_BATCH "
                f"({CFG['batch_size']} -> 128/64).")
    if "e" in res:
        raise res["e"]
    return res["b"]


# ======================================================================================
# PREP: index tars with range requests, select shots per official split, fetch members
# ======================================================================================
def hf_headers():
    tok = os.environ.get("HF_TOKEN")
    return {"Authorization": f"Bearer {tok}"} if tok else {}


import threading
_TLS = threading.local()


def _session():
    """Per-thread requests.Session: TLS keep-alive turns ~1-3 s/request into ~0.2 s."""
    s = getattr(_TLS, "session", None)
    if s is None:
        import requests
        s = _TLS.session = requests.Session()
    return s


class _Deadline(Exception):
    pass


_RL_LOCK = threading.Lock()
_RL_NEXT = [0.0]        # earliest wall-clock time the next request may start (shared throttle)
_RL_RPS = [float(os.environ.get("TM_HF_RPS", 8))]    # current allowed req/s (adaptive)
_RL_RPS_MIN = max(1.0, float(os.environ.get("TM_HF_RPS_MIN", 2)))    # floor when throttled
_RL_429 = [0]           # consecutive-ish 429 counter (decays on success)


def _throttle():
    """Block until this thread may issue the next HF range request.

    All threads share one token budget (req/s). The rate is ADAPTIVE: each 429
    halves it (down to _RL_RPS_MIN) and every ~50 clean requests nudges it back up
    toward the configured ceiling. So an over-aggressive TM_HF_RPS self-corrects
    instead of failing — the setting sets the ceiling, the server decides the pace.
    """
    with _RL_LOCK:
        now = time.time()
        wait = _RL_NEXT[0] - now
        if wait > 0:
            time.sleep(wait)
        _RL_NEXT[0] = max(now, _RL_NEXT[0]) + 1.0 / max(0.5, _RL_RPS[0])


def _throttle_on_429():
    """Called on HTTP 429: halve the shared rate (floor _RL_RPS_MIN) so all threads slow."""
    with _RL_LOCK:
        _RL_429[0] += 1
        _RL_RPS[0] = max(_RL_RPS_MIN, _RL_RPS[0] / 2.0)
        return _RL_RPS[0]


def _throttle_on_ok():
    """Called on success: slowly recover toward the configured ceiling."""
    with _RL_LOCK:
        _RL_429[0] = max(0, _RL_429[0] - 1)
        if _RL_429[0] == 0:
            ceiling = float(os.environ.get("TM_HF_RPS", 8))
            _RL_RPS[0] = min(ceiling, _RL_RPS[0] * 1.05)


def ranged_get(url, a, b, tries=10, deadline=None):
    last = None
    for i in range(tries):
        if deadline is not None and time.time() > deadline:
            raise _Deadline()
        _throttle()
        try:
            r = _session().get(url, headers={**hf_headers(), "Range": f"bytes={a}-{b}"}, timeout=60)
            if r.status_code in (200, 206):
                if r.status_code == 200 and len(r.content) > (b - a + 1):  # server ignored Range
                    raise RuntimeError("Range ignored by server")
                _throttle_on_ok()
                return r.content
            last = f"HTTP {r.status_code}"
            if r.status_code == 429:
                # 429 = Too Many Requests. Shrink the shared rate (adaptive), honour
                # Retry-After, and push the shared clock forward so siblings slow too.
                rate = _throttle_on_429()
                ra = r.headers.get("Retry-After")
                try:
                    pause = float(ra) if ra is not None else 0.0
                except ValueError:
                    pause = 0.0
                pause = max(pause, min(120.0, 3.0 * (2 ** i)))
                with _RL_LOCK:
                    _RL_NEXT[0] = max(_RL_NEXT[0], time.time() + pause)
                time.sleep(pause + random.random() * 2.0)
                if i == 2:
                    log(f"  HF rate-limiting (429): shared rate -> {rate:.1f} req/s (auto-adapting)")
                continue
            if r.status_code in (500, 502, 503, 504):
                time.sleep(min(60.0, 2.0 * (2 ** i)) + random.random())
                continue
            time.sleep(min(30.0, 1.0 * (2 ** i)))
        except _Deadline:
            raise
        except Exception as e:  # noqa
            last = repr(e)
            time.sleep(min(30.0, 1.0 * (2 ** i)) + random.random())
    raise RuntimeError(f"range {a}-{b} failed after {tries} tries: {last}")


def make_ranged_reader(url, total, deadline=None, span=4096):
    """Small-read-optimised reader for parse_tar_headers.

    Header walks issue many 512-byte reads a few KB apart; HF resolve latency makes
    one-request-per-read prohibitively slow, so each read pulls a `span`-byte window
    and serves nearby reads from a one-window cache.
    """
    buf = {"start": -1, "data": b""}

    def read(a, n):
        end = min(a + n, total)
        if a >= total or end <= a:
            return b""
        if buf["start"] <= a and end <= buf["start"] + len(buf["data"]):
            o = a - buf["start"]
            return buf["data"][o:o + (end - a)]
        b = min(a + max(span, n) - 1, total - 1)
        data = ranged_get(url, a, b, deadline=deadline)
        buf["start"], buf["data"] = a, data
        return data[: end - a]

    return read


def _octal_or_b256(field):
    if field[0] & 0x80:
        return int.from_bytes(field, "big") & ((1 << (8 * len(field) - 1)) - 1)
    s = field.split(b"\0")[0].strip()
    return int(s, 8) if s else 0


def parse_tar_headers(read, total):
    """Walk a tar using read(offset, length)->bytes. Returns [(name, data_offset, size)]."""
    members, off, pending = [], 0, None
    while off + 512 <= total:
        h = read(off, 512)
        if len(h) < 512 or h.count(0) == 512:
            break
        name = h[:100].split(b"\0")[0].decode("utf8", "replace")
        if h[257:262] == b"ustar" and h[345:500].strip(b"\0"):
            name = h[345:500].split(b"\0")[0].decode("utf8", "replace") + "/" + name
        size = _octal_or_b256(h[124:136])
        flag = h[156:157]
        data_off = off + 512
        nxt = data_off + ((size + 511) // 512) * 512
        if flag in (b"L", b"x", b"g", b"K"):
            data = read(data_off, size)
            if flag == b"L":
                pending = data.split(b"\0")[0].decode("utf8", "replace")
            elif flag == b"x":
                for rec in data.split(b"\n"):
                    if b" path=" in rec:
                        pending = rec.split(b" path=", 1)[1].decode("utf8", "replace")
            off = nxt
            continue
        if flag in (b"0", b"\0"):
            members.append((pending or name, data_off, size))
        pending = None
        off = nxt
    return members


def build_index(deadline_ts, hard_deadline=None):
    """Index HF tars via range-request header walks. Resumable (index saved per tar).

    Tars are walked in a seeded-shuffled order and indexing stops early once the
    indexed shot bytes cover the download budget (x1.6 margin) - we only need enough
    archives to draw the subset, not the whole 500+ GB repo.

    `deadline_ts` is a SOFT deadline: once it passes, in-flight walks are cancelled
    when we already have enough (stop_reached) OR enough tars to draw a valid subset;
    otherwise indexing continues up to `hard_deadline` (the prep deadline), because an
    under-sized index risks the fatal <5-shot guard. Re-runs continue from the index.
    """
    hard_deadline = hard_deadline or deadline_ts
    from huggingface_hub import HfApi, hf_hub_url
    P = paths("random")
    os.makedirs(os.path.dirname(P["index"]), exist_ok=True)
    idx = json.load(open(P["index"])) if os.path.exists(P["index"]) else {"tars": {}, "shots": {}}
    api = HfApi()
    files = [f for f in api.list_repo_tree(CFG["repo_id"], repo_type="dataset", recursive=True)
             if getattr(f, "size", None) is not None and f.path.endswith(".tar")]
    files.sort(key=lambda f: f.path)
    log(f"HF repo lists {len(files)} tar files, {sum(f.size for f in files) / 1e9:.1f} GB total")
    indexed_bytes = sum(v[2] for v in idx["shots"].values())
    stop_gb = CFG["index_stop_gb"] or max(3.0, 1.6 * CFG["data_budget_gb"])
    min_tars = max(1, CFG["index_min_tars"])
    log(f"index: {len(idx['shots'])} shots / {indexed_bytes / 1e9:.1f} GB indexed (cached) across {len(idx['tars'])} tars; "
        f"stop submitting beyond {stop_gb:.1f} GB and >= {min_tars} tars")
    todo = [f for f in files if f.path not in idx["tars"]]
    random.Random(CFG["seed"]).shuffle(todo)  # seeded spread across the archive

    def stop_reached():
        return indexed_bytes >= stop_gb * 1e9 and len(idx["tars"]) >= min_tars

    def usable():
        # enough to select a valid subset even if a few tars still fail
        return len(idx["tars"]) >= min(3, min_tars)

    def work(f):
        url = hf_hub_url(CFG["repo_id"], f.path, repo_type="dataset")
        # reads may run until the HARD deadline; the loop cancels in-flight walks at the
        # soft deadline when the index is already usable.
        read = make_ranged_reader(url, f.size, deadline=hard_deadline)
        nreq = [0]
        mem = parse_tar_headers(lambda a, n: (nreq.__setitem__(0, nreq[0] + 1), read(a, n))[1], f.size)
        return f.path, url, mem

    with ThreadPoolExecutor(CFG["index_threads"]) as ex:
        it = iter(todo)
        pending = {}
        retries = {}
        inflight = set()

        def submit():
            if stop_reached():
                return False
            try:
                f = next(it)
            except StopIteration:
                # second pass: re-queue tars that failed transiently (e.g. HTTP 429),
                # never ones still being walked in another thread.
                cand = [f for f in todo
                        if f.path not in idx["tars"] and f.path not in inflight and retries.get(f.path, 0) < 3]
                if not cand:
                    return False
                f = cand[0]
                retries[f.path] = retries.get(f.path, 0) + 1
            fu = ex.submit(work, f)
            pending[fu] = f.path
            inflight.add(f.path)
            return True

        need = stop_gb * 1e9 - indexed_bytes
        cum, n_need = 0, 0
        if need > 0:
            for f in todo:
                cum += f.size
                n_need += 1
                if cum >= need:
                    break
        min_tars_left = max(0, min_tars - len(idx["tars"]))
        initial = min(CFG["index_threads"], max(n_need, min_tars_left), len(todo))
        log(f"submitting {initial} tars now (need ~{n_need} for budget, {min_tars_left} for archive spread)")
        for _ in range(initial):
            if not submit():
                break
        last_tick = time.time()
        while pending:
            now = time.time()
            if now > hard_deadline:
                log(f"index HARD deadline reached ({len(idx['tars'])} tars, {indexed_bytes / 1e9:.1f} GB): "
                    "cancelling remaining tar walks (download phase needs the rest)")
                for fu in pending:
                    fu.cancel()
                break
            if now > deadline_ts and (usable() or stop_reached()):
                log(f"index soft deadline reached with a usable index ({len(idx['tars'])} tars, "
                    f"{indexed_bytes / 1e9:.1f} GB): reserving the rest of prep for downloads")
                for fu in pending:
                    fu.cancel()
                break
            if now > deadline_ts and not usable():
                # too small to guarantee a valid subset -> keep going to the hard deadline
                if time.time() - last_tick > 120:
                    log(f"  index past soft deadline but only {len(idx['tars'])} tars: continuing to hard deadline")
                    last_tick = time.time()
            if stop_reached():
                log(f"enough archives indexed ({indexed_bytes / 1e9:.1f} GB across {len(idx['tars'])} tars): "
                    "cancelling remaining tar walks")
                for fu in pending:
                    fu.cancel()
                break
            try:
                done_fu = next(as_completed(list(pending), timeout=30), None)
            except TimeoutError:
                if time.time() - last_tick > 120:
                    log(f"  indexing... {len(pending)} tars in flight, {len(idx['tars'])} done, {indexed_bytes / 1e9:.1f} GB")
                    last_tick = time.time()
                continue
            fpath = pending.pop(done_fu)
            inflight.discard(fpath)
            try:
                path, url, mem = done_fu.result()
            except _Deadline:
                log(f"  {fpath}: walk aborted at indexing deadline (will resume next run)")
                continue
            except Exception as e:  # noqa
                log(f"  index FAILED for {fpath}: {repr(e)[:200]}")
                submit()
                continue
            n = 0
            for name, off, size in mem:
                base = os.path.basename(name)
                if not base.endswith(".zarr.zip"):
                    continue
                try:
                    sid = int(base[: -len(".zarr.zip")])
                except ValueError:
                    log("  skip unparsable member", name)
                    continue
                if str(sid) not in idx["shots"]:
                    idx["shots"][str(sid)] = [url, off, size]
                    indexed_bytes += size
                n += 1
            idx["tars"][path] = n
            log(f"  indexed {path}: {n} shots ({indexed_bytes / 1e9:.1f} GB total indexed)")
            json.dump(idx, open(P["index"], "w"))
            submit()
    missing = [f.path for f in files if f.path not in idx["tars"]]
    if missing and not stop_reached():
        log(f"WARNING: {len(missing)} tars not indexed (deadline/network?): {missing[:5]} - re-run prep to continue")
    log(f"index: {len(idx['shots'])} shots across {len(idx['tars'])} tars ({indexed_bytes / 1e9:.1f} GB)")
    return idx


def _quota_select(budget, official, avail, rng):
    """Random-priority selection of shots under a byte quota (order stays random)."""
    rng.shuffle(avail)
    used, chosen = 0, []
    for s in avail:
        sz = s[1]
        if used + sz > budget:
            continue  # keep scanning: smaller shots may still fit
        chosen.append(s[0])
        used += sz
    return chosen, used


def choose_shots(split, idx):
    from tokamark.data_split import read_data_split_csv
    csv, _, _ = split_files(split)
    tr, te, va = read_data_split_csv(csv)
    official = {"train": [int(s) for s in tr], "val": [int(s) for s in va], "test": [int(s) for s in te]}
    total = sum(len(v) for v in official.values())
    free = shutil.disk_usage(paths(split)["data"]).free
    budget = min(CFG["data_budget_gb"] * 1e9, max(0.0, free - 6e9))
    log(f"[{split}] official counts: " + ", ".join(f"{k}={len(v)}" for k, v in official.items()) + f" (total {total}); byte budget {budget / 1e9:.1f} GB")
    rng = random.Random(CFG["select_seed"])   # NOT TM_SEED: all seeds train on the same shots
    MIN_SHOTS = 5  # matches the fatal guard in cmd_prep
    sel, report = {}, {}
    for k, shots in official.items():
        avail = [(s, idx["shots"][str(s)][2]) for s in shots if str(s) in idx["shots"]]
        quota = budget * len(shots) / total
        rng.shuffle(avail)
        # guarantee >= MIN_SHOTS per split even when the byte quota is tiny: the 5
        # smallest available (cost-bounded), then fill the rest of the quota randomly.
        guard = sorted(avail, key=lambda t: t[1])[:MIN_SHOTS]
        guard_ids = {s for s, _ in guard}
        used1 = sum(sz for _, sz in guard)
        rest = [t for t in avail if t[0] not in guard_ids]
        extra, used2 = _quota_select(max(0.0, quota - used1), shots, rest, rng)
        chosen = [s for s, _ in guard] + extra
        used = used1 + used2
        sel[k] = chosen
        report[k] = dict(official=len(shots), in_archives=len(avail), selected=len(chosen), gb=round(used / 1e9, 2),
                         fraction_of_official=round(len(chosen) / max(1, len(shots)), 4))
    log(f"[{split}] selection: {json.dumps(report)}")
    return sel, report


def fetch_member(sid, meta, data_dir):
    url, off, size = meta
    final = os.path.join(data_dir, f"{sid}.zarr")
    if os.path.isdir(final):
        return sid, True, "cached"
    zpath = os.path.join(data_dir, f"{sid}.zip.part")
    tmpd = os.path.join(data_dir, f"{sid}.extract")
    try:
        with open(zpath, "wb") as fh:
            pos, CH = off, 64 * 2 ** 20
            while pos < off + size:
                end = min(pos + CH, off + size) - 1
                fh.write(ranged_get(url, pos, end))
                pos = end + 1
        if os.path.getsize(zpath) != size:
            raise RuntimeError("size mismatch")
        shutil.rmtree(tmpd, ignore_errors=True)
        with zipfile.ZipFile(zpath) as z:
            z.extractall(tmpd)
        root = tmpd
        if not (os.path.exists(os.path.join(root, "zarr.json")) or os.path.exists(os.path.join(root, ".zgroup"))):
            kids = [d for d in os.listdir(tmpd) if os.path.isdir(os.path.join(tmpd, d))]
            if len(kids) == 1 and (os.path.exists(os.path.join(tmpd, kids[0], "zarr.json")) or os.path.exists(os.path.join(tmpd, kids[0], ".zgroup"))):
                root = os.path.join(tmpd, kids[0])
            else:
                raise RuntimeError("no zarr root found in archive")
        import zarr
        zarr.open_group(root, mode="r")  # must open
        os.replace(root, final)
        return sid, True, "ok"
    except Exception as e:  # noqa
        return sid, False, repr(e)[:200]
    finally:
        shutil.rmtree(tmpd, ignore_errors=True)
        if os.path.exists(zpath):
            os.remove(zpath)


def cmd_prep(split):
    setup_imports()
    P = paths(split)
    os.makedirs(P["data"], exist_ok=True)
    os.makedirs(P["base"], exist_ok=True)
    deadline = T0 + CFG["prep_deadline_h"] * 3600
    # Indexing gets a soft sub-deadline (a fraction of the prep window) so a slow or
    # re-throttled index cannot starve downloads. But if the index is still short of the
    # size it needs when the soft deadline hits, build_index is allowed to use up to the
    # hard prep deadline instead - an under-sized index (which risks the fatal <5-shot
    # guard) is worse than a shorter download window. See build_index(soft, hard).
    index_frac = float(os.environ.get("TM_INDEX_TIME_FRAC", 0.25))
    index_soft = min(deadline, T0 + CFG["prep_deadline_h"] * 3600 * index_frac)
    idx = build_index(index_soft, hard_deadline=deadline)

    reuse = None
    if os.path.exists(P["sel"]) and not CFG["force_reselect"]:
        prev = json.load(open(P["sel"]))
        if prev.get("split") == split and min((len(v) for v in prev.get("shots", {}).values()), default=0) >= 5:
            reuse = prev
            log("reusing previous selection.json "
                f"({json.dumps({k: len(v) for k, v in reuse['shots'].items()})}); "
                "set TM_FORCE_RESELECT=1 to re-select shots")
    if reuse:
        sel, report = reuse["shots"], reuse["report"]
        for k in sel:
            report.setdefault(k, {})
            report[k]["official"] = report[k].get("official", len(sel[k]))
    else:
        sel, report = choose_shots(split, idx)

    # round-robin across splits so a deadline cut keeps the proportions; skip cached shots
    prep_splits = [s.strip() for s in os.environ.get("TM_PREP_SPLITS", "train,val,test").split(",") if s.strip()]
    order, lists = [], {k: [s for s in v if not os.path.isdir(os.path.join(P["data"], f"{s}.zarr"))]
                        for k, v in sel.items() if k in prep_splits}
    n_cached = sum(len(sel[k]) for k in prep_splits) - sum(len(lists[k]) for k in prep_splits)
    if n_cached:
        log(f"{n_cached} selected shots already on disk (cached)")
    while any(lists.values()):
        for k in prep_splits:
            if lists.get(k):
                order.append((k, lists[k].pop()))
    done, failed = set(), {}
    nworkers = int(os.environ.get("TM_FETCH_WORKERS", 4))
    attempts = {}
    inflight = set()
    with ThreadPoolExecutor(nworkers) as ex:
        it = iter(order)
        futs = {}
        def submit():
            try:
                k, s = next(it)
            except StopIteration:
                # re-queue shots that failed transiently (HTTP 429 etc.), bounded
                # retries, and never ones already in flight in another thread.
                cand = [(k, s) for (k, s) in order
                        if s not in done and s not in inflight and attempts.get(s, 0) < 3]
                if not cand:
                    return False
                k, s = cand[0]
                attempts[s] = attempts.get(s, 0) + 1
            futs[ex.submit(fetch_member, s, idx["shots"][str(s)], P["data"])] = (k, s)
            inflight.add(s)
            return True
        for _ in range(nworkers * 2):
            submit()
        while futs:
            fu = next(as_completed(list(futs)))
            k, s = futs.pop(fu)
            inflight.discard(s)
            sid, ok, msg = fu.result()
            if ok:
                done.add(sid)
                failed.pop(sid, None)
            else:
                failed[sid] = msg
            if len(done) % 25 == 0:
                log(f"  fetched {len(done)}/{len(order)} shots, failed {len(failed)}")
            if time.time() < deadline:
                submit()
        # one last pass for still-failed shots while time remains (paced = fewer 429s)
        for (k, s) in [t for t in order if t[1] not in done]:
            if time.time() >= deadline or s in done or attempts.get(s, 0) >= 3:
                continue
            attempts[s] = attempts.get(s, 0) + 1
            sid, ok, msg = fetch_member(s, idx["shots"][str(s)], P["data"])
            if ok:
                done.add(sid); failed.pop(sid, None)
            else:
                failed[sid] = msg
    final = {}
    for k, v in sel.items():
        if k in prep_splits:
            final[k] = [s for s in v if (s in done or os.path.isdir(os.path.join(P["data"], f"{s}.zarr")))]
        else:
            final[k] = v  # preserve the selected shot list for splits not being fetched
    if time.time() >= deadline and any(len(final[k]) < len(sel[k]) for k in prep_splits):
        log(f"prep deadline reached: using the {sum(len(final[k]) for k in prep_splits)} shots fetched so far "
            f"({sum(len(sel[k]) for k in prep_splits) - sum(len(final[k]) for k in prep_splits)} still missing)")
    report2 = {k: dict(report.get(k, {}), downloaded=len(final[k])) for k in final}
    meta = dict(split=split, shots=final, report=report2, failed=failed,
                config=dict(seed=CFG["select_seed"], task=CFG["task"], data_budget_gb=CFG["data_budget_gb"],
                            repo_id=CFG["repo_id"], git_revs=CFG["git_revs"], selected_at_min=round((time.time() - T0) / 60, 1)))
    json.dump(meta, open(P["sel"], "w"), indent=1)
    log(f"[{split}] FINAL shot counts: " + json.dumps({k: len(v) for k, v in final.items()}))
    if failed:
        log(f"  {len(failed)} shots failed and were excluded; first errors: {list(failed.items())[:3]}")
    req_splits = [k for k in prep_splits if k in final]
    if req_splits and min(len(final[k]) for k in req_splits) < 5:
        raise SystemExit(f"FATAL: fewer than 5 shots in some requested split ({req_splits}); refusing to train on a misleading subset. "
                         "Check network/index logs above, extend TM_DATA_GB / TM_PREP_DEADLINE_H, and re-run prep.")


# ======================================================================================
# Datasets / model (mirrors tokamark_baseline/run_training.py exactly, minus subset_of_shots)
# ======================================================================================
def build_data(split, which, shuffle_windows=True):
    setup_imports()
    import torch
    from torch.utils.data import DataLoader
    from tokamark.tasks import get_task_config, get_task_metadata
    from tokamark.tools.transforms.compose_transform import ComposeTransforms
    from tokamark.data import initialize_MAST_dataset, initialize_TokaMark_dataset
    from src.model_transform import ModelTransform_1, ModelTransform_2
    from src.trainer import model_collate_fn
    from utils import seed_worker

    P = paths(split)
    sel = json.load(open(P["sel"]))
    assert sel["split"] == split, "selection file belongs to another split protocol!"
    _, stats, outl = split_files(split)
    patch_stats_module(stats)
    cfgt = get_task_config(task_name=CFG["task"])
    cfgt["stride_window"] = 0.005  # same override as run_training.py for task_1-x/2-x
    meta = get_task_metadata(config_task=cfgt, verbose=False)
    tf = ComposeTransforms([ModelTransform_1(meta | cfgt), ModelTransform_2(meta | cfgt)])
    nw = CFG["dl_workers"]
    if nw < 0:
        nw = max(2, min(8, os.cpu_count() or 2))
    smgr = {"base_local_zarr_path": P["data"]}
    loaders, g = {}, torch.Generator()
    g.manual_seed(CFG["seed"])
    # Tensor IPC must not go through /dev/shm (Docker default is 64 MB; a 512-window
    # batch deadlocks the workers silently). file_system uses /tmp instead.
    torch.multiprocessing.set_sharing_strategy("file_system")
    for w in which:
        mast = initialize_MAST_dataset(config_task=cfgt, shots_list=sel["shots"][w], local_flag=True, use_std_scaling=True,
                                       stats_metadata_file_path=stats, use_nan_filling=False, remove_outliers=True,
                                       outlier_metadata_file=outl, remove_bad_efit_rating=True,
                                       store_manager_settings=smgr, verbose=False)
        kw = dict(test_mode=True)
        if w == "test":
            kw["shuffle_windows"] = False
        ds = initialize_TokaMark_dataset(dataset=mast, task_metadata=meta, config_metadata=cfgt, custom_transform=tf, **kw)
        if ds is None:
            raise RuntimeError(f"{w} dataset empty after official filtering (outliers/bad EFIT)")
        # TokaMarkDataset is an IterableDataset (no __getitem__), so probe via the
        # iterator: time how long one window takes to materialise. A real failure here
        # (empty dataset, bad zarr) is fatal, not a harmless warning.
        t_probe = time.time()
        first = next(iter(ds), None)
        dt = time.time() - t_probe
        if first is None:
            raise RuntimeError(f"{w} dataset yielded no windows after official filtering "
                               f"(outliers/bad EFIT / empty shots) - cannot train on it")
        est = dt * CFG["batch_size"] / max(1, min(max(nw, 1), CFG["batch_size"]))
        log(f"{w} dataset: {len(sel['shots'][w])} shots ({type(ds).__name__}, streaming windows); "
            f"first window took {dt:.2f}s "
            f"(=> first batch of {CFG['batch_size']} will take roughly {est / 60:.1f} min)")
        extra = dict(drop_last=True) if w == "train" else {}
        if nw > 0:
            extra["persistent_workers"] = True
        loaders[w] = DataLoader(ds, batch_size=CFG["batch_size"], num_workers=nw, collate_fn=model_collate_fn,
                                worker_init_fn=seed_worker, generator=g, pin_memory=torch.cuda.is_available(),
                                prefetch_factor=2 if nw > 0 else None, **extra)
        log(f"{w} loader ready: {len(sel['shots'][w])} shots, batch_size={CFG['batch_size']}, workers={nw}")
    return loaders, meta | cfgt


# ======================================================================================
# MODULE 2: FAILURE-AWARE ARCHITECTURE & PINN LOSS (MASK DROP + FLUX-LOOP CONSISTENCY)
# ======================================================================================
FLUX_LOOP_COORDS = (
    (2, 13), (2, 22), (2, 32), (2, 42), (2, 51),
    (60, 6), (60, 13), (60, 19), (60, 26), (60, 32),
    (60, 38), (60, 45), (60, 51), (60, 58), (60, 62)
)


# ---- measured-flux PINN (TM_PINN_MODE=measured, default) ------------------------------------------------------------
# Real (R, Z) [m] of the 15 flux loops = input group 0, channel order (cnn_mlp/src/study/geometry/mast_sensor_geometry.json,
# FAIR-MAST level-2 geometry of shot 30421). psi grid: R = 0.06 + 0.03 i (column), Z = -2 + 0.0625 j (row), psi[z_row, r_col].
FLUX_LOOP_RZ = ((0.1785, 0.6249), (0.1785, 0.3199), (0.1785, 0.0149), (0.1785, -0.3231), (0.1785, -0.9331),
                (1.0354, -1.0886), (1.163, 1.08259), (1.0354, 1.08309), (1.5984, -1.04443), (1.4005, -1.14403),
                (1.4001, 1.15079), (1.7463, -0.44224), (1.5495, -0.54724), (1.7493, 0.44221), (1.4025, -0.8876))
PSI_GRID = dict(r0=0.06, dr=0.03, z0=-2.0, dz=0.0625, n=65)
# Measured flux through a loop = 2*pi*psi_EFIT (Wb vs Wb/rad): fit on 16 local shots / 1083 windows, slope 6.2-6.3 on the four
# well-resolved inboard loops, offset -0.002 Wb (first window sample). rms |2*pi*psi(loop) - measured| in standardised-psi units
# per loop (all shots, first window sample): the inboard CC loops 0-4 agree to ~0.03, the outer loops only to 0.23-0.32 (EFIT psi
# is not accurate next to the PF coils), so each loop is weighted min(1, (0.1/resid)^2).
FLUX_LOOP_RESID = (0.030, 0.028, 0.029, 0.029, 0.031, 0.255, 0.261, 0.231, 0.317, 0.290, 0.304, 0.258, 0.230, 0.238, 0.286)
FLUX_LOOP_WEIGHT = tuple(min(1.0, (0.1 / r) ** 2) for r in FLUX_LOOP_RESID)
FLUX_LOOP_DEAD_WB = 2e-3      # a loop whose |flux| stays below 2 mWb over the whole window is dead (CC04 reads ~0 in the 27xxx shots)
_FL_STATS = {}                # {mu_f, sd_f, mu_p, sd_p}: input flux-loop / output psi standardisation, set from the task metadata


def set_flux_loop_stats(meta):
    """Take the standardisation the dataset really applied (meta['input'][...flux_loop_flux], meta['output'][...psi])."""
    f, p = meta["input"]["magnetics-flux_loop_flux"], meta["output"]["equilibrium-psi"]
    _FL_STATS.update(mu_f=float(f["mean"]), sd_f=float(f["std"]), mu_p=float(p["mean"]), sd_p=float(p["std"]))
    return _FL_STATS


def flux_loop_sample(pred_psi):
    """Bilinear sample of a (B, 1, 65, 65) psi map at the 15 real loop positions -> (B, 15). Differentiable."""
    import torch
    g = PSI_GRID
    c = torch.tensor([(R - g["r0"]) / g["dr"] for R, _ in FLUX_LOOP_RZ], dtype=torch.float64)
    r = torch.tensor([(Z - g["z0"]) / g["dz"] for _, Z in FLUX_LOOP_RZ], dtype=torch.float64)
    c0, r0 = c.floor().clamp(0, g["n"] - 2), r.floor().clamp(0, g["n"] - 2)
    fc, fr = (c - c0).to(pred_psi.dtype).to(pred_psi.device), (r - r0).to(pred_psi.dtype).to(pred_psi.device)
    c0, r0 = c0.long().to(pred_psi.device), r0.long().to(pred_psi.device)
    f = pred_psi[:, 0]
    return ((1 - fr) * ((1 - fc) * f[:, r0, c0] + fc * f[:, r0, c0 + 1])
            + fr * ((1 - fc) * f[:, r0 + 1, c0] + fc * f[:, r0 + 1, c0 + 1]))


def flux_loop_measured_target(x0):
    """x0: input group 0, (B, W, Cin, T, 15), standardised. -> (target (B,15) in psi standardisation, alive (B,15) bool).

    Target = first window sample (closest in time to the psi slice, which is the window start): raw = x*sd_f + mu_f [Wb],
    psi = raw / (2*pi) [Wb/rad], standardised with the psi stats. Alive = not NaN-filled (all-zero) and not a ~0 dead loop."""
    import math as _m
    S = _FL_STATS
    if not S:
        raise RuntimeError("flux-loop stats not set: call set_flux_loop_stats(meta) before using TM_PINN_MODE=measured")
    x = x0.reshape(x0.shape[0], -1, x0.shape[-1])                   # (B, W*Cin*T, 15), time ascending
    raw_first = x[:, 0] * S["sd_f"] + S["mu_f"]
    target = (raw_first / (2 * _m.pi) - S["mu_p"]) / S["sd_p"]
    alive = (x.abs().amax(1) > 0) & ((x * S["sd_f"] + S["mu_f"]).abs().amax(1) >= FLUX_LOOP_DEAD_WB)
    return target, alive


def flux_loop_pinn_loss_measured(pred_psi, x0, health):
    """Real-geometry consistency: psi_pred(loop position) vs the loop's own MEASURED flux, same window.

    health (B, 15) float: availability of each loop (failure-mask; 0 = lost -> no term). Weighted by FLUX_LOOP_WEIGHT."""
    import torch
    target, alive = flux_loop_measured_target(x0)
    w = torch.tensor(FLUX_LOOP_WEIGHT, dtype=pred_psi.dtype, device=pred_psi.device) * health * alive.to(pred_psi.dtype)
    err = (flux_loop_sample(pred_psi) - target.detach()) ** 2
    return (w * err).sum(1).mean() / len(FLUX_LOOP_RZ)


def flux_loop_pinn_loss(pred_psi, true_psi, mask_fl=None):
    """LEGACY (TM_PINN_MODE=legacy): hard-coded pixels vs the TRUE psi y. Enforce boundary flux consistency at 15 pixels."""
    loss = 0.0
    for r_idx, z_idx in FLUX_LOOP_COORDS:
        diff_sq = (pred_psi[:, 0, z_idx, r_idx] - true_psi[:, 0, z_idx, r_idx]) ** 2
        if mask_fl is not None:
            diff_sq = diff_sq * mask_fl
        loss = loss + diff_sq.mean()
    return loss / len(FLUX_LOOP_COORDS)


def channel_layout(x):
    """Sensors per input group: (B, W, Cin, T, N) -> N sensors on the last axis; (B, W, Cin, T) -> 1 sensor."""
    return [int(t.shape[-1]) if t.dim() >= 5 else 1 for t in x]


def _zero_sensor(t, b, n, sl=slice(None)):
    """Zero sensor n of sample b (optionally only the time slice sl, which indexes the T axis)."""
    if t.dim() >= 5:
        t[b, :, :, sl, n] = 0.0
    else:
        t[b, :, :, sl] = 0.0


def _apply_structured_failure_channels(x, p_fail):
    """Per-sensor version for the transformer: mask (B, sum_g N_g) with 0 = sensor missing.

    Same three scenario families as the group-level version, at sensor granularity:
    40% k in 1..6 random individual sensors, 30% one correlated group, 30% temporal gap on k in 1..6 sensors."""
    import torch
    n_ch = channel_layout(x)
    offs = [0]
    for n in n_ch:
        offs.append(offs[-1] + n)
    C, B = offs[-1], x[0].shape[0]
    mask = torch.ones((B, C), device=x[0].device, dtype=x[0].dtype)
    x_aug = [t.clone() for t in x]
    groups = [[0], [1, 2, 3], [4], [5, 6], [7]]
    where = lambda c: next(g for g in range(len(n_ch)) if offs[g] <= c < offs[g + 1])  # noqa: E731
    for b in range(B):
        if torch.rand(1).item() >= p_fail:
            continue
        r = torch.rand(1).item()
        if r < 0.70 and r >= 0.40:
            chans = [c for g in groups[int(torch.randint(0, len(groups), (1,)).item())] for c in range(offs[g], offs[g + 1])]
        else:
            chans = torch.randperm(C)[:int(torch.randint(1, 7, (1,)).item())].tolist()
        for c in chans:
            g = where(c)
            if r < 0.70:
                _zero_sensor(x_aug[g], b, c - offs[g])
            else:
                L = x_aug[g].shape[-2] if x_aug[g].dim() >= 5 else x_aug[g].shape[-1]
                half = max(1, L // 2)
                start = int(torch.randint(0, max(1, L - half + 1), (1,)).item())
                _zero_sensor(x_aug[g], b, c - offs[g], slice(start, start + half))
            mask[b, c] = 0.0
    return x_aug, mask


def apply_structured_failure(x, p_fail=0.5, per_channel=False):
    """Structured failure augmentation: drops channels, correlated groups, or temporal gaps.

    per_channel=False (CNN models): mask (B, 8) over the 8 sensor groups. per_channel=True (transformer): mask (B, 115)."""
    import torch
    if per_channel:
        return _apply_structured_failure_channels(x, p_fail)
    B = x[0].shape[0]
    dev = x[0].device
    dtype = x[0].dtype
    mask = torch.ones((B, 8), device=dev, dtype=dtype)
    x_aug = [t.clone() for t in x]
    groups = [[0], [1, 2, 3], [4], [5, 6], [7]]
    for b in range(B):
        if torch.rand(1).item() < p_fail:
            r = torch.rand(1).item()
            if r < 0.40:
                k = int(torch.randint(1, 4, (1,)).item())
                chans = torch.randperm(8)[:k].tolist()
                for c in chans:
                    x_aug[c][b] = 0.0
                    mask[b, c] = 0.0
            elif r < 0.70:
                grp = groups[int(torch.randint(0, len(groups), (1,)).item())]
                for c in grp:
                    x_aug[c][b] = 0.0
                    mask[b, c] = 0.0
            else:
                k = int(torch.randint(1, 3, (1,)).item())
                chans = torch.randperm(8)[:k].tolist()
                for c in chans:
                    L = x_aug[c].shape[-1]
                    half = max(1, L // 2)
                    start = int(torch.randint(0, max(1, L - half + 1), (1,)).item())
                    x_aug[c][b, :, :, start:start + half] = 0.0
                    mask[b, c] = 0.0
    return x_aug, mask


def train_loss(runner, crit, x, y):
    """One training-step loss, shared by every model/variant: (total, mse, pinn). pinn is None unless variant mask_pinn.

    plain: MSE on raw inputs. mask: MSE with structured failure augmentation + health mask. mask_pinn: + lambda * flux-loop
    consistency (not even evaluated when TM_PINN_LAMBDA = 0)."""
    if not CFG["failure_aware"]:
        loss = crit(runner(*x), y)
        return loss, loss, None
    per_ch = CFG["model"].lower().strip() == "transformer"   # transformer: per-sensor mask (B, 115)
    x_train, mask_train = apply_structured_failure(x, p_fail=CFG["fail_aug_p"], per_channel=per_ch)
    out = runner(*x_train, mask_train)
    loss_mse = crit(out, y)
    if CFG["pinn_lambda"] <= 0:
        return loss_mse, loss_mse, None
    if CFG["pinn_mode"] == "measured":
        # per-loop availability: transformer -> its 15 sensor columns; CNN models -> the flux-loop group flag for all loops
        health = mask_train[:, :15] if per_ch else mask_train[:, :1].expand(-1, 15)
        loss_fl = flux_loop_pinn_loss_measured(out[0], x[0], health)     # x = clean window (before augmentation)
        return loss_mse + CFG["pinn_lambda"] * loss_fl, loss_mse, loss_fl
    # legacy: weight by the health of the flux-loop group (group 0 = its first n0 sensor columns)
    mask_fl = mask_train[:, :channel_layout(x)[0]].mean(1) if per_ch else mask_train[:, 0]
    loss_fl = flux_loop_pinn_loss(out[0], y[0], mask_fl=mask_fl)
    return loss_mse + CFG["pinn_lambda"] * loss_fl, loss_mse, loss_fl


def _resolve_mask(model, args, kwargs):
    """Split forward args into (inputs, exogenous, health mask (B, n_in)).

    A missing mask is inferred from the inputs: a branch that is exactly zero counts as dead."""
    import torch
    n_in, n_exo = len(model.input_branches), len(model.exogenous_branches)
    inputs, exogenous = args[:n_in], args[n_in:n_in + n_exo]
    mask = kwargs.get("mask", None)
    if mask is None and len(args) > n_in + n_exo:
        mask = args[n_in + n_exo]
    B, dev, dtype = inputs[0].shape[0], inputs[0].device, inputs[0].dtype
    if mask is None:
        mask = torch.ones((B, n_in), device=dev, dtype=dtype)
        for i, x_chan in enumerate(inputs):
            is_zero = (x_chan.abs().sum(dim=tuple(range(1, x_chan.dim()))) == 0.0)
            mask[:, i] = torch.where(is_zero, 0.0, 1.0)
    elif mask.dim() == 1:
        mask = mask.unsqueeze(0).expand(B, -1)
    return inputs, exogenous, mask


def _first_shapes(loader):
    """(input, exogenous, output) per-sample shapes from the first usable window of the loader's dataset."""
    for first_window in loader.dataset:
        try:
            return ([arr.shape for arr in first_window["input"]], [arr.shape for arr in first_window["exogenous"]],
                    [arr.shape for arr in first_window["y"]])
        except Exception:
            continue
    return [], [], []


def make_failure_aware_model(loader, meta, shapes=None):
    """Module 2 network for the chosen backbone (TM_MODEL = mlp | lstm).

    Both keep the upstream one-CNN-encoder-per-sensor layout. Each encoder output is gated by its sensor's health
    flag, and the flag vector is also fed to the backbone (MLP bottleneck / LSTM input) so it knows what is missing."""
    setup_imports()
    import torch
    import torch.nn as nn
    from torch.utils.checkpoint import checkpoint
    is_lstm = CFG["model"].lower().strip() == "lstm"
    if is_lstm:
        from src.multi_conv_lstm_model import LSTM_v3 as Base, bb_factor
    else:
        from src.multi_conv_mlp_model import MultiConv_MLP as Base, bb_factor
    from tokamark.tools.utils import get_device

    device = get_device()
    input_shapes, exogenous_shapes, output_shapes = shapes or _first_shapes(loader)

    class MultiConv_FailureAware_MLP(Base):
        def __init__(self, in_shapes, exo_shapes, out_shapes, dict_meta, D=16):
            super().__init__(in_shapes, exo_shapes, out_shapes, dict_meta, D=D)
            self.n_in = len(self.input_branches)
            in_features = self.D * bb_factor * self.n_in * self.W_in + self.n_in
            self.encoder_mlp = nn.Sequential(
                nn.Linear(in_features, 2 * self.D * bb_factor),
                nn.ReLU(),
                nn.Linear(2 * self.D * bb_factor, self.D * bb_factor),
            )

        def forward(self, *args, **kwargs):
            n_in = self.n_in
            n_exo = len(self.exogenous_branches)
            inputs = args[:n_in]
            exogenous = args[n_in:n_in + n_exo]
            mask = kwargs.get("mask", None)
            if mask is None and len(args) > n_in + n_exo:
                mask = args[n_in + n_exo]
            B = inputs[0].shape[0]
            dev = inputs[0].device
            dtype = inputs[0].dtype
            if mask is None:
                mask = torch.ones((B, n_in), device=dev, dtype=dtype)
                for i, x_chan in enumerate(inputs):
                    is_zero = (x_chan.abs().sum(dim=tuple(range(1, x_chan.dim()))) == 0.0)
                    mask[:, i] = torch.where(is_zero, 0.0, 1.0)
            elif mask.dim() == 1:
                mask = mask.unsqueeze(0).expand(B, -1)

            input_branch_outputs = []
            for i, (branch, x) in enumerate(zip(self.input_branches, inputs)):
                out = checkpoint(self._run_cnn_encoder, branch, x, use_reentrant=False)
                m_i = mask[:, i].view(-1, 1, 1)
                out = out * m_i
                input_branch_outputs.append(out)

            input_seq = torch.cat(input_branch_outputs, dim=2)
            enc_in = input_seq.reshape(B, -1)
            enc_in = torch.cat([enc_in, mask], dim=1)
            context = self.encoder_mlp(enc_in)

            if len(self.exogenous_branches) > 0 and len(exogenous) > 0:
                exo_branch_outputs = [checkpoint(self._run_cnn_encoder, b, x, use_reentrant=False)
                                      for b, x in zip(self.exogenous_branches, exogenous)]
                exo_seq = torch.cat(exo_branch_outputs, dim=2)
                exo_flat = exo_seq.reshape(B, -1)
                decoder_in = torch.cat([context, exo_flat], dim=1)
            else:
                decoder_in = context

            dec_flat = self.decoder_mlp(decoder_in)
            dec_out = dec_flat.unsqueeze(1).repeat(1, self.W_out, 1)

            outputs = []
            for branch, goal_shape in zip(self.output_branches, self.output_shapes):
                out = checkpoint(self._run_cnn_decoder, branch, goal_shape, dec_out, use_reentrant=False)
                outputs.append(out)
            return outputs

    class MultiConv_FailureAware_LSTM(Base):
        def __init__(self, in_shapes, exo_shapes, out_shapes, dict_meta, D=16):
            super().__init__(in_shapes, exo_shapes, out_shapes, dict_meta, D=D)
            self.n_in = len(self.input_branches)
            old = self.encoder_lstm  # same LSTM, with room for the health flags appended to every timestep
            self.encoder_lstm = nn.LSTM(input_size=old.input_size + self.n_in, hidden_size=old.hidden_size,
                                        num_layers=old.num_layers, batch_first=True, dropout=old.dropout)

        def forward(self, *args, **kwargs):
            inputs, exogenous, mask = _resolve_mask(self, args, kwargs)

            input_branch_outputs = []
            for i, (branch, x) in enumerate(zip(self.input_branches, inputs)):
                out = checkpoint(self._run_cnn_encoder, branch, x, use_reentrant=False)
                input_branch_outputs.append(out * mask[:, i].view(-1, 1, 1))  # mask feature gating h_i * m_i
            input_seq = torch.cat(input_branch_outputs, dim=2)
            input_seq = torch.cat([input_seq, mask.unsqueeze(1).expand(-1, input_seq.shape[1], -1)], dim=2)

            enc_out, (h, c) = self.encoder_lstm(input_seq)
            decoder_input = self.start_token.to(enc_out.device).repeat(enc_out.size(0), self.W_out, 1)
            if len(self.exogenous_branches) > 0 and len(exogenous) > 0:
                exo_branch_outputs = [checkpoint(self._run_cnn_encoder, b, x, use_reentrant=False)
                                      for b, x in zip(self.exogenous_branches, exogenous)]
                decoder_input = torch.cat([decoder_input, torch.cat(exo_branch_outputs, dim=2)], dim=2)
            dec_out, _ = self.decoder_lstm(decoder_input, (h, c))

            return [checkpoint(self._run_cnn_decoder, branch, goal_shape, dec_out, use_reentrant=False)
                    for branch, goal_shape in zip(self.output_branches, self.output_shapes)]

    FailureAware = MultiConv_FailureAware_LSTM if is_lstm else MultiConv_FailureAware_MLP
    return FailureAware(in_shapes=input_shapes, exo_shapes=exogenous_shapes, out_shapes=output_shapes,
                        dict_meta=meta, D=16).to(device)


def make_transformer_model(loader, meta, shapes=None):
    """One token per sensor (115 = 15+40+18+18+12+10+1+1) in a Transformer; missing sensors are DROPPED.

    Each sensor's 5 ms series -> per-group Linear (group time lengths differ: 25/250/20) + learned sensor-ID and
    sensor-group embeddings. A missing sensor is removed from attention via key_padding_mask and from the masked-mean
    pool (never zero-filled). A learned always-valid CLS token keeps the all-sensors-lost case finite. Pooling =
    masked mean of sensor tokens + learned-query attention pooling. The head feeds the SAME decoder_mlp and 65x65
    ConvTranspose output branches as the CNN models (inherited from MultiConv_MLP)."""
    setup_imports()
    import torch
    import torch.nn as nn
    from src.multi_conv_mlp_model import MultiConv_MLP as Base, bb_factor
    from tokamark.tools.utils import get_device

    in_shapes, exo_shapes, out_shapes = shapes or _first_shapes(loader)
    d, heads, layers, p = CFG["d_model"], CFG["tx_heads"], CFG["tx_layers"], CFG["dropout"]
    assert d % heads == 0, f"d_model {d} not divisible by tx_heads {heads}"

    class SensorEmbed(nn.Module):
        """Group branch: window of one diagnostic group -> (B, N_sensors, d_model) tokens."""
        def __init__(self, shape):  # per-sample shape without batch axis, e.g. (1, 1, 25, 15) or (1, 1, 20)
            super().__init__()
            self.n = int(shape[-1]) if len(shape) >= 4 else 1
            L = int(math.prod(shape)) // self.n
            self.proj = nn.Sequential(nn.Linear(L, d), nn.GELU(), nn.Linear(d, d))

        def raw(self, x):  # (B, N, L): one row per sensor
            x = x.movedim(-1, 1) if x.dim() >= 5 else x.unsqueeze(1)
            return x.reshape(x.shape[0], self.n, -1)

        def forward(self, x):
            return self.proj(self.raw(x))

    class MultiConv_Transformer(Base):
        def __init__(self):
            super().__init__(in_shapes, exo_shapes, out_shapes, meta, D=16)
            self.input_branches = nn.ModuleList([SensorEmbed(s) for s in in_shapes])   # replaces the CNN encoders
            self.n_in = len(self.input_branches)
            self.n_sensors = sum(b.n for b in self.input_branches)
            self.register_buffer("group_id", torch.tensor([g for g, b in enumerate(self.input_branches) for _ in range(b.n)]),
                                 persistent=False)
            self.sensor_emb = nn.Embedding(self.n_sensors, d)
            self.group_emb = nn.Embedding(self.n_in, d)
            self.cls = nn.Parameter(torch.zeros(1, 1, d))
            self.query = nn.Parameter(torch.randn(1, 1, d) * 0.02)
            for e in (self.sensor_emb, self.group_emb):
                nn.init.normal_(e.weight, std=0.02)
            layer = nn.TransformerEncoderLayer(d, heads, dim_feedforward=2 * d, dropout=p, activation="gelu",
                                               batch_first=True, norm_first=True)
            self.encoder = nn.TransformerEncoder(layer, layers, norm=nn.LayerNorm(d), enable_nested_tensor=False)
            self.pool = nn.MultiheadAttention(d, heads, dropout=p, batch_first=True)
            ctx = self.D * bb_factor
            self.encoder_mlp = nn.Sequential(nn.Linear(2 * d, 2 * ctx), nn.ReLU(), nn.Linear(2 * ctx, ctx))

        def sensor_mask(self, inputs, mask):
            """(B, n_sensors) bool, True = sensor present. mask: None (infer: an exactly-zero sensor is missing, the
            fallback rule of the failure-aware CNNs), (B or 1, n_in) group health, or (B or 1, n_sensors) per sensor."""
            B = inputs[0].shape[0]
            if mask is None:
                return torch.cat([b.raw(x).abs().sum(-1) != 0 for b, x in zip(self.input_branches, inputs)], dim=1)
            if mask.dim() == 1:
                mask = mask.unsqueeze(0)
            if mask.shape[-1] == self.n_in:
                mask = mask[:, self.group_id]
            elif mask.shape[-1] != self.n_sensors:
                raise ValueError(f"mask has {mask.shape[-1]} columns; expected {self.n_in} groups or {self.n_sensors} sensors")
            return (mask > 0.5).expand(B, -1)

        def forward(self, *args, **kwargs):
            n_in, n_exo = self.n_in, len(self.exogenous_branches)
            inputs, exogenous = args[:n_in], args[n_in:n_in + n_exo]
            mask = kwargs.get("mask", None)
            if mask is None and len(args) > n_in + n_exo:
                mask = args[n_in + n_exo]
            B = inputs[0].shape[0]
            ok = self.sensor_mask(inputs, mask)
            tok = torch.cat([b(x) for b, x in zip(self.input_branches, inputs)], dim=1)
            tok = tok + self.sensor_emb.weight + self.group_emb(self.group_id)
            seq = torch.cat([self.cls.expand(B, -1, -1), tok], dim=1)
            valid = torch.cat([ok.new_ones(B, 1), ok], dim=1)
            h = self.encoder(seq, src_key_padding_mask=~valid)
            sens = h[:, 1:].masked_fill(~ok.unsqueeze(-1), 0.0)
            mean = sens.sum(1) / ok.sum(1, keepdim=True).clamp(min=1).to(h.dtype)
            att = self.pool(self.query.expand(B, -1, -1), h, h, key_padding_mask=~valid, need_weights=False)[0].squeeze(1)
            context = self.encoder_mlp(torch.cat([mean, att], dim=1))
            if n_exo > 0 and len(exogenous) > 0:
                exo = torch.cat([self._run_cnn_encoder(b, x) for b, x in zip(self.exogenous_branches, exogenous)], dim=2)
                context = torch.cat([context, exo.reshape(B, -1)], dim=1)
            dec_out = self.decoder_mlp(context).unsqueeze(1).repeat(1, self.W_out, 1)
            return [self._run_cnn_decoder(br, gs, dec_out) for br, gs in zip(self.output_branches, self.output_shapes)]

    return MultiConv_Transformer().to(get_device())


def add_latent_dropout(model, p):
    """Dropout on the latent that feeds each psi decoder branch (same knob for every model; p=0 -> no-op)."""
    if p > 0:
        import torch.nn.functional as F
        for br in model.output_branches:
            br.register_forward_pre_hook(lambda m, a: (F.dropout(a[0], p, m.training),) + tuple(a[1:]))
    return model


def count_params(model):
    return sum(q.numel() for q in model.parameters())


def make_model(loader, meta, shapes=None):
    setup_imports()
    model_type = CFG["model"].lower().strip()
    if model_type == "transformer":
        m = make_transformer_model(loader, meta, shapes)
    elif CFG["failure_aware"]:
        m = make_failure_aware_model(loader, meta, shapes)
    elif model_type == "lstm":
        from src.multi_conv_lstm_model import create_lstm_architecture
        m = create_lstm_architecture(dataloader_=loader, dict_metadata=meta, verbose=False)
    else:
        from src.multi_conv_mlp_model import create_cnn_architecture
        m = create_cnn_architecture(dataloader_=loader, dict_metadata=meta, verbose=False)
    return add_latent_dropout(m, CFG["dropout"])


# ======================================================================================
# TRAIN: resumable, deadline-aware, optional DataParallel (benchmarked)
# ======================================================================================
def atomic_save(obj, path):
    import torch
    tmp = path + ".tmp"
    torch.save(obj, tmp)
    os.replace(tmp, path)


def pick_parallel(model, loader, dev, criterion, move):
    """Return (module_to_call, label). Benchmarks 1 GPU vs DataParallel on one cached batch."""
    import torch, torch.nn as nn
    n = torch.cuda.device_count()
    if n < 2 or CFG["use_dp"] == "0":
        return model, f"single GPU ({n} visible)"
    if CFG["use_dp"] == "1":
        return nn.DataParallel(model), f"DataParallel x{n} (forced)"
    try:
        batch = first_batch_timed(loader, tag="benchmark batch")
        x, y = move(batch, dev)
        def bench(m):
            opt = torch.optim.SGD(m.parameters(), lr=0.0)  # lr 0: benchmark does not change weights
            m.train()
            for i in range(14):
                if i == 4:
                    torch.cuda.synchronize(); t = time.time()
                opt.zero_grad()
                if CFG["failure_aware"]:
                    c_mask = torch.ones((x[0].shape[0], 8), device=dev, dtype=x[0].dtype)
                    criterion(m(*x, c_mask), y).backward()
                else:
                    criterion(m(*x), y).backward()
                opt.step()
            torch.cuda.synchronize()
            return (time.time() - t) / 10
        sd = {k: v.clone() for k, v in model.state_dict().items()}
        t1 = bench(model)
        dp = nn.DataParallel(model)
        t2 = bench(dp)
        model.load_state_dict(sd)
        log(f"GPU benchmark (compute only, same batch): 1 GPU {t1 * 1e3:.0f} ms/step, DataParallel {t2 * 1e3:.0f} ms/step")
        if t2 < t1 / 1.15:
            return dp, f"DataParallel x{n} (benchmark {t1 / t2:.2f}x faster)"
        return model, "single GPU (DataParallel not clearly faster)"
    except Exception as e:  # noqa
        log("parallel benchmark failed, using single GPU:", repr(e)[:200])
        return model, "single GPU (benchmark error)"


def cmd_train(split):
    setup_imports()
    import torch
    from src.trainer import MultiOutputMSELoss, move_batch_to_device
    from utils import set_seed
    set_seed(CFG["seed"])
    P = paths(split)
    os.makedirs(P["model_dir"], exist_ok=True)
    sel = json.load(open(P["sel"]))
    run_config = dict(task=CFG["task"], model=CFG["model"], split=split, seed=CFG["seed"], batch_size=CFG["batch_size"],
                      lr=CFG["lr"], weight_decay=CFG["weight_decay"], max_steps=CFG["max_steps"],
                      validate_every=CFG["validate_every"], patience=CFG["patience"],
                      train_deadline_h=CFG["train_deadline_h"], use_dp=CFG["use_dp"],
                      failure_aware=CFG["failure_aware"], fail_aug_p=CFG["fail_aug_p"],
                      pinn_lambda=CFG["pinn_lambda"], variant=CFG["variant"], run_tag=CFG["run_tag"],
                      pinn_mode=CFG["pinn_mode"] if CFG["failure_aware"] and CFG["pinn_lambda"] > 0 else "none",
                      dropout=CFG["dropout"], d_model=CFG["d_model"] if CFG["model"] == "transformer" else None,
                      tuned=os.environ.get("TM_TUNED", ""),
                      git_revs=CFG["git_revs"],
                      shots={k: len(v) for k, v in sel["shots"].items()},
                      report=sel["report"])
    rc_path = os.path.join(P["out"], "run_config.json")
    if os.path.exists(rc_path) and not CFG["train_from_scratch"]:   # resuming: never silently switch the PINN definition
        old = json.load(open(rc_path)).get("pinn_mode", "legacy" if run_config["pinn_mode"] != "none" else "none")
        if old != run_config["pinn_mode"]:
            raise SystemExit(f"FATAL: {rc_path} was trained with pinn_mode={old}, now {run_config['pinn_mode']}; "
                             "use a new TM_RUN_TAG or set TM_TRAIN_FROM_SCRATCH=1")
    json.dump(run_config, open(rc_path, "w"), indent=1)
    log(f"run_config: {json.dumps(run_config)}")
    dev = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    log(f"device={dev}, gpus={torch.cuda.device_count()}, cpus={os.cpu_count()}")
    phase = {"p": "building dataloaders", "step": 0}

    def _heartbeat():
        while True:
            time.sleep(60)
            log(f"  heartbeat: {phase['p']} (step {phase['step']})")

    if not getattr(cmd_train, "_hb", False):   # one heartbeat thread even when `tune` calls cmd_train repeatedly
        cmd_train._hb = True
        threading.Thread(target=_heartbeat, daemon=True).start()
    t_train0 = time.time()
    loaders, meta = build_data(split, ["train", "val"])
    set_flux_loop_stats(meta)
    phase["p"] = "building model"
    t_model = time.time()
    log("building model (shape inference walks the dataset for one sample)...")
    model = make_model(loaders["train"], meta).to(dev)
    n_params = count_params(model)
    log(f"model built in {time.time() - t_model:.1f}s, {n_params / 1e6:.3f}M params")
    crit = MultiOutputMSELoss()

    # ---- fail-fast smoke check (a few seconds): shapes, NaNs, forward/backward ----
    phase["p"] = "first batch (smoke)"
    b = first_batch_timed(loaders["train"], tag="first train batch")
    sid, wid, xs, ys = b
    x, y = move_batch_to_device(b, dev)
    if CFG["failure_aware"]:
        clean_mask = torch.ones((x[0].shape[0], 8), device=dev, dtype=x[0].dtype)
        out = model(*x, clean_mask)
    else:
        out = model(*x)
    log("smoke: input shapes", [tuple(t.shape) for t in x], "| target shapes", [tuple(t.shape) for t in y],
        "| output shapes", [tuple(o.shape) for o in out])
    assert all(o.shape == t.shape for o, t in zip(out, y)), "output/target shape mismatch"
    assert all(torch.isfinite(t).all() for t in x), "non-finite inputs after collate"
    l = crit(out, y); l.backward()
    assert torch.isfinite(l), "non-finite initial loss"
    log(f"smoke OK: initial loss {l.item():.4f}")
    model.zero_grad(set_to_none=True)

    opt = torch.optim.Adam(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=3, min_lr=1e-6)
    st = dict(step=0, best=float("inf"), no_improve=0, epoch=0, oom=0,
              hist=dict(step=[], train_loss=[], val_loss=[], lr=[], t=[], mse=[], pinn=[]))
    last = os.path.join(P["model_dir"], "last.pt")
    if os.path.exists(last) and not CFG["train_from_scratch"]:
        ck = torch.load(last, map_location=dev, weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); st = ck["state"]
        if "sched" in ck and ck["sched"] is not None:
            sched.load_state_dict(ck["sched"])
        st.setdefault("oom", 0)
        st["hist"].setdefault("t", [])
        st["hist"].setdefault("lr", [])
        st["hist"].setdefault("mse", [])
        st["hist"].setdefault("pinn", [])
        log(f"RESUMED from step {st['step']} (best val {st['best']:.5f})")
    elif CFG["train_from_scratch"] and os.path.exists(last):
        log("TM_TRAIN_FROM_SCRATCH=1: ignoring existing checkpoint, training from step 0")
    runner, label = pick_parallel(model, loaders["train"], dev, crit, move_batch_to_device) if st["step"] == 0 else (model, "single GPU (resumed)")
    log("parallel mode:", label)

    @torch.no_grad()
    def validate():
        phase["p"] = "validation"
        runner.eval(); tot = n = 0
        for vb in loaders["val"]:
            if vb is None: continue
            vx, vy = move_batch_to_device(vb, dev)
            bs = vy[0].shape[0]
            if CFG["failure_aware"]:
                clean_mask = torch.ones((bs, 8), device=dev, dtype=vx[0].dtype)
                lv = crit(runner(*vx, clean_mask), vy)
            else:
                lv = crit(runner(*vx), vy)
            tot += lv.item() * bs; n += bs
        phase["p"] = "training loop"
        if n == 0:
            raise RuntimeError("validation loader produced zero samples - check selection.json / val shots")
        return tot / n

    def save_last():
        sched_st = sched.state_dict() if sched is not None else None
        atomic_save(dict(model=model.state_dict(), opt=opt.state_dict(), sched=sched_st, state=st), last)
        atomic_save(st["hist"], os.path.join(P["model_dir"], "training_history.pt"))
        json.dump(st["hist"], open(os.path.join(P["model_dir"], "training_history.json"), "w"))

    deadline = T0 + CFG["train_deadline_h"] * 3600
    phase["p"] = "training loop"
    it, run_loss, errs, bad, oom, t_last_ck, t_tr = iter(loaders["train"]), [], 0, 0, 0, time.time(), time.time()
    stop_reason = "max_steps"
    run_parts = []  # (mse, pinn) per step, for the loss-component plot
    while st["step"] < CFG["max_steps"]:
        if time.time() > deadline:
            stop_reason = "time deadline"; break
        phase["p"] = f"waiting for batch (step {st['step']})"
        try:
            batch = next(it)
        except StopIteration:
            it = iter(loaders["train"]); st["epoch"] += 1; run_loss = []; run_parts = []; log(f"epoch {st['epoch']} done"); continue
        except Exception as e:  # noqa  (e.g. a corrupt shot inside a worker)
            errs += 1; log(f"data error #{errs}: {repr(e)[:200]}")
            if errs > 20: raise
            it = iter(loaders["train"]); continue
        if batch is None: continue
        phase["p"] = f"train step {st['step'] + 1}"
        try:
            runner.train()
            x, y = move_batch_to_device(batch, dev)
            opt.zero_grad(set_to_none=True)
            loss, loss_mse, loss_fl = train_loss(runner, crit, x, y)
            if loss_fl is not None:
                run_parts.append((loss_mse.item(), loss_fl.item()))
            if not torch.isfinite(loss):
                bad += 1; log(f"non-finite loss, skipping (#{bad})")
                if bad > 200: raise RuntimeError("too many non-finite losses")
                continue
            bad = 0
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()
        except torch.cuda.OutOfMemoryError:
            oom += 1; st["oom"] = oom
            torch.cuda.empty_cache()
            log(f"CUDA OOM at step {st['step']} (#{oom}) - batch skipped; current batch_size={CFG['batch_size']}")
            if oom >= 3:
                raise RuntimeError(
                    f"CUDA out of memory {oom}x with batch_size={CFG['batch_size']} on this GPU. "
                    "Re-run the train cell with a smaller TM_BATCH in the config cell (e.g. 256 or 128); "
                    f"training will resume from {last} without losing progress.")
            continue
        st["step"] += 1; run_loss.append(loss.item()); phase["step"] = st["step"]
        if st["step"] <= 10 or st["step"] % 20 == 0:
            el = time.time() - t_tr; t_tr = time.time()
            log(f"step {st['step']} loss {sum(run_loss) / len(run_loss):.4f} "
                f"({(CFG['batch_size'] if st['step'] <= 10 else 20 * CFG['batch_size']) / el:.0f} samples/s)")
        if st["step"] == 1:
            save_last()  # keep progress even if the session dies early
        if st["step"] % CFG["validate_every"] == 0:
            v = validate(); h = st["hist"]
            cur_lr = opt.param_groups[0]["lr"]
            for k, val in zip(("step", "train_loss", "val_loss", "lr", "t"), (st["step"], sum(run_loss) / len(run_loss), v, cur_lr, time.time() - T0)):
                h[k].append(val)
            if run_parts:
                h["mse"].append(sum(a for a, _ in run_parts) / len(run_parts))
                h["pinn"].append(sum(b for _, b in run_parts) / len(run_parts))
            sched.step(v)
            if v < st["best"]:
                st["best"], st["no_improve"] = v, 0
                atomic_save(model.state_dict(), os.path.join(P["model_dir"], "best_model.pt"))
            else:
                st["no_improve"] += 1
            log(f"VAL step {st['step']} val_loss {v:.5f} best {st['best']:.5f} no_improve {st['no_improve']}/{CFG['patience']}")
            save_last(); t_last_ck = time.time()
            if st["no_improve"] >= CFG["patience"]:
                stop_reason = "early stopping (official patience rule)"; break
        elif time.time() - t_last_ck > CFG["ckpt_every_s"]:
            save_last(); t_last_ck = time.time()
    v = validate()
    if v < st["best"] or not os.path.exists(os.path.join(P["model_dir"], "best_model.pt")):
        st["best"] = v; atomic_save(model.state_dict(), os.path.join(P["model_dir"], "best_model.pt"))
    save_last()
    summary = dict(stop_reason=stop_reason, steps=st["step"], epochs=st["epoch"], best_val=st["best"],
                   parallel=label, cuda_oom_events=st.get("oom", 0), data_errors=errs,
                   duration_min=round((time.time() - T0) / 60, 1),          # since session start (includes prep)
                   train_duration_min=round((time.time() - t_train0) / 60, 1),
                   model=CFG["model"], variant=CFG["variant"], run_tag=CFG["run_tag"], params=n_params,
                   model_dir=P["model_dir"], **{k: run_config[k] for k in ("task", "split", "seed", "batch_size", "lr",
                                                                           "weight_decay", "dropout", "d_model", "pinn_lambda",
                                                                           "fail_aug_p", "pinn_mode", "tuned", "validate_every", "patience",
                                                                           "git_revs", "shots", "report")})
    json.dump(summary, open(os.path.join(P["out"], "train_summary.json"), "w"), indent=1)
    log(f"TRAIN DONE ({stop_reason}) at step {st['step']}, best val {st['best']:.5f}; summary -> train_summary.json")
    return summary


# ======================================================================================
# TUNE: small random search with the SAME training loop (cmd_train), validation split only
# ======================================================================================
TUNE_KEYS = ("lr", "weight_decay", "dropout", "d_model")


def cmd_tune(split):
    """TM_TUNE_TRIALS trials of TM_TUNE_STEPS steps each over lr / weight decay / dropout (+ d_model for the transformer).

    Every model sees the same sampled (lr, wd, dropout) list (TM_TUNE_SEED), so budget and search space are equal.
    Only train/val shots are used (the test split is never touched). Writes
    out/<split>/tune/<model>_<variant>/tuning.json with trials + `chosen`; point TM_TUNED at that file for the final runs."""
    P = paths(split)
    tdir = os.path.join(P["base"], "tune", f"{CFG['model'].lower().strip()}_{CFG['variant']}")
    os.makedirs(tdir, exist_ok=True)
    rng, rng_d = random.Random(CFG["tune_seed"]), random.Random(CFG["tune_seed"] + 1)
    plan = [dict(lr=10 ** rng.uniform(-4.5, -3.0), weight_decay=10 ** rng.uniform(-6.0, -3.0), dropout=rng.choice([0.0, 0.1, 0.2]),
                 d_model=rng_d.choice([64, 128])) for _ in range(CFG["tune_trials"])]
    use_dm = CFG["model"].lower().strip() == "transformer"
    keys = [k for k in TUNE_KEYS if k != "d_model" or use_dm]
    saved = {k: CFG[k] for k in ("seed", "run_tag", "max_steps", "validate_every", "patience", "train_from_scratch", *TUNE_KEYS)}
    trials = []
    for i, t in enumerate(plan):
        hpar = {k: t[k] for k in keys}
        tag = f"tune/{CFG['model'].lower().strip()}_{CFG['variant']}/t{i}"
        CFG.update(hpar, seed=CFG["tune_seed"], run_tag=tag, max_steps=CFG["tune_steps"], validate_every=CFG["tune_steps"] + 1,
                   patience=10 ** 9, train_from_scratch=True)
        rec = dict(trial=i, **hpar)
        tout = paths(split)["out"]
        sp = os.path.join(tout, "train_summary.json")
        try:
            if os.path.exists(sp):                      # finished in an earlier session: reuse
                summ = json.load(open(sp))
            else:
                summ = cmd_train(split)
            rec.update(val_loss=summ["best_val"], steps=summ["steps"], params=summ["params"])
        except Exception as e:  # noqa  (e.g. OOM for d_model=128): record and carry on
            rec.update(val_loss=None, error=repr(e)[:200])
        finally:
            CFG.update(saved)
        shutil.rmtree(os.path.join(tout, "model"), ignore_errors=True)   # keep the json, drop the checkpoints
        trials.append(rec)
        log(f"TUNE trial {i}: {json.dumps(rec)}")
    ok = [r for r in trials if r.get("val_loss") is not None and math.isfinite(r["val_loss"])]
    if not ok:
        raise SystemExit("FATAL: every tuning trial failed; see tuning log above")
    best = min(ok, key=lambda r: r["val_loss"])
    out = dict(model=CFG["model"], variant=CFG["variant"], split=split, selection="lowest validation loss (validation split only)",
               budget=dict(trials=CFG["tune_trials"], steps_per_trial=CFG["tune_steps"], batch_size=CFG["batch_size"]),
               tune_seed=CFG["tune_seed"], trials=trials, chosen_trial=best["trial"], chosen={k: best[k] for k in keys})
    path = os.path.join(tdir, "tuning.json")
    json.dump(out, open(path, "w"), indent=1)
    log(f"TUNE DONE: chosen {json.dumps(out['chosen'])} -> {path}")
    log(f"  final runs: set TM_TUNED={path}")
    return out


# ======================================================================================
# EVAL: official evaluator on test shots, un-standardized NRMSE
# ======================================================================================
def cmd_eval(split):
    setup_imports()
    import torch
    from tokamark.evaluator import WindowMetricsAccumulator, compute_metrics
    from src.evaluator import cnn_unstd_evaluation_per_shot
    P = paths(split)
    os.makedirs(P["results"], exist_ok=True)
    if not os.path.exists(os.path.join(P["model_dir"], "best_model.pt")):
        raise SystemExit(f"FATAL: {P['model_dir']}best_model.pt not found - run train first")
    loaders, meta = build_data(split, ["test"])
    model = make_model(loaders["test"], meta)
    acc = WindowMetricsAccumulator(CFG["task"])
    cnn_unstd_evaluation_per_shot(test_dataloader=loaders["test"], config_task=meta, cnn_model=model,
                                  output_dir=P["model_dir"], accumulator=acc)
    compute_metrics(task=CFG["task"], output_dir=P["results"], window_metrics_accumulator=acc,
                    save_windows_metrics=True, save_task_metrics=True)
    sel = json.load(open(P["sel"]))
    log(f"[{split}] EVAL DONE. Results in {P['results']}")
    log(f"[{split}] shot counts used: {json.dumps(sel['report'])}")
    for f in sorted(os.listdir(P["results"])):
        if f.endswith((".json", ".yaml", ".yml")):
            print(f"--- {f}\n" + open(os.path.join(P["results"], f)).read()[:3000])
    for root, _, files in os.walk(P["results"]):
        for f in sorted(files):
            if f.endswith(".csv"):
                print(f"--- {os.path.join(root, f)}")
                with open(os.path.join(root, f)) as fh:
                    for i, line in enumerate(fh):
                        if i > 12: print("  ..."); break
                        print("  " + line.rstrip())


# ======================================================================================
# PLOT: training dashboard, reconstruction samples, LCFS overlay, error diagnostics, per-shot error
# (never blocks core results)
# ======================================================================================
_DR_CM, _DZ_CM = 180.0 / 64, 300.0 / 64          # MAST 65x65 grid: cm per pixel (R, Z)
_PSI_EXTENT = [0.2, 2.0, -1.5, 1.5]               # R [m], Z [m]; array rows = Z, columns = R


def _run_title():
    kind = {"plain": "baseline", "mask": "failure-aware", "mask_pinn": "failure-aware + PINN"}[CFG["variant"]]
    return f"TokaMark {CFG['task']} CNN+{CFG['model'].upper()} ({kind})"


def _lcfs_segments(field, level):
    """Contour vertices (col, row) of `field` at `level`; short speckle segments are dropped."""
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots()
    segs = [s for s in ax.contour(field, levels=[level]).allsegs[0] if len(s) >= 8]
    plt.close(fig)
    return segs


def _boundary_offset_cm(true, pred, frac=0.35):
    """Mean symmetric distance (cm) between the true and predicted LCFS, both cut at the TRUE field's flux level."""
    import numpy as np
    level = true.min() + frac * (true.max() - true.min())
    st, sp = _lcfs_segments(true, level), _lcfs_segments(pred, level)
    if not st or not sp:
        return None
    a = np.concatenate(st) * [_DR_CM, _DZ_CM]
    b = np.concatenate(sp) * [_DR_CM, _DZ_CM]
    d = np.sqrt(((a[:, None, :] - b[None, :, :]) ** 2).sum(-1))
    return float(0.5 * (d.min(1).mean() + d.min(0).mean()))


def _fig_loss_dashboard(h, path, title):
    import numpy as np
    import matplotlib.pyplot as plt
    steps, tr, va = np.array(h["step"]), np.array(h["train_loss"]), np.array(h["val_loss"])
    best_i = int(np.argmin(va))
    fig, ax = plt.subplots(2, 2, figsize=(12, 7.5))
    a = ax[0, 0]
    a.plot(steps, tr, color="#4C78A8", lw=1.8, label="train (running mean)")
    a.plot(steps, va, color="#E4572E", lw=1.8, label="validation")
    a.axvline(steps[best_i], color="grey", ls="--", lw=1)
    a.scatter([steps[best_i]], [va[best_i]], color="#E4572E", zorder=5, s=45)
    a.annotate(f"best val {va[best_i]:.4f}\nstep {int(steps[best_i])}", (steps[best_i], va[best_i]),
               textcoords="offset points", xytext=(12, -42), fontsize=8,
               arrowprops=dict(arrowstyle="->", color="grey", lw=0.8))
    a.set_title("Train vs validation loss"); a.set_xlabel("step"); a.set_ylabel("MultiOutputMSELoss (standardised)")
    a.legend(); a.grid(alpha=0.3)
    a = ax[0, 1]
    a.fill_between(steps, 0, va - tr, color="#9467BD", alpha=0.35)
    a.plot(steps, va - tr, color="#9467BD", lw=1.5)
    a.axvline(steps[best_i], color="grey", ls="--", lw=1)
    a.set_title("Generalisation gap (val - train)"); a.set_xlabel("step"); a.set_ylabel("loss gap"); a.grid(alpha=0.3)
    a = ax[1, 0]
    if h.get("lr"):
        a.semilogy(steps[:len(h["lr"])], h["lr"], color="#2CA02C", drawstyle="steps-post", lw=1.8)
    a.set_title("Learning rate (ReduceLROnPlateau)"); a.set_xlabel("step"); a.set_ylabel("lr"); a.grid(alpha=0.3, which="both")
    a = ax[1, 1]
    if h.get("mse") and h.get("pinn") and len(h["mse"]) == len(steps):
        a.plot(steps, h["mse"], color="#4C78A8", lw=1.8, label="MSE term")
        a.set_ylabel("MSE term", color="#4C78A8"); a.set_xlabel("step")
        b = a.twinx()
        b.plot(steps, h["pinn"], color="#F58518", lw=1.8, label="flux-loop PINN term (unweighted)")
        b.set_ylabel("PINN term", color="#F58518")
        a.set_title("Loss components (train, running mean)")
    elif h.get("t"):
        a.plot(steps[:len(h["t"])], np.array(h["t"]) / 60, color="#17BECF", lw=1.8)
        a.set_title("Wall-clock time"); a.set_xlabel("step"); a.set_ylabel("minutes since session start")
    a.grid(alpha=0.3)
    fig.suptitle(title, fontsize=12)
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    fig.savefig(path, dpi=130)
    plt.close(fig)


def _fig_samples(true, pred, labels, path, title):
    """true/pred: (n, H, W) in Wb. One row per window: true | pred | |error|."""
    import numpy as np
    import matplotlib.pyplot as plt
    n = len(true)
    fig, axes = plt.subplots(n, 3, figsize=(12, 3.6 * n), squeeze=False)
    for i in range(n):
        err = np.abs(pred[i] - true[i])
        lo, hi = min(true[i].min(), pred[i].min()), max(true[i].max(), pred[i].max())
        for j, (img, ttl, cm, kw) in enumerate((
                (true[i], "True psi", "viridis", dict(vmin=lo, vmax=hi)),
                (pred[i], "Predicted psi", "viridis", dict(vmin=lo, vmax=hi)),
                (err, f"|Error|  (MAE {err.mean():.4f} Wb, max {err.max():.4f})", "magma", {}))):
            im = axes[i][j].imshow(img, cmap=cm, origin="lower", extent=_PSI_EXTENT, aspect="auto", **kw)
            axes[i][j].set_title(f"{ttl}\n{labels[i]}" if j == 0 else ttl, fontsize=9)
            axes[i][j].set_xlabel("R [m]", fontsize=8); axes[i][j].set_ylabel("Z [m]", fontsize=8)
            fig.colorbar(im, ax=axes[i][j], fraction=0.046)
    fig.suptitle(title, fontsize=12)
    fig.tight_layout(rect=(0, 0, 1, 0.97))
    fig.savefig(path, dpi=130)
    plt.close(fig)


def _fig_boundary(true, pred, labels, path, title):
    import numpy as np
    import matplotlib.pyplot as plt
    n = len(true)
    fig, axes = plt.subplots(1, n, figsize=(5.2 * n, 5.2), squeeze=False)
    for i in range(n):
        a = axes[0][i]
        level = true[i].min() + 0.35 * (true[i].max() - true[i].min())
        a.imshow(true[i], cmap="Greys", origin="lower", extent=_PSI_EXTENT, aspect="auto", alpha=0.55)
        x = np.linspace(_PSI_EXTENT[0], _PSI_EXTENT[1], true[i].shape[1])
        z = np.linspace(_PSI_EXTENT[2], _PSI_EXTENT[3], true[i].shape[0])
        a.contour(x, z, true[i], levels=[level], colors="#1F77B4", linewidths=2.2)
        a.contour(x, z, pred[i], levels=[level], colors="#D62728", linewidths=1.8, linestyles="--")
        off = _boundary_offset_cm(true[i], pred[i])
        a.set_title(f"{labels[i]}\nLCFS offset: " + (f"{off:.2f} cm" if off is not None else "n/a"), fontsize=9)
        a.set_xlabel("R [m]"); a.set_ylabel("Z [m]")
    axes[0][0].plot([], [], color="#1F77B4", lw=2.2, label="true LCFS")
    axes[0][0].plot([], [], color="#D62728", lw=1.8, ls="--", label="predicted LCFS")
    axes[0][0].legend(loc="upper right", fontsize=8)
    fig.suptitle(title + " - last closed flux surface (35% level of the true field)", fontsize=11)
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    fig.savefig(path, dpi=130)
    plt.close(fig)


def _fig_error_diagnostics(err_map, win_mae, sc_t, sc_p, path, title):
    import numpy as np
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(2, 2, figsize=(12, 9))
    a = ax[0, 0]
    im = a.imshow(err_map, cmap="magma", origin="lower", extent=_PSI_EXTENT, aspect="auto")
    fig.colorbar(im, ax=a, fraction=0.046, label="mean |error| [Wb]")
    fl = np.array(FLUX_LOOP_COORDS)
    a.scatter(_PSI_EXTENT[0] + fl[:, 0] / 64 * 1.8, _PSI_EXTENT[2] + fl[:, 1] / 64 * 3.0, s=22, facecolors="none",
              edgecolors="cyan", linewidths=1, label="15 flux loops (PINN)")
    a.set_title("Mean absolute error over all test windows"); a.set_xlabel("R [m]"); a.set_ylabel("Z [m]")
    a.legend(loc="upper right", fontsize=8)
    a = ax[0, 1]
    a.hist(win_mae, bins=40, color="#4C78A8", alpha=0.9)
    a.axvline(np.mean(win_mae), color="#E4572E", lw=1.8, label=f"mean {np.mean(win_mae):.4f}")
    a.axvline(np.median(win_mae), color="k", ls="--", lw=1.5, label=f"median {np.median(win_mae):.4f}")
    a.set_title(f"Per-window MAE ({len(win_mae)} windows)"); a.set_xlabel("MAE [Wb]"); a.set_ylabel("windows")
    a.legend(); a.grid(alpha=0.3)
    a = ax[1, 0]
    hb = a.hexbin(sc_t, sc_p, gridsize=60, bins="log", cmap="viridis", mincnt=1)
    lim = [min(sc_t.min(), sc_p.min()), max(sc_t.max(), sc_p.max())]
    a.plot(lim, lim, color="white", lw=1.2, ls="--")
    r2 = 1.0 - np.sum((sc_t - sc_p) ** 2) / max(np.sum((sc_t - sc_t.mean()) ** 2), 1e-12)
    a.set_title(f"Predicted vs true psi (R2 = {r2:.4f})"); a.set_xlabel("true psi [Wb]"); a.set_ylabel("predicted psi [Wb]")
    fig.colorbar(hb, ax=a, fraction=0.046, label="log10 count")
    a = ax[1, 1]
    fl_err = np.array([err_map[z, r] for r, z in FLUX_LOOP_COORDS])
    a.bar(range(len(fl_err)), fl_err, color="#F58518")
    a.axhline(err_map.mean(), color="k", ls="--", lw=1.3, label=f"grid-wide mean {err_map.mean():.4f}")
    a.set_title("Error at the 15 flux-loop pixels (PINN anchor points)")
    a.set_xlabel("flux loop #"); a.set_ylabel("mean |error| [Wb]"); a.legend(); a.grid(alpha=0.3, axis="y")
    fig.suptitle(title, fontsize=12)
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    fig.savefig(path, dpi=130)
    plt.close(fig)


def _fig_per_shot(csv_path, path, title):
    import csv as _csv
    import numpy as np
    import matplotlib.pyplot as plt
    rows = list(_csv.DictReader(open(csv_path)))
    by_shot = {}
    for r in rows:
        by_shot.setdefault(r["shot_id"], []).append((int(r["window_index"]), float(r["MAE"])))
    order = sorted(by_shot, key=lambda s: np.mean([m for _, m in by_shot[s]]))
    means = [float(np.mean([m for _, m in by_shot[s]])) for s in order]
    overall = float(np.mean([float(r["MAE"]) for r in rows]))
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    a = ax[0]
    a.bar(range(len(order)), means, color=["#E4572E" if m > overall else "#4C78A8" for m in means])
    a.axhline(overall, color="k", ls="--", lw=1.3, label=f"all windows {overall:.4f}")
    a.set_xticks(range(len(order))); a.set_xticklabels(order, rotation=90, fontsize=6)
    a.set_title(f"Mean MAE per test shot ({len(order)} shots, red = above average)")
    a.set_ylabel("MAE [Wb]"); a.legend(); a.grid(alpha=0.3, axis="y")
    a = ax[1]
    grid = np.linspace(0, 1, 21)
    curves = []
    for s in by_shot:
        w = sorted(by_shot[s])
        if len(w) < 5:
            continue
        pos = np.linspace(0, 1, len(w))
        curves.append(np.interp(grid, pos, [m for _, m in w]))
    if curves:
        c = np.array(curves)
        a.fill_between(grid, np.percentile(c, 10, axis=0), np.percentile(c, 90, axis=0), color="#4C78A8", alpha=0.25,
                       label="10-90th percentile over shots")
        a.plot(grid, np.median(c, axis=0), color="#4C78A8", lw=2, label="median shot")
    a.set_title("Error along the discharge (normalised time)"); a.set_xlabel("normalised time in shot"); a.set_ylabel("window MAE [Wb]")
    a.legend(); a.grid(alpha=0.3)
    fig.suptitle(title, fontsize=12)
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    fig.savefig(path, dpi=130)
    plt.close(fig)


def cmd_plot(split):
    setup_imports()
    import numpy as np
    import torch
    import matplotlib
    matplotlib.use("Agg")
    P = paths(split)
    os.makedirs(P["figures"], exist_ok=True)
    title = f"{_run_title()} - {split} split, seed {CFG['seed']}"

    # (a) training dashboard - from the saved history
    try:
        h = torch.load(os.path.join(P["model_dir"], "training_history.pt"), map_location="cpu", weights_only=False)
        _fig_loss_dashboard(h, os.path.join(P["figures"], "loss_curves.png"), title)
        log(f"saved loss_curves.png ({len(h['step'])} validation points)")
    except Exception as e:  # noqa
        log("loss-curve plot failed:", repr(e)[:200])

    # (b)-(d) run the best model over the test windows once and feed the reconstruction figures
    try:
        from src.trainer import move_batch_to_device
        dev = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        loaders, meta = build_data(split, ["test"])
        model = make_model(loaders["test"], meta).to(dev)
        model.load_state_dict(torch.load(os.path.join(P["model_dir"], "best_model.pt"), map_location=dev, weights_only=False))
        model.eval()
        ps = meta["output"]["equilibrium-psi"]
        mu, sd = float(ps["mean"]), float(ps["std"])
        err_sum, n_win, win_mae, sc_t, sc_p, first = None, 0, [], [], [], None
        rng = np.random.default_rng(0)
        for bi, batch in enumerate(loaders["test"]):
            if batch is None:
                continue
            if bi >= 8:
                break
            x, y = move_batch_to_device(batch, dev)
            with torch.no_grad():
                yp = model(*x)
            t = y[0][:, 0].float().cpu().numpy() * sd + mu
            p = yp[0][:, 0].float().cpu().numpy() * sd + mu
            e = np.abs(p - t)
            err_sum = e.sum(0) if err_sum is None else err_sum + e.sum(0)
            n_win += len(e)
            win_mae.extend(e.mean((1, 2)).tolist())
            pick = rng.choice(t.size, size=min(20000, t.size), replace=False)
            sc_t.append(t.ravel()[pick]); sc_p.append(p.ravel()[pick])
            if first is None:
                first = (t, p, [int(s) for s in batch[0]])
        if n_win == 0:
            raise RuntimeError("no test windows")
        win_mae = np.array(win_mae)
        t, p, sids = first
        wm = np.abs(p - t).mean((1, 2))
        pick = [int(np.argmin(wm)), int(np.argsort(wm)[len(wm) // 2]), int(np.argmax(wm))]
        labels = [f"{nm}: shot {sids[i]}, MAE {wm[i]:.4f} Wb" for nm, i in zip(("best", "median", "worst"), pick)]
        _fig_samples(t[pick], p[pick], labels, os.path.join(P["figures"], "psi_reconstruction_samples.png"), title)
        log("saved psi_reconstruction_samples.png")
        _fig_boundary(t[pick], p[pick], labels, os.path.join(P["figures"], "boundary_overlay.png"), title)
        log("saved boundary_overlay.png")
        _fig_error_diagnostics(err_sum / n_win, win_mae, np.concatenate(sc_t), np.concatenate(sc_p),
                               os.path.join(P["figures"], "test_error_diagnostics.png"), title)
        log(f"saved test_error_diagnostics.png ({n_win} test windows)")
    except Exception as e:  # noqa
        log("reconstruction figures failed (core results unaffected):", repr(e)[:300])

    # (e) per-shot error from the official evaluator's window metrics
    try:
        wm_csv = os.path.join(P["results"], CFG["task"], "windows_metrics.csv")
        _fig_per_shot(wm_csv, os.path.join(P["figures"], "per_shot_error.png"), title)
        log("saved per_shot_error.png")
    except Exception as e:  # noqa
        log("per-shot figure failed (run eval first):", repr(e)[:200])
    log(f"figures in {P['figures']}")


# ======================================================================================
# SUMMARY: one human-readable recap of the whole run
# ======================================================================================
def cmd_summary(split):
    setup_imports()
    import csv as _csv
    P = paths(split)
    lines = [f"# TokaMark {CFG['task']} baseline run - {split} split", ""]
    def add_json(fname, title):
        p = P["sel"] if fname == "selection.json" else os.path.join(P["out"], fname)
        if os.path.exists(p):
            lines.append(f"## {title}")
            lines.append("```json")
            lines.append(json.dumps(json.load(open(p)), indent=1))
            lines.append("```")
        else:
            lines.append(f"## {title}\n\nMISSING: {p}")
        lines.append("")
    add_json("selection.json", "Data subset & split (official membership, budget-limited selection)")
    add_json("run_config.json", "Run configuration")
    add_json("train_summary.json", "Training summary")
    lines.append("## Evaluation (official evaluator, un-standardised NRMSE)")
    tm = os.path.join(P["results"], CFG["task"], "task_metrics.csv")
    if os.path.exists(tm):
        rows = list(_csv.DictReader(open(tm)))
        for r in rows:
            name = r["feature_name"]
            lines.append(f"- **{name}**: NRMSE={float(r['NRMSE_mean']):.4f} (+/-{float(r['NRMSE_std_pop']):.4f}), "
                         f"RMSE={float(r['RMSE_mean']):.5f}, MAE={float(r['MAE_mean']):.5f}, n_shots={r['n_shots']}")
    else:
        lines.append(f"MISSING: {tm} (eval not run?)")
    lines.append("")
    lines.append("## Artifacts")
    for label, p in (("selection", P["sel"]), ("run config", os.path.join(P["out"], "run_config.json")),
                     ("train summary", os.path.join(P["out"], "train_summary.json")),
                     ("best model", os.path.join(P["model_dir"], "best_model.pt")),
                     ("resume checkpoint", os.path.join(P["model_dir"], "last.pt")),
                     ("history", os.path.join(P["model_dir"], "training_history.pt")),
                     ("results dir", P["results"]), ("figures dir", P["figures"])):
        lines.append(f"- {label}: `{p}` {'OK' if os.path.exists(p) else '**MISSING**'}")
    text = "\n".join(lines)
    with open(os.path.join(P["out"], "summary.md"), "w") as fh:
        fh.write(text)
    print(text)
    log(f"summary -> {os.path.join(P['out'], 'summary.md')}")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("cmd", choices=["prep", "train", "tune", "eval", "plot", "summary"])
    ap.add_argument("--split", default="random", choices=["random", "temporal"])
    a = ap.parse_args()
    {"prep": cmd_prep, "train": cmd_train, "tune": cmd_tune, "eval": cmd_eval, "plot": cmd_plot, "summary": cmd_summary}[a.cmd](a.split)


In [ ]:
%%writefile /kaggle/working/robustness.py
#!/usr/bin/env python
"""TokaMark task_1-3 sensor-robustness harness: corrupt | eval-robust | report.

Stress-tests a *trained, frozen* task_1-3 model under the six sensor-failure
scenarios of Gupta (arXiv 2607.11915), on the TEST split only, with optional
imputation. Reuses the exact pipeline helpers (build_data / make_model) and the
official un-standardise + hierarchical-metric protocol so the CLEAN baseline
reproduces the eval-time NRMSE before any corruption is trusted.

Commands:
  python robustness.py corrupt     --split random     # build corrupted test loaders + manifest
  python robustness.py eval-robust --split random     # run the sweep, save metrics
  python robustness.py report      --split random     # figures + robustness.md

All settings come from TM_* env vars (same CFG contract as pipeline.py). Robustness
knobs:
  TM_ROB_SCENARIOS  comma list; default: random_dropout,channel_ablation,category_abl,
                    temporal_gap,group_failure,disruption_proximate  (or 'all')
  TM_ROB_IMPUTE     zero,mean,ffill (comma list); default: zero,mean,ffill
  TM_ROB_SEED       RNG seed for corruption (default TM_SEED)
  TM_ROB_CLEAN_ONLY 1 -> only the clean pass (sanity check), skip corruption
  TM_ROB_BATCH      forward batch size (default TM_BATCH)
  TM_ROB_PHYS_STRIDE  every Nth test window also gets the Module 1 physics metrics (boundary / axis / X-point
                    error in cm, wall-alarm rates), stored as "PHYS" in robust_metrics.json (default 100; 0 = off)
"""
import os, sys, json, time, math, argparse
import numpy as np

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))

import pipeline as P
from pipeline import CFG, T0, log, setup_imports, build_data, make_model

# Module 1 (physics metrics in cm) is optional here: it lives in ../modules/ in the repo, next to this file on Kaggle (the notebook writes it there).
sys.path.insert(0, os.path.join(os.path.dirname(os.path.dirname(os.path.abspath(__file__))), "modules"))
try:
    from physics_evaluation import evaluate_window_physics, aggregate_physics_metrics
except ImportError:  # e.g. the Kaggle notebook, which only writes pipeline.py and this file
    evaluate_window_physics = aggregate_physics_metrics = None
try:  # Module 1 v2: EFIT truth per (shot, window); without it the physics sample falls back to the heuristic mode
    from efit_truth import TruthIndex
except ImportError:
    TruthIndex = None
try:
    from aggregate_seeds import bootstrap_ci
except ImportError:
    bootstrap_ci = None

_TRUTH = {}  # split -> TruthIndex (built once per process)


def get_truth(split):
    """TruthIndex over the split's data dir with the official outlier yaml (same files build_data uses)."""
    if TruthIndex is None:
        return None
    if split not in _TRUTH:
        _TRUTH[split] = TruthIndex(P.paths(split)["data"], outlier_yaml=P.split_files(split)[2])
    return _TRUTH[split]

TASK = CFG["task"]

SIGNALS = [
    "magnetics/flux_loop_flux",
    "magnetics/b_field_pol_probe_ccbv_field",
    "magnetics/b_field_pol_probe_obr_field",
    "magnetics/b_field_pol_probe_obv_field",
    "magnetics/b_field_tor_probe_saddle_voltage",
    "pf_active/coil_current",
    "pf_active/solenoid_current",
    "summary/ip",
]
SIGNAL_SHORT = ["flux_loop", "ccbv", "obr", "obv", "saddle_v", "coil_i", "solenoid_i", "ip"]

CATEGORY_GROUPS = {
    "flux_loop": [0],
    "pickup_fields": [1, 2, 3],
    "saddle_voltage": [4],
    "active_coils": [5],
    "solenoid": [6],
    "plasma_current": [7],
}
CORRELATED_GROUPS = {
    "magnetic_probes": [1, 2, 3],
    "power_supply": [5, 6],
    "magnetics_all": [0, 1, 2, 3, 4],
}

SCENARIOS = ["random_dropout", "channel_ablation", "category_abl",
             "temporal_gap", "group_failure", "disruption_proximate"]
SEVERITIES = {
    "random_dropout": [0.10, 0.25, 0.50],
    "channel_ablation": [1, 3, 6],
    "category_abl": [1],
    "temporal_gap": [0.20, 0.40, 0.60],
    "group_failure": [1],
    "disruption_proximate": [0.10, 0.25, 0.50],
}
IMPUTE_MODES = ("zero", "mean", "ffill")


def envf(k, d):
    return type(d)(os.environ.get(k, d))


def rob_cfg():
    scen = os.environ.get("TM_ROB_SCENARIOS", "all")
    if scen.strip() == "all":
        scen = SCENARIOS
    else:
        scen = [s.strip() for s in scen.split(",") if s.strip()]
    imp = [m.strip() for m in os.environ.get("TM_ROB_IMPUTE", "zero,mean,ffill").split(",") if m.strip()]
    bad = [s for s in scen if s not in SCENARIOS]
    bad += [m for m in imp if m not in IMPUTE_MODES]
    if bad:
        raise SystemExit(f"FATAL: unknown robustness options: {bad}")
    return dict(
        scenarios=scen,
        imputes=imp,
        seed=envf("TM_ROB_SEED", CFG["seed"]),
        clean_only=os.environ.get("TM_ROB_CLEAN_ONLY", "0") == "1",
        batch=envf("TM_ROB_BATCH", CFG["batch_size"]),
        phys_stride=envf("TM_ROB_PHYS_STRIDE", 4),  # ~1 in 4 windows (hash-selected, same set in every pass); 0 = off
        resume=os.environ.get("TM_ROB_RESUME", "0") == "1",
    )


def rob_paths(split):
    pp = P.paths(split)  # honours TM_RUN_TAG: out/<split>/<tag>/ (same dir load_test_loader_and_model reads the model from)
    out = pp["out"]
    rdir = os.environ.get("TM_ROB_DIR") or os.path.join(out, "robustness")
    return dict(
        dir=rdir,
        manifest=os.path.join(rdir, "robust_manifest.json"),
        metrics=os.path.join(rdir, "robust_metrics.json"),
        windows=os.path.join(rdir, "robust_windows.csv"),
        figures=os.path.join(rdir, "figures"),
        report=os.path.join(rdir, "robustness.md"),
        model_dir=pp["model_dir"],
    )


# ======================================================================================
# CORRUPTION PRIMITIVES  (operate on a per-channel torch tensor x: (B, W, C, L))
# ======================================================================================
def _as_np(t):
    return t.detach().cpu().numpy() if hasattr(t, "detach") else np.asarray(t)


def _mask_random_dropout(arr, sev, rng):
    """Zero a fraction `sev` of the *finite, non-zero* values (Gupta's definition).

    Returns (masked_array, hole_mask) where hole_mask marks exactly the values we zeroed.
    """
    m = np.isfinite(arr) & (arr != 0)
    idx = np.flatnonzero(m.ravel())
    out = arr.copy()
    holes = np.zeros_like(arr, dtype=bool)
    if idx.size == 0:
        return out, holes
    k = int(round(sev * idx.size))
    drop = rng.choice(idx, size=min(k, idx.size), replace=False)
    out.ravel()[drop] = 0.0
    holes.ravel()[drop] = True
    return out, holes


def _time_axis(arr):
    """Time axis of a collated branch: profile branches are (B, 1, 1, T, n_sensors) -> -2; scalar branches (B, 1, 1, T) -> -1."""
    return -2 if arr.ndim >= 5 else -1


def _mask_temporal_gap(arr, frac, rng, where="random"):
    """Zero a contiguous fraction `frac` of the TIME axis (all sensors of the branch)."""
    out = arr.copy()
    holes = np.zeros_like(arr, dtype=bool)
    ax = _time_axis(arr)
    L = arr.shape[ax]
    g = max(1, int(round(frac * L)))
    if where == "front":
        s = 0
    elif where == "end":
        s = L - g
    else:
        s = int(rng.integers(0, max(1, L - g + 1)))
    sl = [slice(None)] * arr.ndim
    sl[ax] = slice(s, s + g)
    out[tuple(sl)] = 0.0
    holes[tuple(sl)] = True
    return out, holes


def _zero_channels(arr, chans=None):
    """Zero whole sensors. chans=None -> the whole branch (every sensor, all time). Sensor axis is -1 on profile branches
    (B, 1, 1, T, n_sensors); scalar branches (solenoid, ip) are a single sensor, so they are blanked entirely."""
    out = arr.copy()
    holes = np.zeros_like(arr, dtype=bool)
    if chans is None or arr.ndim < 5:
        out[...] = 0.0
        holes[...] = True
    else:
        out[..., list(chans)] = 0.0
        holes[..., list(chans)] = True
    return out, holes


def _impute(masked, holes, mode, channel_mean=0.0):
    """Fill ONLY the positions flagged in `holes`. mode in {zero,mean,ffill}.

    'zero'   -> leave the zeros (TokaMark default; no filling).
    'mean'   -> put `channel_mean` (TRAIN-split signal mean) in every hole.
    'ffill'  -> carry the last valid (non-hole) value forward along time, per channel.
    """
    if mode == "zero":
        return masked
    out = np.array(masked, dtype=masked.dtype, copy=True)
    if not holes.any():
        return out
    if mode == "mean":
        out[holes] = float(channel_mean)
        return out
    if mode == "ffill":
        ax = _time_axis(out)  # carry the last valid value forward along TIME, per sensor
        om, hm = np.moveaxis(out, ax, 0), np.moveaxis(holes, ax, 0)  # views: writes land in `out`
        last = np.zeros(om.shape[1:], out.dtype)
        for t in range(om.shape[0]):
            h = hm[t]
            om[t][h] = last[h]
            last = np.where(h, last, om[t])
        return out
    raise ValueError(mode)


# ======================================================================================
# SCENARIO APPLICATION  (returns a new inputs list, corruption bookkeeping dict)
# ======================================================================================
def apply_scenario(inputs, scenario, sev, impute, train_means, rng):
    """inputs: list of 8 torch tensors (B, W, C, L). Returns (corrupted_inputs, info).

    Corruption is applied on CPU in numpy, then re-wrapped to torch tensors matching
    each input's original dtype. Imputation 'mean' uses `train_means` (per signal, a
    scalar computed from the TRAIN split) so nothing about the test split leaks.
    """
    import torch
    info = dict(scenario=scenario, severity=sev, impute=impute, details={}, dead_channels=[])
    # channel-set scenarios are deterministic per task (draw once, not per channel)
    ablate_chans = None
    if scenario == "channel_ablation":
        n = min(int(sev), len(SIGNALS))
        ablate_chans = sorted(rng.choice(len(SIGNALS), size=n, replace=False).tolist())
        info["details"]["channels_zeroed"] = [SIGNAL_SHORT[c] for c in ablate_chans]
        info["dead_channels"] = ablate_chans  # whole sensors lost: the failure-aware model is told via its health mask
    out = []
    for i, t in enumerate(inputs):
        arr = _as_np(t).astype(np.float64)
        masked, holes = arr, np.zeros_like(arr, dtype=bool)
        if scenario == "random_dropout":
            masked, holes = _mask_random_dropout(arr, float(sev), rng)
        elif scenario == "channel_ablation":
            if i in ablate_chans:
                masked, holes = _zero_channels(arr)
        elif scenario == "temporal_gap":
            masked, holes = _mask_temporal_gap(arr, float(sev), rng, where="random")
        elif scenario == "disruption_proximate":
            masked, holes = _mask_temporal_gap(arr, float(sev), rng, where="end")
        else:
            raise ValueError(scenario)
        masked = _impute(masked, holes, impute, channel_mean=train_means.get(i, 0.0))
        out.append(torch.as_tensor(masked, dtype=getattr(t, "dtype", None)))
    return out, info


def apply_group(inputs, scenario, sev, impute, train_means, rng, group_key, group_chans):
    """Like apply_scenario but zeroes a fixed set of channels (category / group runs)."""
    import torch
    group_chans = [c for c in group_chans if 0 <= c < len(inputs)]
    info = dict(scenario=scenario, severity=sev, impute=impute, dead_channels=list(group_chans),
                details={"group": group_key, "channels": [SIGNAL_SHORT[c] for c in group_chans]})
    out = []
    for i, t in enumerate(inputs):
        arr = _as_np(t).astype(np.float64)
        if i in group_chans:
            masked, holes = _zero_channels(arr)
        else:
            masked, holes = arr, np.zeros_like(arr, dtype=bool)
        masked = _impute(masked, holes, impute, channel_mean=train_means.get(i, 0.0))
        out.append(torch.as_tensor(masked, dtype=getattr(t, "dtype", None)))
    return out, info


# ======================================================================================
# METRICS  (reuse the official hierarchy: per-window -> per-shot -> across shots)
# ======================================================================================
def _psi_mean_std(meta):
    m = meta["output"]["equilibrium-psi"]
    return float(m["mean"]), float(m["std"])


def window_metrics(y_true, y_pred):
    """y_true/y_pred: (N, 65, 65) numpy, real units. Returns per-window RMSE, MAE arrays."""
    diff = y_pred - y_true
    rmse = np.sqrt(np.nanmean(diff ** 2, axis=(1, 2)))
    mae = np.nanmean(np.abs(diff), axis=(1, 2))
    return rmse, mae


def aggregate(shot_ids, rmse, mae, std):
    """Per-window -> per-shot mean -> across-shot mean/std; plus window-level stats.

    Mirrors tokamark.evaluator: RMSE per window, then averaged per shot, then NRMSE =
    shot_RMSE / output_std, then mean/std across shots (population sd, ddof=0).
    """
    shot_ids = np.asarray(shot_ids, dtype=np.int64)
    rmse = np.asarray(rmse, dtype=np.float64)
    mae = np.asarray(mae, dtype=np.float64)
    uniq = np.unique(shot_ids)
    shot_rmse, shot_mae = [], []
    for s in uniq:
        m = shot_ids == s
        # Official TokaMark: shot RMSE = sqrt(mean(window_RMSE ** 2))
        shot_rmse.append(float(np.sqrt(np.nanmean(rmse[m] ** 2))))
        shot_mae.append(float(np.nanmean(mae[m])))
    shot_rmse = np.asarray(shot_rmse)
    shot_mae = np.asarray(shot_mae)
    return dict(
        n_shots=int(len(uniq)),
        n_windows=int(len(rmse)),
        NRMSE=float(np.nanmean(shot_rmse / std)),
        NRMSE_std=float(np.nanstd(shot_rmse / std, ddof=0)),
        NMAE=float(np.nanmean(shot_mae / std)),
        NMAE_std=float(np.nanstd(shot_mae / std, ddof=0)),
        RMSE=float(np.nanmean(shot_rmse)),
        RMSE_std=float(np.nanstd(shot_rmse, ddof=0)),
        MAE=float(np.nanmean(shot_mae)),
        MAE_std=float(np.nanstd(shot_mae, ddof=0)),
        window_RMSE_mean=float(np.nanmean(rmse)),
        window_RMSE_median=float(np.nanmedian(rmse)),
        window_RMSE_p90=float(np.nanpercentile(rmse, 90)),
        window_RMSE_max=float(np.nanmax(rmse)),
    )


# ======================================================================================
# INFERENCE PASS
# ======================================================================================
def load_test_loader_and_model(split, batch=None):
    setup_imports()
    import torch
    loaders, meta = build_data(split, ["test"])
    if batch is not None:
        # rebuild the loader with the requested batch size (build_data uses CFG batch)
        from torch.utils.data import DataLoader
        from src.trainer import model_collate_fn
        from utils import seed_worker
        ds = loaders["test"].dataset
        nw = CFG["dl_workers"]
        if nw < 0:
            nw = max(2, min(8, os.cpu_count() or 2))
        g = torch.Generator(); g.manual_seed(CFG["seed"])
        loaders["test"] = DataLoader(ds, batch_size=batch, num_workers=nw, collate_fn=model_collate_fn,
                                     worker_init_fn=seed_worker, generator=g,
                                     pin_memory=torch.cuda.is_available(),
                                     prefetch_factor=2 if nw > 0 else None)
    model = make_model(loaders["test"], meta)
    dev = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model.load_state_dict(torch.load(os.path.join(P.paths(split)["model_dir"], "best_model.pt"),   # honours TM_RUN_TAG
                                     map_location=dev, weights_only=False))
    model.to(dev).eval()
    n_gpu = torch.cuda.device_count()
    if n_gpu > 1 and CFG["use_dp"] != "0":  # inference is pure forward passes: split every batch across all GPUs
        model = torch.nn.DataParallel(model)
    log(f"inference on {max(n_gpu, 1) if CFG['use_dp'] != '0' else 1} GPU(s) ({n_gpu} visible)")
    return loaders["test"], model, meta, dev


def train_channel_means(split):
    """Per-signal mean computed on the TRAIN subset (for mean-imputation, no leakage)."""
    setup_imports()
    try:
        P_ = P.paths(split)
        sel = json.load(open(P_["sel"]))
        train_shots = [s for s in sel.get("shots", {}).get("train", [])
                       if os.path.isdir(os.path.join(P_["data"], f"{s}.zarr"))]
        if len(train_shots) >= 5:
            loaders, _ = build_data(split, ["train"])
            return _train_means_pass(loaders["train"])
        else:
            log(f"only {len(train_shots)} train shots on disk; using baseline 0.0 for mean imputation")
            return {i: 0.0 for i in range(len(SIGNALS))}
    except Exception as e:  # noqa
        log("train-channel-mean pass failed, falling back to zeros:", repr(e)[:200])
        return {i: 0.0 for i in range(len(SIGNALS))}


def _train_means_pass(loader, max_batches=40):
    sums = None; cnts = None
    seen = 0
    for batch in loader:
        if batch is None:
            continue
        x = batch[2]
        if sums is None:
            sums = [0.0] * len(x); cnts = [0] * len(x)
        for i, t in enumerate(x):
            a = _as_np(t).astype(np.float64)
            a = a[np.isfinite(a) & (a != 0)]
            sums[i] += float(a.sum()); cnts[i] += int(a.size)
        seen += 1
        if seen >= max_batches:
            break
    means = {}
    for i in range(len(sums)):
        means[i] = (sums[i] / cnts[i]) if cnts[i] > 0 else 0.0
    log(f"train channel means (first {seen} batches): " +
        ", ".join(f"{SIGNAL_SHORT[i]}={means[i]:.3g}" for i in sorted(means)))
    return means


PHYS_REC_KEYS = ("boundary_mean_cm", "boundary_hausdorff_cm", "axis_shift_cm", "xpoint_err_cm", "kappa_err", "delta_err",
                 "boundary_failed", "is_false_alarm", "is_missed_alarm")


def phys_selected(shot, win, stride):
    """Order-independent window selection: the same (shot, window) pairs are scored in every pass."""
    return stride > 0 and (int(shot) * 7919 + int(win) * 104729) % stride == 0


def _num(v):
    v = np.asarray(v).item() if np.ndim(v) == 0 else v
    return bool(v) if isinstance(v, (bool, np.bool_)) else float(v)


def phys_summary(recs, rows, no_truth, no_lcfs, boot=1000):
    """Aggregate EFIT-mode records + the counts the report must quote + shot-level bootstrap CIs (95 %)."""
    out = dict(agg=aggregate_physics_metrics(recs) if recs else {})
    out["counts"] = dict(n_selected=len(recs) + no_truth, n_used=len(recs), n_no_truth=no_truth, n_no_efit_lcfs=no_lcfs,
                         n_boundary_fail=int(sum(1 for r in recs if r.get("boundary_failed"))))
    out["records"] = rows
    if bootstrap_ci is not None and rows:
        out["ci95"] = {k: bootstrap_ci([r["shot"] for r in rows], [r[k] for r in rows], n_boot=boot)
                       for k in ("boundary_mean_cm", "axis_shift_cm", "boundary_hausdorff_cm")}
    return out


def run_pass(loader, model, dev, mu, sd, corrupt_fn=None, accumulate_spatial=False, return_viz=False, phys_stride=0, truth=None):
    """Forward the whole test loader; return aggregated metrics + per-window rows.

    corrupt_fn: callable(x_list) -> (x_corrupted_list, info) applied per batch, or None
                for the clean pass.
    phys_stride: ~1/N of the windows (hash-selected, same set in every pass) also get the Module 1 v2 physics metrics,
                 scored against `truth` (a TruthIndex); windows the index has no truth for are skipped and counted.
    A failure-aware model is given a health mask: 0 for every sensor the scenario removed entirely
    (channel / category / group loss), 1 for partial damage (dropout, gaps), which it must handle unaided.
    """
    import torch
    phys_rows, phys_meta, phys_no_truth, phys_no_lcfs, n_seen = [], [], 0, 0, 0
    sid_all, wid_all, rmse_all, mae_all = [], [], [], []
    spatial_sum = None; spatial_cnt = None
    viz_data = None
    n_batches = 0
    with torch.no_grad():
        for batch in loader:
            if batch is None:
                continue
            shot_id, window_id, x, y = batch
            x = [t.to(dev, dtype=torch.float32) for t in x]
            xin, info = x, {}
            if corrupt_fn is not None:
                xin_cpu, info = corrupt_fn(x)
                xin = [t.to(dev, dtype=torch.float32) for t in xin_cpu]
            if CFG["failure_aware"]:
                health = torch.ones((xin[0].shape[0], len(xin)), device=dev)
                health[:, info.get("dead_channels", [])] = 0.0
                y_pred = model(*xin, health)
            else:
                y_pred = model(*xin)
            if not isinstance(y_pred, (list, tuple)):
                y_pred = [y_pred]
            yt = y[0].detach().cpu().squeeze(1).numpy().astype(np.float64) * sd + mu
            yp = y_pred[0].detach().cpu().squeeze(1).numpy().astype(np.float64) * sd + mu
            yt = yt.reshape(yt.shape[0], -1, 65, 65)
            yp = yp.reshape(yp.shape[0], -1, 65, 65)
            # flatten any extra window axis into the sample axis
            n = yt.shape[0] * yt.shape[1]
            yt2 = yt.reshape(n, 65, 65)
            yp2 = yp.reshape(n, 65, 65)
            if return_viz and viz_data is None and n > 0:
                sids_raw = np.asarray(shot_id).reshape(-1)
                viz_data = dict(yt=yt2[0].tolist(), yp=yp2[0].tolist(),
                                shot_id=int(sids_raw[0]) if len(sids_raw) > 0 else 0)
            if phys_stride and evaluate_window_physics is not None:
                sids_b, wids_b = np.asarray(shot_id).reshape(-1), np.asarray(window_id).reshape(-1)
                per = n // max(1, len(sids_b))  # extra window axis flattened into samples (normally 1)
                for j in range(n):
                    s_j, w_j = int(sids_b[j // per]), int(wids_b[j // per])
                    if not phys_selected(s_j, w_j, phys_stride):
                        continue
                    tr = truth.get(s_j, w_j) if truth is not None else None
                    if truth is not None and tr is None:
                        phys_no_truth += 1
                        continue
                    rec = evaluate_window_physics(yt2[j], yp2[j], truth=tr) if tr is not None else evaluate_window_physics(yt2[j], yp2[j])
                    if tr is not None and tr.get("lcfs_rz_m") is None:
                        phys_no_lcfs += 1
                    phys_rows.append(rec)
                    phys_meta.append(dict(shot=s_j, win=w_j, **{k: _num(rec[k]) for k in PHYS_REC_KEYS if k in rec}))
            n_seen += n
            rmse, mae = window_metrics(yt2, yp2)
            rmse = rmse.reshape(yt.shape[0], yt.shape[1]).mean(axis=1)
            mae = mae.reshape(yt.shape[0], yt.shape[1]).mean(axis=1)
            sid_all.append(np.asarray(shot_id).reshape(-1))
            wid_all.append(np.asarray(window_id).reshape(-1))
            rmse_all.append(rmse); mae_all.append(mae)
            if accumulate_spatial:
                err = np.abs(yp2 - yt2)
                s = np.nansum(err, axis=0); c = np.isfinite(err).sum(axis=0)
                spatial_sum = s if spatial_sum is None else spatial_sum + s
                spatial_cnt = c if spatial_cnt is None else spatial_cnt + c
            n_batches += 1
    sid = np.concatenate(sid_all) if len(sid_all) > 0 else np.array([])
    rmse = np.concatenate(rmse_all) if len(rmse_all) > 0 else np.array([])
    mae = np.concatenate(mae_all) if len(mae_all) > 0 else np.array([])
    m = aggregate(sid, rmse, mae, sd)
    m["n_batches"] = n_batches
    if phys_rows:
        ps = phys_summary(phys_rows, phys_meta, phys_no_truth, phys_no_lcfs)
        m["PHYS"] = ps["agg"]  # schema unchanged (report/build_assets.py reads it); everything below is additive
        m["PHYS_counts"], m["PHYS_records"] = ps["counts"], ps["records"]
        if "ci95" in ps:
            m["PHYS_ci95"] = ps["ci95"]
    if accumulate_spatial and spatial_sum is not None and spatial_cnt is not None:
        m["spatial_abs_err"] = (spatial_sum / np.maximum(spatial_cnt, 1)).tolist()
    if return_viz and viz_data is not None:
        m["viz"] = viz_data
    m["_windows"] = dict(shot_id=sid.tolist(), rmse=rmse.tolist(), mae=mae.tolist())
    return m


def build_sweep_tasks(rc):
    """Expand the sweep into a list of task dicts (id, callable-corruption spec)."""
    tasks = []
    for scen in rc["scenarios"]:
        for sev in SEVERITIES[scen]:
            for imp in rc["imputes"]:
                if scen in ("category_abl", "group_failure"):
                    groups = CATEGORY_GROUPS if scen == "category_abl" else CORRELATED_GROUPS
                    for gname, gchans in groups.items():
                        tasks.append(dict(id=f"{scen}|{gname}|{imp}", scenario=scen,
                                          severity=sev, impute=imp, group=gname, chans=gchans))
                else:
                    tasks.append(dict(id=f"{scen}|{sev}|{imp}", scenario=scen,
                                      severity=sev, impute=imp, group=None, chans=None))
    return tasks


def make_corrupt_fn(task, rng_seed, train_means):
    """Return a callable(x_list)->(x_corrupt,info) with a fresh RNG per task (seeded)."""
    rng = np.random.default_rng(rng_seed)
    if task["scenario"] in ("category_abl", "group_failure"):
        def fn(x):
            return apply_group(x, task["scenario"], task["severity"], task["impute"],
                               train_means, rng, task["group"], task["chans"])
    else:
        def fn(x):
            return apply_scenario(x, task["scenario"], task["severity"], task["impute"],
                                  train_means, rng)
    return fn


# ======================================================================================
# COMMANDS
# ======================================================================================
def cmd_corrupt(split):
    """Build + persist the sweep manifest (and verify the clean baseline gate)."""
    rc = rob_cfg()
    P_ = rob_paths(split); os.makedirs(P_["dir"], exist_ok=True)
    tasks = build_sweep_tasks(rc)
    json.dump(dict(config=rc, tasks=tasks, signals=SIGNALS, categories=CATEGORY_GROUPS,
                   correlated_groups=CORRELATED_GROUPS, severities=SEVERITIES),
              open(P_["manifest"], "w"), indent=1)
    log(f"corruption manifest: {len(tasks)} tasks -> {P_['manifest']}")
    log("sweep tasks:\n  " + "\n  ".join(t["id"] for t in tasks))


def cmd_eval_robust(split):
    rc = rob_cfg()
    P_ = rob_paths(split); os.makedirs(P_["dir"], exist_ok=True)
    loader, model, meta, dev = load_test_loader_and_model(split, batch=rc["batch"])
    eval_sweep(loader, model, dev, meta, split, rc, P_)


def eval_sweep(loader, model, dev, meta, split, rc, P_, train_means=None):
    """Clean pass + the corruption sweep. `loader` may be any re-iterable of collated batches (the local CPU run feeds a
    disk cache). TM_ROB_RESUME=1 reuses finished passes from an existing robust_metrics.json (checkpoint after every pass)."""
    mu, sd = _psi_mean_std(meta)
    truth = get_truth(split) if rc["phys_stride"] else None
    if rc["phys_stride"] and truth is None:
        log("WARNING: efit_truth not importable -> physics sample uses the heuristic mode (NOT EFIT truth)")
    prev = json.load(open(P_["metrics"])) if rc.get("resume") and os.path.exists(P_["metrics"]) else None
    if prev and "clean" in prev and not rc["clean_only"]:
        log(f"RESUME: {len(prev.get('corrupted', {}))} finished passes found in {P_['metrics']}")

    # ---- CLEAN GATE: must reproduce the eval-time NRMSE before corruption is trusted
    log("CLEAN pass (sensor-healthy baseline)...")
    clean = prev["clean"] if prev and "clean" in prev and not rc["clean_only"] else \
        run_pass(loader, model, dev, mu, sd, accumulate_spatial=True, return_viz=True, phys_stride=rc["phys_stride"], truth=truth)
    viz_samples = prev["viz_samples"] if prev and "viz_samples" in prev and "clean" in prev else dict(clean=clean.pop("viz", None), tasks={})
    log(f"CLEAN: NRMSE={clean['NRMSE']:.4f} (±{clean['NRMSE_std']:.4f}) NMAE={clean['NMAE']:.4f} "
        f"RMSE={clean['RMSE']:.5f} MAE={clean['MAE']:.5f} n_shots={clean['n_shots']} n_windows={clean['n_windows']}")
    targets = None
    mp = os.path.join(P_["model_dir"], "..", "results", "task_1-3", "task_metrics.csv")
    if os.path.exists(mp):
        import csv as _csv
        with open(mp) as fh:
            for row in _csv.DictReader(fh):
                feat = row.get("feature_name", "")
                if feat in ("equilibrium-psi", "equilibrium_psi", "task_1-3") or "psi" in feat:
                    val = row.get("NRMSE_mean") or row.get("NRMSE")
                    if val:
                        try:
                            targets = float(val)
                        except Exception:
                            pass
                        break
    if targets is not None:
        d = abs(clean["NRMSE"] - targets)
        ok = d <= max(0.02, 0.15 * targets)
        log(f"clean-gate: harness NRMSE={clean['NRMSE']:.4f} vs eval CSV={targets:.4f} "
            f"(Δ={d:.4f}) -> {'PASS' if ok else 'FAIL (trust nothing downstream)'}")
        if not ok:
            log("WARNING: clean baseline mismatch; corruption numbers below are suspect")

    result = dict(clean=clean, corrupted=dict(prev["corrupted"]) if prev else {}, config=rc, psi_mean=mu, psi_std=sd,
                  viz_samples=viz_samples)
    if rc["clean_only"]:
        clean.pop("_windows", None)
        json.dump(result, open(P_["metrics"], "w"), indent=1)
        log("TM_ROB_CLEAN_ONLY=1 -> wrote clean-only metrics, skipping corruption")
        return
    json.dump(result, open(P_["metrics"], "w"), indent=1)  # clean pass is a checkpoint too

    if train_means is None:
        log("building TRAIN channel means for mean-imputation...")
        train_means = train_channel_means(split)
    result["train_means"] = {str(k): float(v) for k, v in train_means.items()}

    VIZ_TASKS = {"random_dropout|0.5|zero", "random_dropout|0.5|ffill",
                 "category_abl|plasma_current|zero", "category_abl|plasma_current|ffill"}
    tasks = build_sweep_tasks(rc)
    log(f"corruption sweep: {len(tasks)} tasks on the TEST split...")
    rows = []
    for ti, task in enumerate(tasks):
        cf = make_corrupt_fn(task, rc["seed"] * 1000003 + ti, train_means)
        need_viz = task["id"] in VIZ_TASKS
        if task["id"] in result["corrupted"]:
            log(f"[{ti + 1}/{len(tasks)}] {task['id']} done (resume)")
            continue
        log(f"[{ti + 1}/{len(tasks)}] {task['id']} ...")
        try:
            m = run_pass(loader, model, dev, mu, sd, corrupt_fn=cf, return_viz=need_viz, phys_stride=rc["phys_stride"], truth=truth)
        except Exception as e:  # noqa
            log(f"  task {task['id']} FAILED: {repr(e)[:200]}")
            continue
        if need_viz and "viz" in m:
            viz_samples["tasks"][task["id"]] = m.pop("viz", None)
        m.pop("_windows", None)
        m["id"] = task["id"]
        rs = clean["NRMSE"] / m["NRMSE"] if m["NRMSE"] > 0 else float("nan")
        m["RS_vs_clean"] = float(rs)
        result["corrupted"][task["id"]] = m
        log(f"  {task['id']}: NRMSE={m['NRMSE']:.4f} (+{100 * (m['NRMSE'] / clean['NRMSE'] - 1):.1f}%) "
            f"NMAE={m['NMAE']:.4f} RS={rs:.3f}")
        rows.append(dict(id=task["id"], scenario=task["scenario"], severity=task["severity"],
                         impute=task["impute"], group=task["group"], **{k: m[k] for k in (
                             "NRMSE", "NMAE", "RMSE", "MAE", "RS_vs_clean", "n_shots", "n_windows")}))
        json.dump(result, open(P_["metrics"], "w"), indent=1)  # incremental save

    clean.pop("_windows", None)
    json.dump(result, open(P_["metrics"], "w"), indent=1)
    log(f"robustness metrics written -> {P_['metrics']} ({len(rows)} corrupted tasks)")


def cmd_report(split):
    rc = rob_cfg()
    P_ = rob_paths(split)
    if not os.path.exists(P_["metrics"]):
        raise SystemExit(f"FATAL: {P_['metrics']} not found - run eval-robust first")
    res = json.load(open(P_["metrics"]))
    os.makedirs(P_["figures"], exist_ok=True)
    clean = res["clean"]; corrupted = res["corrupted"]

    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    # ---- RS aggregation: worst-case NRMSE per scenario (zero-fill), Gupta-style
    def rs_table():
        worst = {}
        for tid, m in corrupted.items():
            scen, _, imp = tid.split("|")
            if imp != "zero":
                continue
            worst[scen] = max(worst.get(scen, 0.0), m["NRMSE"])
        rs = {scen: (clean["NRMSE"] / v if v else float("nan")) for scen, v in worst.items()}
        return worst, rs

    worst, rs = rs_table()

    # ---- figure 1: degradation curves NRMSE vs severity, per scenario (zero-fill)
    try:
        fig, ax = plt.subplots(figsize=(8, 5))
        for scen in SCENARIOS:
            pts = []
            for tid, m in corrupted.items():
                parts = tid.split("|")
                if parts[0] == scen and parts[-1] == "zero" and len(parts) == 3:
                    try:
                        pts.append((float(parts[1]), m["NRMSE"]))
                    except ValueError:  # named severity (e.g. group_failure|flux_loop): not a point on a numeric curve
                        pass
            if pts:
                pts.sort()
                ax.plot([p[0] for p in pts], [p[1] for p in pts], marker="o", label=scen)
        ax.axhline(clean["NRMSE"], ls="--", c="k", lw=1, label="clean")
        ax.set_xlabel("severity"); ax.set_ylabel("NRMSE (test, un-standardised)")
        ax.set_title(f"Robustness degradation - {TASK} ({split})")
        ax.legend(fontsize=8); ax.grid(alpha=0.3); fig.tight_layout()
        fig.savefig(os.path.join(P_["figures"], "degradation_curves.png"), dpi=120)
        plt.close(fig)
        log("saved degradation_curves.png")
    except Exception as e:  # noqa
        log("degradation curve failed:", repr(e)[:200])

    # ---- figure 2: per-signal importance (category_abl + channel_ablation n=1), zero-fill
    try:
        labels, vals = [], []
        for tid, m in corrupted.items():
            p = tid.split("|")
            if p[0] == "category_abl" and p[-1] == "zero":
                labels.append(p[1]); vals.append(100 * (m["NRMSE"] / clean["NRMSE"] - 1))
        order = np.argsort(vals)[::-1]
        fig, ax = plt.subplots(figsize=(8, 4))
        ax.bar([labels[i] for i in order], [vals[i] for i in order], color="steelblue")
        ax.set_ylabel("% NRMSE increase vs clean"); ax.grid(alpha=0.3, axis="y")
        ax.set_title(f"Diagnostic importance (zero each category) - {TASK}")
        fig.tight_layout(); fig.savefig(os.path.join(P_["figures"], "signal_importance.png"), dpi=120)
        plt.close(fig)
        log("saved signal_importance.png")
    except Exception as e:  # noqa
        log("importance bar failed:", repr(e)[:200])

    # ---- figure 3: spatial error map (clean)
    try:
        sm = np.array(clean.get("spatial_abs_err", []))
        if sm.size:
            fig, ax = plt.subplots(figsize=(4.5, 4))
            im = ax.imshow(sm, cmap="magma")
            ax.set_title("Clean |error| map (65x65)")
            ax.axis("off"); fig.colorbar(im, ax=ax, fraction=0.046)
            fig.tight_layout(); fig.savefig(os.path.join(P_["figures"], "spatial_error_clean.png"), dpi=120)
            plt.close(fig)
            log("saved spatial_error_clean.png")
    except Exception as e:  # noqa
        log("spatial map failed:", repr(e)[:200])

    # ---- figure 4: qualitative 2D psi under corruption & mitigation
    try:
        vs = res.get("viz_samples")
        if vs and vs.get("clean") and vs.get("tasks"):
            yt = np.array(vs["clean"]["yt"])
            yp_clean = np.array(vs["clean"]["yp"])
            sid = vs["clean"].get("shot_id", 0)

            t_drop_z = vs["tasks"].get("random_dropout|0.5|zero") or {}
            t_drop_f = vs["tasks"].get("random_dropout|0.5|ffill") or {}
            t_ip_z = vs["tasks"].get("category_abl|plasma_current|zero") or {}
            t_ip_f = vs["tasks"].get("category_abl|plasma_current|ffill") or {}

            yp_drop_z = np.array(t_drop_z.get("yp", yp_clean))
            yp_drop_f = np.array(t_drop_f.get("yp", yp_clean))
            yp_ip_z = np.array(t_ip_z.get("yp", yp_clean))
            yp_ip_f = np.array(t_ip_f.get("yp", yp_clean))

            fig, axes = plt.subplots(2, 4, figsize=(13, 6.2), layout="constrained")
            rows = [
                ("50% Random Sensor Dropout", yp_drop_z, yp_drop_f),
                ("Complete Plasma Current (Ip) Loss", yp_ip_z, yp_ip_f),
            ]
            for r_i, (label, yp_z, yp_f) in enumerate(rows):
                err_clean = float(np.mean(np.abs(yp_clean - yt)))
                err_z = float(np.mean(np.abs(yp_z - yt)))
                err_f = float(np.mean(np.abs(yp_f - yt)))
                cols = [
                    (yt, f"Ground Truth $\\psi$ (Shot {sid})", "viridis"),
                    (yp_clean, f"Clean Pred (MAE: {err_clean:.4f} Wb)", "viridis"),
                    (yp_z, f"Fault: {label} [Zero-Fill]\n(MAE: {err_z:.4f} Wb)", "inferno"),
                    (yp_f, f"Mitigated: [Forward-Fill]\n(MAE: {err_f:.4f} Wb)", "viridis"),
                ]
                for c_i, (img, ttl, cmap) in enumerate(cols):
                    ax = axes[r_i, c_i]
                    im = ax.imshow(img, cmap=cmap)
                    ax.set_title(ttl, fontsize=9.5, fontweight="semibold")
                    ax.axis("off")
                    fig.colorbar(im, ax=ax, fraction=0.046)
            out_psi = os.path.join(P_["figures"], "robustness_psi_reconstruction.png")
            fig.savefig(out_psi, dpi=150)
            plt.close(fig)
            log("saved robustness_psi_reconstruction.png")
    except Exception as e:  # noqa
        log("robustness psi plot failed:", repr(e)[:200])

    # ---- figure 5: diagnostic criticality ranking
    try:
        cats, vals = [], []
        for tid, m in corrupted.items():
            p = tid.split("|")
            if p[0] == "category_abl" and p[-1] == "zero":
                cats.append(p[1])
                vals.append(100 * (m["NRMSE"] / clean["NRMSE"] - 1))
        if cats:
            order = np.argsort(vals)
            max_v = max(vals) if vals else 100.0
            fig, ax = plt.subplots(figsize=(7.5, 3.8), layout="constrained")
            bars = ax.barh([cats[i] for i in order], [vals[i] for i in order], color="#2b5c8f", height=0.6)
            ax.axvline(0, color="black", linewidth=0.8)
            ax.set_xlim(0, max_v * 1.25)
            ax.set_xlabel("NRMSE Error Surge Above Clean Baseline (%)", fontsize=10, fontweight="bold")
            ax.set_title("Diagnostic Criticality: Test NRMSE Surge When Sensor Groups Fail", fontsize=11, fontweight="bold")
            for bar, idx in zip(bars, order):
                ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
                        f"+{vals[idx]:.1f}%", va="center", ha="left", fontsize=9, fontweight="semibold")
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)
            fig.savefig(os.path.join(P_["figures"], "robustness_criticality.png"), dpi=200)
            plt.close(fig)
            log("saved robustness_criticality.png")
    except Exception as e:  # noqa
        log("robustness criticality failed:", repr(e)[:200])

    # ---- figure 6: online fault mitigation comparison
    try:
        drop_sevs = [0.10, 0.25, 0.50]
        z_vals = [clean["NRMSE"]]
        m_vals = [clean["NRMSE"]]
        f_vals = [clean["NRMSE"]]
        labels = ["Clean (p=0)"] + [f"p = {s:.2f}" for s in drop_sevs]
        for s in drop_sevs:
            z_vals.append(corrupted.get(f"random_dropout|{s}|zero", {}).get("NRMSE", clean["NRMSE"]))
            m_vals.append(corrupted.get(f"random_dropout|{s}|mean", {}).get("NRMSE", clean["NRMSE"]))
            f_vals.append(corrupted.get(f"random_dropout|{s}|ffill", {}).get("NRMSE", clean["NRMSE"]))
        x = np.arange(len(labels))
        width = 0.24
        fig, ax = plt.subplots(figsize=(7.5, 3.8), layout="constrained")
        ax.bar(x - width, z_vals, width, label="Zero-Fill (Default Baseline)", color="#dc2626")
        ax.bar(x, m_vals, width, label="Mean-Fill (Dataset Mean)", color="#f59e0b")
        ax.bar(x + width, f_vals, width, label="Forward-Fill (Physics-Informed)", color="#10b981")
        ax.axhline(clean["NRMSE"], color="#64748b", linestyle="--", linewidth=1.0, label="Clean Baseline")
        ax.set_ylabel("Test NRMSE (Lower is Better)", fontsize=10, fontweight="bold")
        ax.set_title("Online Fault Mitigation: Imputation Under Random Dropouts", fontsize=11, fontweight="bold")
        ax.set_xticks(x)
        ax.set_xticklabels(labels, fontsize=9.5, fontweight="semibold")
        ax.legend(frameon=True, fontsize=8.5, loc="upper left")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        fig.savefig(os.path.join(P_["figures"], "robustness_mitigation.png"), dpi=200)
        plt.close(fig)
        log("saved robustness_mitigation.png")
    except Exception as e:  # noqa
        log("robustness mitigation failed:", repr(e)[:200])

    # ---- write robustness.md
    lines = [f"# Robustness report - {TASK} ({split} split)", ""]
    lines += [
        "Sensor-failure study mirroring Gupta (arXiv 2607.11915), on the equilibrium",
        "reconstruction task (Gupta used task_4-4). Model: frozen `best_model.pt`.",
        "**Test split only. Model never retrained on corrupted data.**",
        "",
        "## Clean baseline (sensor-healthy)",
        f"- NRMSE = **{clean['NRMSE']:.4f}** (±{clean['NRMSE_std']:.4f})",
        f"- NMAE = {clean['NMAE']:.4f}, RMSE = {clean['RMSE']:.5f}, MAE = {clean['MAE']:.5f}",
        f"- n_shots = {clean['n_shots']}, n_windows = {clean['n_windows']}",
        "",
        "## Robustness Score (RS) — worst case per scenario, zero-fill",
        "RS = NRMSE_clean / NRMSE_corrupted; 1.0 = no damage, lower = worse.",
        "",
        "| scenario | worst NRMSE | RS |",
        "|---|---|---|",
    ]
    for scen in SCENARIOS:
        if scen in worst:
            lines.append(f"| {scen} | {worst[scen]:.4f} | {rs[scen]:.3f} |")
    lines += ["", "## Full sweep", "",
              "| task | NRMSE | NMAE | RS | n_shots |", "|---|---|---|---|---|"]
    for tid in sorted(corrupted):
        m = corrupted[tid]
        lines.append(f"| {tid} | {m['NRMSE']:.4f} | {m['NMAE']:.4f} | {m['RS_vs_clean']:.3f} | {m['n_shots']} |")
    lines += ["", "## Caveats", "",
              "- Subset data: absolute NRMSE is NOT the official full-set figure.",
              "- mean-imputation uses TRAIN-split channel means (no test leakage).",
              "- Corruption is inference-time only; a model retrained on damaged data",
              "  could behave differently.",
              ""]
    open(P_["report"], "w", encoding="utf-8").write("\n".join(lines))
    log(f"wrote {P_['report']}")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("cmd", choices=["corrupt", "eval-robust", "report"])
    ap.add_argument("--split", default="random", choices=["random", "temporal"])
    args = ap.parse_args()
    log(f"robustness {args.cmd} split={args.split} task={TASK}")
    if args.cmd == "corrupt":
        cmd_corrupt(args.split)
    elif args.cmd == "eval-robust":
        cmd_eval_robust(args.split)
    else:
        cmd_report(args.split)
    log("done")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/physics_evaluation.py
#!/usr/bin/env python
"""Module 1 (v2): Physics-Aware Evaluation Layer for 2D tokamak equilibrium reconstruction.

Scores a predicted psi map in physical units (cm) instead of pixel NRMSE: plasma boundary, magnetic axis, X-points,
elongation / triangularity, and wall clearance.

Three modes of evaluate_window_physics(yt_2d, yp_2d, truth=None, mode=None):
  "efit"      (default when `truth` is given, see efit_truth.py): scored against TokaMark's EFIT outputs. psi_b is the flux of
              the TRUE field sampled along the EFIT LCFS; the predicted boundary is the loop of the predicted psi at psi_b
              around the EFIT axis; the axis, X-points, kappa and delta are compared with EFIT's own numbers.
  "heuristic" (default without `truth`): the original flux-fraction method (LCFS = psi_min + 0.35 * range, argmax axis,
              saddle search), but with the REAL grid geometry.
  "legacy"    the heuristic method with the old WRONG geometry (R 0.2..2.0 m, Z +-1.5 m), to reproduce the old numbers.

Grid geometry, read from the TokaMark zarr (equilibrium/major_radius, equilibrium/z; the model target is the raw 65x65
psi, no crop or resample): psi[z_row, r_col], R = 0.06 + 0.03 i m (0.06..1.98), Z = -2 + 0.0625 j m (-2..+2), i.e.
3.00 cm per column and 6.25 cm per row. The old constants (2.8125 / 4.6875 cm) were wrong by +6.7% / +25%.
"""
from collections import namedtuple

import numpy as np
import contourpy
from matplotlib.path import Path
from scipy.ndimage import map_coordinates, minimum_filter
from scipy.spatial import cKDTree

# ---- real geometry (verified against the zarr by test_physics_evaluation.test_geometry_matches_zarr) ----
GRID_SIZE = 65
R_MIN_M, R_MAX_M = 0.06, 1.98
Z_MIN_M, Z_MAX_M = -2.0, +2.0
DR_CM = (R_MAX_M - R_MIN_M) * 100.0 / (GRID_SIZE - 1)  # 3.00 cm / column
DZ_CM = (Z_MAX_M - Z_MIN_M) * 100.0 / (GRID_SIZE - 1)  # 6.25 cm / row
LEGACY_DR_CM, LEGACY_DZ_CM = 2.8125, 4.6875  # what v1 assumed (R 0.2..2.0 m, Z -1.5..+1.5 m): wrong

Geom = namedtuple("Geom", "r0 dr z0 dz")  # cm: R = r0 + dr * col, Z = z0 + dz * row
GEOM = Geom(R_MIN_M * 100.0, DR_CM, Z_MIN_M * 100.0, DZ_CM)
GEOM_LEGACY = Geom(0.0, LEGACY_DR_CM, 0.0, LEGACY_DZ_CM)  # v1 measured from the grid corner

# Wall clearance. The TokaMark zarr has NO vessel / limiter signal, so the real MAST wall is not available. PROXY: the
# edge of the EFIT computational grid (R 6..198 cm, Z -200..+200 cm). This is looser than the real vessel (the centre
# column is at R ~ 20 cm), so clearances are over-estimated; treat alarm rates as a protocol demonstration, not a safety
# claim. Assign a different (rmin, rmax, zmin, zmax) here when a real wall becomes available.
WALL_BOX_CM = (R_MIN_M * 100.0, R_MAX_M * 100.0, Z_MIN_M * 100.0, Z_MAX_M * 100.0)
WALL_BOX_LEGACY_CM = (0.0, 180.0, 0.0, 300.0)  # v1 box in corner-relative coordinates
SAFETY_CLEARANCE_CM = 5.0

LCFS_FRAC = 0.35  # heuristic mode only: psi_LCFS = psi_min + frac * (psi_max - psi_min)

# X-point = saddle (det H < 0, zero of grad psi within 1 px) whose flux is within XPOINT_PSI_TOL * range of psi_LCFS.
XPOINT_PSI_TOL = 0.15  # heuristic modes: their 35% level is not the separatrix flux, so the window is wide
# efit mode cuts at the true psi_b. Chosen on error-free EFIT psi (8 shots, 356 windows: 15% would flag 95% of the windows
# that have no X-point, 2% flags 3%); re-measured on 16 shots / 941 windows with xpoint_diagnostic.py: at 2% of the range
# 93.8% of windows with an EFIT X-point have a saddle within 10 cm and 6.4% of windows without one have a spurious saddle.
XPOINT_PSI_TOL_EFIT = 0.02
XPOINT_HIT_CM = 10.0  # a predicted X-point is a hit when within this distance of the true one
# A prediction with no contour at all at psi_b (flat, offset or inverted field) has no boundary to compare. It is NOT dropped
# from the means (that would reward the worst predictions): it is scored with this fixed penalty (cm), flagged
# boundary_failed, and counted in boundary_fail_rate_pct. The value is a convention (order of the plasma size), not a
# measurement.
BOUNDARY_FAIL_CM = 50.0
LEVEL_BUMPS = (0.0, 0.002, 0.005, 0.01, 0.02)  # fraction of the psi range tried when no closed loop exists at psi_b


def _geom(mode, truth=None):
    if mode == "legacy":
        return GEOM_LEGACY
    g = truth.get("grid") if truth else None
    return GEOM if not g else Geom(g["r0"] * 100.0, g["dr"] * 100.0, g["z0"] * 100.0, g["dz"] * 100.0)


def _to_cm(px, geom):
    px = np.asarray(px, dtype=np.float64)
    return np.column_stack([geom.r0 + px[:, 0] * geom.dr, geom.z0 + px[:, 1] * geom.dz])


def grid_to_physical_cm(r_idx, z_idx, geom=GEOM):
    """Convert (col, row) pixel indices (may be fractional) to (R, Z) in cm."""
    return geom.r0 + r_idx * geom.dr, geom.z0 + z_idx * geom.dz


def lcfs_level(field_2d, frac=LCFS_FRAC):
    """Heuristic psi_LCFS = psi_min + frac * (psi_max - psi_min) of a (true) field, or None for a flat field."""
    f_min, f_max = float(field_2d.min()), float(field_2d.max())
    return None if f_max - f_min < 1e-6 else f_min + frac * (f_max - f_min)


def extract_lcfs_contour(field_2d, frac=LCFS_FRAC, level=None, min_len=8, geom=GEOM):
    """Heuristic contour(s) at `level` (default: the 35% level of this field) as (R_cm, Z_cm) points, or None.

    All segments of at least `min_len` points are kept (two flux lobes give two loops; speckle is dropped). Pass the level
    of the TRUE field when scoring a prediction: psi_LCFS is a physical value.
    """
    if level is None:
        level = lcfs_level(field_2d, frac)
        if level is None:
            return None
    segs = contourpy.contour_generator(z=np.asarray(field_2d, dtype=np.float64)).lines(level)
    segs = [sg for sg in segs if len(sg) >= min_len]
    if not segs:
        return None
    return _to_cm(np.concatenate(segs), geom)


# ---------------------------------------------------------------- magnetic axis
def _refine_peak(f, z, r):
    """Separable 3-point parabola refinement of the maximum of `f` at integer (z, r), on the UNMASKED field."""
    r_sub = z_sub = 0.0
    if 0 < z < f.shape[0] - 1 and 0 < r < f.shape[1] - 1:
        dr_ = 2.0 * f[z, r] - f[z, r + 1] - f[z, r - 1]
        if abs(dr_) > 1e-12:
            r_sub = float(np.clip(0.5 * (f[z, r + 1] - f[z, r - 1]) / dr_, -0.5, 0.5))
        dz_ = 2.0 * f[z, r] - f[z + 1, r] - f[z - 1, r]
        if abs(dz_) > 1e-12:
            z_sub = float(np.clip(0.5 * (f[z + 1, r] - f[z - 1, r]) / dz_, -0.5, 0.5))
    return r + r_sub, z + z_sub


def get_subpixel_peak(field_2d, region=None, sign=1.0):
    """(r_px, z_px) of the extremum of the field (maximum of sign * field) with sub-pixel refinement.

    `region` is an optional boolean mask: the integer peak is searched inside it, but the sub-pixel refinement uses the
    unmasked field, so a peak on the region border is not distorted (identical fields give identical peaks).
    """
    f = sign * np.asarray(field_2d, dtype=np.float64)
    g = f if region is None else np.where(region, f, -np.inf)
    if not np.isfinite(g).any():
        g = f
    z, r = np.unravel_index(np.argmax(g), g.shape)
    return _refine_peak(f, z, r)


# ---------------------------------------------------------------- X-points
def find_xpoints(field_2d, level=None, psi_tol=XPOINT_PSI_TOL, side="lower"):
    """All X-points of psi as (r_px, z_px, psi, |det H|), best candidate first. side: "lower", "upper" or "both".

    X-point: grad(psi) = 0 and det(Hessian) < 0. Candidates are saddle pixels at a local minimum of |grad psi|, refined
    by one Newton step (rejected if the zero is more than one pixel away). With `level` (the LCFS flux) candidates whose
    flux differs from it by more than psi_tol * (psi_max - psi_min) are dropped and the rest ordered by closeness to it.
    Returns [] when there is none: it never invents one.
    """
    f = np.asarray(field_2d, dtype=np.float64)
    rng = float(f.max() - f.min())
    if rng < 1e-12:
        return []
    gz, gr = np.gradient(f)
    g2 = gr ** 2 + gz ** 2
    hrr = np.gradient(gr, axis=1)
    hzz = np.gradient(gz, axis=0)
    hrz = 0.5 * (np.gradient(gr, axis=0) + np.gradient(gz, axis=1))
    det = hrr * hzz - hrz ** 2
    cand = (g2 == minimum_filter(g2, size=3)) & (det < 0)
    cand[[0, -1], :] = False
    cand[:, [0, -1]] = False
    half = f.shape[0] // 2
    rows = np.arange(f.shape[0])[:, None]
    if side == "lower":
        cand &= rows < half
    elif side == "upper":
        cand &= rows > half
    found = []
    for z, r in zip(*np.nonzero(cand)):
        h = np.array([[hrr[z, r], hrz[z, r]], [hrz[z, r], hzz[z, r]]])
        try:
            d = -np.linalg.solve(h, np.array([gr[z, r], gz[z, r]]))
        except np.linalg.LinAlgError:
            continue
        if abs(d[0]) > 1.0 or abs(d[1]) > 1.0:
            continue
        rr, zz = r + d[0], z + d[1]
        psi = float(map_coordinates(f, [[zz], [rr]], order=1, mode="nearest")[0])
        if level is not None and abs(psi - level) > psi_tol * rng:
            continue
        found.append((float(rr), float(zz), psi, float(abs(det[z, r]))))
    found.sort(key=(lambda c: abs(c[2] - level)) if level is not None else (lambda c: -c[3]))
    return found


def find_divertor_xpoint(field_2d, level=None, psi_tol=XPOINT_PSI_TOL):
    """Best lower-divertor X-point as (r_px, z_px), or None."""
    xs = find_xpoints(field_2d, level=level, psi_tol=psi_tol, side="lower")
    return None if not xs else (xs[0][0], xs[0][1])


# ---------------------------------------------------------------- heuristic / legacy metrics
def _densify(poly_cm, step=0.5):
    """Resample a polyline to ~`step` cm spacing so point-to-point distances approximate point-to-curve distances."""
    p = np.asarray(poly_cm, dtype=np.float64)
    seg = np.hypot(*np.diff(p, axis=0).T)
    s = np.concatenate([[0.0], np.cumsum(seg)])
    if s[-1] < 1e-9:
        return p
    u = np.arange(0.0, s[-1], step)
    return np.column_stack([np.interp(u, s, p[:, 0]), np.interp(u, s, p[:, 1])])


def curve_distances(a_cm, b_cm):
    """(symmetric mean, Hausdorff) distance in cm between two curves (arrays of (R, Z) cm), densified to 0.5 cm."""
    a, b = _densify(a_cm), _densify(b_cm)
    d_ab = cKDTree(b).query(a)[0]
    d_ba = cKDTree(a).query(b)[0]
    return 0.5 * float(d_ab.mean() + d_ba.mean()), float(max(d_ab.max(), d_ba.max()))


def compute_boundary_error(yt_2d, yp_2d, frac=LCFS_FRAC, geom=GEOM):
    """Heuristic mean and Hausdorff boundary error (cm) between the 35% contours of truth and prediction.

    Both are cut at the flux level of the TRUE field. Falls back to a pixel-error based estimate when a contour is missing.
    """
    level = lcfs_level(yt_2d, frac)
    ct = extract_lcfs_contour(yt_2d, frac=frac, geom=geom)
    cp = None if level is None else extract_lcfs_contour(yp_2d, frac=frac, level=level, geom=geom)
    if ct is None or cp is None:
        diff = np.abs(yp_2d - yt_2d)
        err_px = float(diff.mean()) / (float(yt_2d.max() - yt_2d.min()) + 1e-6) * 10.0
        return err_px * geom.dr, err_px * 2.0 * geom.dz
    d_t_to_p = cKDTree(cp).query(ct)[0]
    d_p_to_t = cKDTree(ct).query(cp)[0]
    return 0.5 * float(d_t_to_p.mean() + d_p_to_t.mean()), float(max(d_t_to_p.max(), d_p_to_t.max()))


def compute_axis_shift(yt_2d, yp_2d, geom=GEOM):
    """Heuristic axis shift (cm): argmax of the prediction vs argmax of the truth, sub-pixel.

    A two-lobe field has two near-equal peaks and the prediction may favour the other one, so the predicted peak is
    searched in the same vertical half (+3 rows over the midline) as the true peak. The sub-pixel step uses the unmasked prediction, so identical
    fields give exactly 0.
    """
    rt, zt = get_subpixel_peak(yt_2d)
    half = yt_2d.shape[0] // 2
    region = np.zeros(np.shape(yp_2d), dtype=bool)
    if zt < half:  # the half of the true peak plus 3 rows over the midline, so a peak straddling it is not cut in two
        region[:half + 3] = True
    else:
        region[half - 3:] = True
    rp, zp = get_subpixel_peak(yp_2d, region=region)
    return float(np.hypot((rp - rt) * geom.dr, (zp - zt) * geom.dz)), (rt, zt), (rp, zp)


def compute_xpoint_error(yt_2d, yp_2d, frac=LCFS_FRAC, geom=GEOM):
    """Heuristic lower X-point comparison: (error_cm or nan, true_found, pred_found). Both fields at the TRUE 35% level."""
    level = lcfs_level(yt_2d, frac)
    xt = find_divertor_xpoint(yt_2d, level=level)
    xp = None if level is None else find_divertor_xpoint(yp_2d, level=level)
    if xt is None or xp is None:
        return float("nan"), xt is not None, xp is not None
    return float(np.hypot((xp[0] - xt[0]) * geom.dr, (xp[1] - xt[1]) * geom.dz)), True, True


# ---------------------------------------------------------------- loops and shape
def _area(p):
    x, y = p[:, 0], p[:, 1]
    return 0.5 * abs(float(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1))))


def loop_around(field_2d, level, point_px):
    """Largest closed contour (pixel coords) of `field_2d` at `level` that encloses `point_px` = (r_px, z_px), or None."""
    best, best_a = None, 0.0
    for sg in contourpy.contour_generator(z=np.asarray(field_2d, dtype=np.float64)).lines(level):
        if len(sg) < 8 or not np.allclose(sg[0], sg[-1]) or not Path(sg).contains_point(point_px):
            continue
        a = _area(sg)
        if a > best_a:
            best, best_a = sg, a
    return best


def loop_around_bumped(field_2d, level, point_px, sign=1.0):
    """loop_around at `level`; if none exists, retry slightly inside (towards the axis). Returns (loop_px, bump_fraction)."""
    rng = float(np.max(field_2d) - np.min(field_2d))
    for b in LEVEL_BUMPS:
        lp = loop_around(field_2d, level + sign * b * rng, point_px)
        if lp is not None:
            return lp, b
    return None, float("nan")


def main_lcfs_loop(field_2d, level, geom=GEOM):
    """Heuristic: outermost closed contour at `level` that encloses the argmax axis, in cm (R, Z), or None."""
    if level is None:
        return None
    f = np.asarray(field_2d, dtype=np.float64)
    lp = loop_around(f, level, get_subpixel_peak(f))
    return None if lp is None else _to_cm(lp, geom)


def _vertex_r(r, z, top, refine):
    """R of the highest (top) / lowest point of a boundary; `refine` fits a parabola z(r) to the points near the extreme."""
    zi = np.argmax(z) if top else np.argmin(z)
    if not refine:
        return r[zi]
    zs = z.max() if top else z.min()
    near = np.abs(z - zs) <= 0.06 * (z.max() - z.min())
    if near.sum() >= 4 and np.ptp(r[near]) > 1e-6:
        c = np.polyfit(r[near], z[near], 2)
        if c[0] * (-1 if top else 1) > 0:  # opens the right way: a rounded top / bottom
            rv = -c[1] / (2 * c[0])
            if r[near].min() <= rv <= r[near].max():
                return rv
    return r[zi]


def shape_parameters(loop_cm, refine=False):
    """Boundary shape of a (R, Z) polyline in cm, or None. EFIT / IMAS definitions:
    a = (Rmax - Rmin) / 2, R_geo = (Rmax + Rmin) / 2, kappa = (Zmax - Zmin) / (2a),
    delta_upper = (R_geo - R(Zmax)) / a, delta_lower = (R_geo - R(Zmin)) / a, delta = their mean.
    dict(kappa, delta_upper, delta_lower, delta, a_cm, r_geo_cm). On EFIT's own LCFS polygon this reproduces TokaMark's
    elongation, triangularity_upper/lower and minor_radius to 1e-4 (checked on 356 windows), so these ARE EFIT's
    definitions. `refine` (parabola at the extremes) is NOT EFIT's definition and makes it worse (bias -0.045).
    """
    if loop_cm is None or len(loop_cm) < 4:
        return None
    r, z = loop_cm[:, 0], loop_cm[:, 1]
    a = 0.5 * (r.max() - r.min())
    if a < 1e-6:
        return None
    r0 = 0.5 * (r.max() + r.min())
    du = (r0 - _vertex_r(r, z, True, refine)) / a
    dl = (r0 - _vertex_r(r, z, False, refine)) / a
    return dict(kappa=float((z.max() - z.min()) / (2.0 * a)), delta_upper=float(du), delta_lower=float(dl),
                delta=float(0.5 * (du + dl)), a_cm=float(a), r_geo_cm=float(r0))


def compute_shape_error(yt_2d, yp_2d, frac=LCFS_FRAC, geom=GEOM):
    """Heuristic (kappa_true, delta_true, kappa_pred, delta_pred); nan where no closed loop around the axis exists."""
    level = lcfs_level(yt_2d, frac)
    st = shape_parameters(main_lcfs_loop(yt_2d, level, geom))
    sp = shape_parameters(main_lcfs_loop(yp_2d, level, geom))
    nan = float("nan")
    return (st["kappa"] if st else nan, st["delta"] if st else nan, sp["kappa"] if sp else nan, sp["delta"] if sp else nan)


def compute_wall_proximity(contour_cm, wall=None):
    """Minimum clearance (cm) from a boundary to the wall box `wall` = (rmin, rmax, zmin, zmax). See WALL_BOX_CM: a PROXY."""
    if contour_cm is None:
        return 50.0  # safe default if no boundary
    rmin, rmax, zmin, zmax = WALL_BOX_CM if wall is None else wall
    r_pts, z_pts = contour_cm[:, 0], contour_cm[:, 1]
    return max(0.0, float(min(r_pts.min() - rmin, rmax - r_pts.max(), z_pts.min() - zmin, zmax - z_pts.max())))


# ---------------------------------------------------------------- EFIT-grounded scoring
def _px_of(rz_m, g):
    """EFIT (R, Z) in metres (n, 2) -> pixel (col, row) given grid dict g (metres)."""
    rz = np.atleast_2d(np.asarray(rz_m, dtype=np.float64))
    return np.column_stack([(rz[:, 0] - g["r0"]) / g["dr"], (rz[:, 1] - g["z0"]) / g["dz"]])


def _sample(f, px):
    return map_coordinates(f, [px[:, 1], px[:, 0]], order=1, mode="nearest")


def _region_mask(poly_px, shape):
    cc, rr = np.meshgrid(np.arange(shape[1]), np.arange(shape[0]))
    return Path(poly_px).contains_points(np.column_stack([cc.ravel(), rr.ravel()])).reshape(shape)


def efit_psi_b(yt_2d, lcfs_px):
    """(psi_b, spread_pct): median flux of the true field along the EFIT LCFS, and std(flux) as % of the field's range.

    The spread is a self-consistency check: on the real EFIT psi it is ~0.01% (the LCFS IS a flux contour); a large
    value means the grid / orientation / units do not match the truth.
    """
    v = _sample(np.asarray(yt_2d, dtype=np.float64), lcfs_px)
    rng = float(np.max(yt_2d) - np.min(yt_2d)) + 1e-12
    return float(np.median(v)), float(100.0 * np.std(v) / rng)


def _nan_shape():
    return dict(kappa=np.nan, delta_upper=np.nan, delta_lower=np.nan, delta=np.nan)


def _efit_field_scores(f, truth, geom, g, psi_b, sign, axis_px, lcfs_cm, region, xp_px):
    """Boundary / axis / shape / X-point scores of ONE field (prediction or true psi) against EFIT."""
    loop_px, bump = loop_around_bumped(f, psi_b, axis_px, sign) if axis_px is not None else (None, np.nan)
    closed = loop_px is not None
    if closed:
        loop_cm = _to_cm(loop_px, geom)
    else:  # no closed loop: use whatever segments exist at psi_b, so the error is large and not silently missing
        loop_cm = extract_lcfs_contour(f, level=psi_b, geom=geom)
    failed = False
    if loop_cm is not None and lcfs_cm is not None:
        b_mean, b_haus = curve_distances(lcfs_cm, loop_cm)
    elif lcfs_cm is not None:  # nothing at psi_b in this field
        b_mean = b_haus = BOUNDARY_FAIL_CM
        failed = True
    else:
        b_mean = b_haus = float("nan")
    rp, zp = get_subpixel_peak(f, region=region, sign=sign)
    axis_cm = float("nan")
    if axis_px is not None:
        axis_cm = float(np.hypot((rp - axis_px[0]) * geom.dr, (zp - axis_px[1]) * geom.dz))
    shp = shape_parameters(loop_cm) if closed else None
    cands = find_xpoints(f, level=psi_b, psi_tol=XPOINT_PSI_TOL_EFIT, side="both")
    xd = [min([np.hypot((c[0] - x[0]) * geom.dr, (c[1] - x[1]) * geom.dz) for c in cands], default=np.inf) for x in xp_px]
    return dict(loop_cm=loop_cm, closed=closed, failed=failed, bump=bump, b_mean=b_mean, b_haus=b_haus, axis_cm=axis_cm,
                shape=shp or _nan_shape(), xcands=cands, xdist=xd)


def _evaluate_efit(yt, yp, truth, frac):
    geom = _geom("efit", truth)
    g = truth.get("grid") or dict(r0=R_MIN_M, dr=DR_CM / 100.0, z0=Z_MIN_M, dz=DZ_CM / 100.0)
    nan = float("nan")
    lcfs = truth.get("lcfs_rz_m")
    axis_rz = np.asarray(truth.get("axis_rz_m", [nan, nan]), dtype=np.float64)
    xp_rz = np.asarray(truth.get("xpoints_rz_m", np.zeros((0, 2))), dtype=np.float64).reshape(-1, 2)
    have_lcfs = lcfs is not None and len(lcfs) >= 8
    have_axis = bool(np.all(np.isfinite(axis_rz)))
    lcfs_px = _px_of(lcfs, g) if have_lcfs else None
    axis_px = _px_of(axis_rz, g)[0] if have_axis else None
    xp_px = _px_of(xp_rz, g) if len(xp_rz) else np.zeros((0, 2))

    if have_lcfs:
        psi_b, spread = efit_psi_b(yt, lcfs_px)
    else:  # no EFIT LCFS for this slice: no physical level, boundary / shape scores are undefined (nan), counted in aggregate
        psi_b, spread = nan, nan
    sign = 1.0
    if have_axis and np.isfinite(psi_b):
        sign = 1.0 if float(_sample(yt, axis_px[None])[0]) >= psi_b else -1.0
    elif have_axis:
        sign = 1.0 if float(_sample(yt, axis_px[None])[0]) >= float(np.median(yt)) else -1.0
    region = _region_mask(lcfs_px, yt.shape) if have_lcfs and not (~_region_mask(lcfs_px, yt.shape)).all() else None
    if region is not None and not region.any():
        region = None
    lcfs_cm = _to_cm(lcfs_px, geom) if have_lcfs else None

    if np.isfinite(psi_b):
        sp = _efit_field_scores(yp, truth, geom, g, psi_b, sign, axis_px, lcfs_cm, region, xp_px)
        so = _efit_field_scores(yt, truth, geom, g, psi_b, sign, axis_px, lcfs_cm, region, xp_px)
    else:  # axis only
        sp = dict(loop_cm=None, closed=False, failed=False, bump=nan, b_mean=nan, b_haus=nan, shape=_nan_shape(), xcands=[], xdist=[np.inf] * len(xp_px),
                  axis_cm=nan)
        so = dict(sp)
        if have_axis:
            for s, f in ((sp, yp), (so, yt)):
                rp, zp = get_subpixel_peak(f, region=region, sign=sign)
                s["axis_cm"] = float(np.hypot((rp - axis_px[0]) * geom.dr, (zp - axis_px[1]) * geom.dz))

    # wall clearance: truth = EFIT LCFS polygon, prediction = predicted loop (PROXY wall, see WALL_BOX_CM)
    d_true = compute_wall_proximity(lcfs_cm) if have_lcfs else nan
    d_pred = compute_wall_proximity(sp["loop_cm"]) if np.isfinite(psi_b) else nan
    fa = bool(np.isfinite(d_true) and np.isfinite(d_pred) and d_pred < SAFETY_CLEARANCE_CM <= d_true)
    ma = bool(np.isfinite(d_true) and np.isfinite(d_pred) and d_true < SAFETY_CLEARANCE_CM <= d_pred)

    # X-points: the primary EFIT X-point is the one whose true-field flux is closest to psi_b (the active separatrix)
    k = len(xp_px)
    if k and np.isfinite(psi_b):
        prim = int(np.argmin(np.abs(_sample(yt, xp_px) - psi_b)))
        x_err = sp["xdist"][prim]
        x_err = nan if not np.isfinite(x_err) else float(x_err)
        x_or = so["xdist"][prim]
        x_or = nan if not np.isfinite(x_or) else float(x_or)
        n_hit = int(sum(d <= XPOINT_HIT_CM for d in sp["xdist"]))
    else:
        x_err = x_or = nan
        n_hit = 0
    kt = truth.get("kappa", nan)
    dt = truth.get("delta", nan)
    return dict(
        boundary_mean_cm=sp["b_mean"], boundary_hausdorff_cm=sp["b_haus"], axis_shift_cm=sp["axis_cm"],
        xpoint_err_cm=x_err, xpoint_true_found=bool(k), xpoint_pred_found=bool(len(sp["xcands"])),
        kappa_true=float(kt), kappa_pred=sp["shape"]["kappa"], kappa_err=abs(sp["shape"]["kappa"] - kt),
        delta_true=float(dt), delta_pred=sp["shape"]["delta"], delta_err=abs(sp["shape"]["delta"] - dt),
        d_wall_true_cm=d_true if np.isfinite(d_true) else 50.0, d_wall_pred_cm=d_pred if np.isfinite(d_pred) else 50.0,
        is_false_alarm=fa, is_missed_alarm=ma,
        mode="efit", psi_b=psi_b, psi_b_spread_pct=spread, efit_lcfs_available=bool(have_lcfs),
        boundary_closed=bool(sp["closed"]), boundary_failed=bool(sp["failed"]), level_bump=sp["bump"],
        delta_upper_pred=sp["shape"]["delta_upper"], delta_lower_pred=sp["shape"]["delta_lower"],
        delta_upper_true=float(truth.get("delta_upper", nan)), delta_lower_true=float(truth.get("delta_lower", nan)),
        xpoint_n_true=int(k), xpoint_n_hit=n_hit, xpoint_oracle_err_cm=x_or, xpoint_true_field_found=bool(len(so["xcands"])),
        boundary_mean_cm_oracle=so["b_mean"], boundary_hausdorff_cm_oracle=so["b_haus"], axis_shift_cm_oracle=so["axis_cm"],
        kappa_oracle=so["shape"]["kappa"], kappa_err_oracle=abs(so["shape"]["kappa"] - kt),
        delta_oracle=so["shape"]["delta"], delta_err_oracle=abs(so["shape"]["delta"] - dt),
        kappa_err_field=abs(sp["shape"]["kappa"] - so["shape"]["kappa"]), delta_err_field=abs(sp["shape"]["delta"] - so["shape"]["delta"]),
    )


def _evaluate_heuristic(yt, yp, frac, mode):
    geom = _geom(mode)
    wall = WALL_BOX_LEGACY_CM if mode == "legacy" else WALL_BOX_CM
    nan = float("nan")
    b_mean_cm, b_hausdorff_cm = compute_boundary_error(yt, yp, frac=frac, geom=geom)
    axis_shift_cm, _, _ = compute_axis_shift(yt, yp, geom=geom)
    xpoint_err_cm, xt_found, xp_found = compute_xpoint_error(yt, yp, frac=frac, geom=geom)
    kt, dt, kp, dp = compute_shape_error(yt, yp, frac=frac, geom=geom)
    level = lcfs_level(yt, frac)
    ct = extract_lcfs_contour(yt, frac=frac, geom=geom)
    cp = None if level is None else extract_lcfs_contour(yp, frac=frac, level=level, geom=geom)
    d_true, d_pred = compute_wall_proximity(ct, wall), compute_wall_proximity(cp, wall)
    return dict(
        boundary_mean_cm=b_mean_cm, boundary_hausdorff_cm=b_hausdorff_cm, axis_shift_cm=axis_shift_cm,
        xpoint_err_cm=xpoint_err_cm, xpoint_true_found=bool(xt_found), xpoint_pred_found=bool(xp_found),
        kappa_true=kt, kappa_pred=kp, kappa_err=abs(kp - kt), delta_true=dt, delta_pred=dp, delta_err=abs(dp - dt),
        d_wall_true_cm=d_true, d_wall_pred_cm=d_pred,
        is_false_alarm=bool(d_pred < SAFETY_CLEARANCE_CM <= d_true),  # predicted breach while the truth was safe
        is_missed_alarm=bool(d_true < SAFETY_CLEARANCE_CM <= d_pred),  # predicted safe while the truth was a breach
        mode=mode, psi_b=level if level is not None else nan, psi_b_spread_pct=nan, efit_lcfs_available=False,
        boundary_closed=bool(cp is not None),
    )


def evaluate_window_physics(yt_2d, yp_2d, frac=LCFS_FRAC, truth=None, mode=None):
    """All physical metrics for one 65x65 window (arrays [z_row, r_col], any affine unit of psi, same unit for both).

    truth: dict from efit_truth.TruthIndex.get(shot_id, window_index) / efit_truth.window_truth. mode: "efit" (needs
    truth, default when given), "heuristic" (default otherwise) or "legacy"; see the module docstring.
    Distances in cm; missing quantities are nan (never a fabricated number), counted by aggregate_physics_metrics.
    """
    yt = np.asarray(yt_2d, dtype=np.float64)
    yp = np.asarray(yp_2d, dtype=np.float64)
    mode = mode or ("efit" if truth is not None else "heuristic")
    if mode == "efit":
        if truth is None:
            raise ValueError("mode='efit' needs the `truth` dict (efit_truth.TruthIndex.get)")
        return _evaluate_efit(yt, yp, truth, frac)
    if mode not in ("heuristic", "legacy"):
        raise ValueError(f"unknown mode {mode!r}")
    return _evaluate_heuristic(yt, yp, frac, mode)


# ---------------------------------------------------------------- aggregation
def _nanmean(x):
    x = np.asarray(x, dtype=np.float64)
    x = x[~np.isnan(x)]
    return float(x.mean()) if x.size else float("nan")


def _col(rs, k):
    return np.array([r.get(k, np.nan) for r in rs], dtype=np.float64)


def aggregate_physics_metrics(results_list):
    """Mean / std across window records. Metrics are averaged over the windows where they exist; counts are returned.

    boundary_* and axis_shift_cm are nan-means (windows without an EFIT LCFS have no boundary score: n_boundary_valid says
    how many do). Predictions with no contour at psi_b count BOUNDARY_FAIL_CM (boundary_fail_rate_pct;
    boundary_mean_cm_valid_only excludes them); boundary_closed_rate_pct is the share of valid windows with a closed
    predicted loop around the axis, and kappa / delta errors exist only for those.
    xpoint_detect_rate_pct: of the windows whose TRUTH has an X-point, the share whose prediction has one within
    XPOINT_HIT_CM of it. xpoint_false_rate_pct: of the windows without a true X-point, the share where the prediction has
    a saddle at the LCFS flux anyway (xpoint_false_rate_truefield_pct is the same detector on the error-free psi, the
    floor of that number). *_oracle keys score the TRUE psi against EFIT (the measurement floor of the method).
    """
    if not results_list:
        return {}
    rs = results_list
    n = len(rs)
    b_means, b_haus, axes, xpoints = (_col(rs, k) for k in ("boundary_mean_cm", "boundary_hausdorff_cm", "axis_shift_cm", "xpoint_err_cm"))
    d_true = _col(rs, "d_wall_true_cm")
    n_safe = int(np.sum(d_true >= SAFETY_CLEARANCE_CM))
    n_hazard = int(np.sum(d_true < SAFETY_CLEARANCE_CM))
    false_alarms = sum(1 for r in rs if r["is_false_alarm"])
    missed_alarms = sum(1 for r in rs if r["is_missed_alarm"])
    valid_b = ~np.isnan(b_means)

    def mean_std(a):
        a = a[~np.isnan(a)]
        return (float(a.mean()), float(a.std())) if a.size else (float("nan"), float("nan"))

    out = {}
    if all("xpoint_true_found" in r for r in rs):
        xt = np.array([r["xpoint_true_found"] for r in rs], bool)
        xpf = np.array([r["xpoint_pred_found"] for r in rs], bool)
        hit = xt & xpf & (xpoints <= XPOINT_HIT_CM)
        out.update(
            xpoint_true_count=int(xt.sum()),
            xpoint_detect_rate_pct=float(100.0 * hit.sum() / xt.sum()) if xt.any() else float("nan"),
            xpoint_false_rate_pct=float(100.0 * (~xt & xpf).sum() / (~xt).sum()) if (~xt).any() else float("nan"),
            xpoint_err_n=int(np.sum(~np.isnan(xpoints))),
            kappa_err=_nanmean(_col(rs, "kappa_err")),
            kappa_err_rel_pct=_nanmean(100.0 * _col(rs, "kappa_err") / _col(rs, "kappa_true")),
            delta_err=_nanmean(_col(rs, "delta_err")),
            kappa_true_mean=_nanmean(_col(rs, "kappa_true")), delta_true_mean=_nanmean(_col(rs, "delta_true")),
            shape_valid_n=int(np.sum(~np.isnan(_col(rs, "kappa_err")))),
        )
    if any(r.get("mode") == "efit" for r in rs):
        ef = [r for r in rs if r.get("mode") == "efit"]
        xt = np.array([r["xpoint_true_found"] for r in ef], bool)
        xo = np.array([r["xpoint_true_field_found"] for r in ef], bool)
        out.update(
            n_efit_windows=len(ef),
            n_boundary_valid=int(valid_b.sum()),
            boundary_closed_rate_pct=float(100.0 * np.mean([r["boundary_closed"] for r in ef if not np.isnan(r["boundary_mean_cm"])]))
            if valid_b.any() else float("nan"),
            boundary_fail_rate_pct=float(100.0 * np.mean([r["boundary_failed"] for r in ef if not np.isnan(r["boundary_mean_cm"])]))
            if valid_b.any() else float("nan"),
            boundary_mean_cm_valid_only=_nanmean([r["boundary_mean_cm"] for r in ef if not r["boundary_failed"]]),
            psi_b_spread_pct_mean=_nanmean(_col(ef, "psi_b_spread_pct")),
            psi_b_spread_pct_max=float(np.nanmax(_col(ef, "psi_b_spread_pct"))) if np.isfinite(_col(ef, "psi_b_spread_pct")).any() else float("nan"),
            xpoint_false_rate_truefield_pct=float(100.0 * (~xt & xo).sum() / (~xt).sum()) if (~xt).any() else float("nan"),
            xpoint_oracle_err_cm=_nanmean(_col(ef, "xpoint_oracle_err_cm")),
            boundary_mean_cm_oracle=_nanmean(_col(ef, "boundary_mean_cm_oracle")),
            boundary_hausdorff_cm_oracle=_nanmean(_col(ef, "boundary_hausdorff_cm_oracle")),
            axis_shift_cm_oracle=_nanmean(_col(ef, "axis_shift_cm_oracle")),
            kappa_err_oracle=_nanmean(_col(ef, "kappa_err_oracle")), delta_err_oracle=_nanmean(_col(ef, "delta_err_oracle")),
            kappa_err_field=_nanmean(_col(ef, "kappa_err_field")), delta_err_field=_nanmean(_col(ef, "delta_err_field")),
        )
    bm, bs = mean_std(b_means)
    am, asd = mean_std(axes)
    hs = b_haus[~np.isnan(b_haus)]
    return dict(
        n_windows=n,
        boundary_mean_cm=bm, boundary_mean_cm_std=bs,
        boundary_hausdorff_cm=_nanmean(b_haus), boundary_hausdorff_cm_p95=float(np.percentile(hs, 95)) if hs.size else float("nan"),
        axis_shift_cm=am, axis_shift_cm_std=asd,
        xpoint_err_cm=_nanmean(xpoints),
        false_alarm_count=false_alarms, false_alarm_rate_pct=float(false_alarms / max(1, n_safe) * 100.0),
        missed_alarm_count=missed_alarms, missed_alarm_rate_pct=float(missed_alarms / max(1, n_hazard) * 100.0),
        **out,
    )


if __name__ == "__main__":
    # Self-test on a synthetic Gaussian: a one-pixel shift in R is DR_CM = 3.0 cm
    rr, zz = np.meshgrid(np.linspace(0, 1, GRID_SIZE), np.linspace(0, 1, GRID_SIZE))
    y_true = np.exp(-((rr - 0.5) ** 2 + (zz - 0.5) ** 2) / 0.05)
    y_pred = np.exp(-((rr - 0.5 - 1.0 / 64) ** 2 + (zz - 0.5) ** 2) / 0.05)
    for k, v in evaluate_window_physics(y_true, y_pred).items():
        print(f"  {k:25}: {v}")


In [ ]:
%%writefile /kaggle/working/efit_truth.py
#!/usr/bin/env python
"""EFIT ground truth for TokaMark task_1-3 windows (Module 1 v2).

TokaMark ships the EFIT reconstruction behind the psi target: magnetic axis, LCFS polygon, X-points, elongation and
triangularity (all in metres, on the zarr `equilibrium` group). This module loads them and aligns them with the
(shot_id, window_index) pairs that `TokaMarkDataset` emits, so a predicted psi can be scored against EFIT itself and not
against a flux-fraction heuristic.

Alignment is a re-implementation of TokaMarkDataset._process_shot/_build_window for task_1-3 (input 5 ms, output 5 ms,
delta -5 ms, stride 5 ms as set in pipeline.py, test_mode=True sparse filter, MastDataset ip_rating mask). It is checked
against the real dataset in `xpoint_diagnostic.py --prove-alignment` and in test_physics_evaluation.py.

Only numpy and zarr are needed (zarr 2 or 3). Arrays are [n_t, ...]; psi is [n_t, z_row, r_col].
"""
import os

import numpy as np

# the 8 input groups of task_1-3 with the nominal dt that TokaMark's metadata uses for padding (random_split_signals_stats)
INPUT_SIGNALS = (
    ("magnetics", "flux_loop_flux", 0.0002),
    ("magnetics", "b_field_pol_probe_ccbv_field", 0.0002),
    ("magnetics", "b_field_pol_probe_obr_field", 0.0002),
    ("magnetics", "b_field_pol_probe_obv_field", 0.0002),
    ("magnetics", "b_field_tor_probe_saddle_voltage", 0.00002),
    ("pf_active", "coil_current", 0.00025),
    ("pf_active", "solenoid_current", 0.00025),
    ("summary", "ip", 0.00025),
)
PSI_DT = 0.005
INPUT_LENGTH, OUTPUT_LENGTH, DELTA, STRIDE = 0.005, 0.005, -0.005, 0.005  # task_1-3 yaml + pipeline.py stride override


def open_shot(zarr_path):
    import zarr
    try:
        return zarr.open_group(str(zarr_path), mode="r", use_consolidated=False)
    except TypeError:  # zarr 2
        return zarr.open_group(str(zarr_path), mode="r")


def _arr(g, path):
    try:
        return np.asarray(g[path][...])
    except KeyError:
        return None


def _time_of(g, group, name):
    """(time, values) of a signal; time is the dimension starting with 'time' (last axis), as TokaMark loads it."""
    try:
        a = g[f"{group}/{name}"]
    except KeyError:
        return np.array([]), np.array([])
    dims = a.attrs.get("_ARRAY_DIMENSIONS", [])
    td = [d for d in dims if str(d).startswith("time")]
    if not td:
        return np.array([]), np.array([])
    v = np.asarray(a[...])
    if v.ndim == 1:
        v = v[None, :]
    return np.asarray(g[f"{group}/{td[0]}"][...], dtype=np.float64), v


def geometry(g):
    """Grid of the psi map from the zarr: dict(r0, dr, z0, dz, n_r, n_z) in metres. Raises when the grid is not uniform."""
    R = np.asarray(g["equilibrium/major_radius"][...], dtype=np.float64)
    Z = np.asarray(g["equilibrium/z"][...], dtype=np.float64)
    dr, dz = np.diff(R), np.diff(Z)
    if not (np.allclose(dr, dr[0], rtol=1e-4) and np.allclose(dz, dz[0], rtol=1e-4)):
        raise ValueError("EFIT grid is not uniform")
    return dict(r0=float(R[0]), dr=float(dr[0]), z0=float(Z[0]), dz=float(dz[0]), n_r=len(R), n_z=len(Z))


def load_efit_truth(zarr_path):
    """All EFIT slices of one shot (metres, native EFIT times), or None when the shot has no equilibrium/psi.

    keys: shot_id, grid, t [n_t], psi [n_t, n_z, n_r] (raw Wb/rad, NaN where EFIT has no solution), axis_rz [n_t, 2],
    lcfs [list of n_t arrays (n, 2) in (R, Z) or None], xpoints [n_t, 2, 2] (slot, (R, Z); NaN = absent),
    kappa, kappa_axis, delta_upper, delta_lower, minor_radius, ip_rating [n_t].
    """
    g = open_shot(zarr_path)
    try:
        psi = np.asarray(g["equilibrium/psi"][...], dtype=np.float64)
    except KeyError:
        return None
    t = np.asarray(g["equilibrium/time"][...], dtype=np.float64)
    n_t = len(t)
    psi = np.moveaxis(psi, -1, 0)  # zarr dims are (z, major_radius, time); TokaMark moves time to the front
    lr, lz = _arr(g, "equilibrium/lcfs_r"), _arr(g, "equilibrium/lcfs_z")
    lcfs = []
    for i in range(n_t):
        if lr is None:
            lcfs.append(None)
            continue
        ok = np.isfinite(lr[:, i]) & np.isfinite(lz[:, i])
        lcfs.append(np.column_stack([lr[ok, i], lz[ok, i]]) if ok.sum() >= 8 else None)
    xr, xz = _arr(g, "equilibrium/x_point_r"), _arr(g, "equilibrium/x_point_z")
    xp = np.full((n_t, 2, 2), np.nan)
    if xr is not None:
        xp[:, :, 0], xp[:, :, 1] = xr.T[:, :2], xz.T[:, :2]

    def col(name):
        a = _arr(g, "equilibrium/" + name)
        return np.full(n_t, np.nan) if a is None else a.astype(np.float64)

    sid = os.path.basename(os.path.normpath(str(zarr_path)))
    sid = int(sid.split(".")[0]) if sid.split(".")[0].isdigit() else -1
    return dict(
        shot_id=sid, grid=geometry(g), t=t, psi=psi,
        axis_rz=np.column_stack([col("magnetic_axis_r"), col("magnetic_axis_z")]),
        lcfs=lcfs, xpoints=xp, kappa=col("elongation"), kappa_axis=col("elongation_axis"),
        delta_upper=col("triangularity_upper"), delta_lower=col("triangularity_lower"),
        minor_radius=col("minor_radius"), ip_rating=col("ip_rating"),
    )


def _pad_counts(t0, t_last, dt, t_start, t_end):
    """(n_left, n_right) pad samples of TokaMarkDataset._pad_time_series_to_interval."""
    n_l = int(np.ceil((t0 - t_start) / dt)) + 1 if t0 > t_start else 0
    n_r = int(np.ceil((t_end - t_last) / dt)) + 1 if t_last < t_end else 0
    return n_l, n_r


def enumerate_windows(zarr_path, outliers=(), truth=None):
    """List of dicts, one per window that TokaMarkDataset(test_mode=True) emits for this shot, in dataset order.

    Each has window_index (the dataset's idx_t, counting dropped windows too), t_cut, t_psi, slice_index (row of the EFIT
    arrays that is the psi target). Windows the dataset drops (psi slice all NaN, or every input group all-NaN) are
    absent. Returns [] for a shot without equilibrium, like the dataset (which cannot find an output).
    """
    truth = truth or load_efit_truth(zarr_path)
    if truth is None or "equilibrium-psi" in outliers:
        return []
    g = open_shot(zarr_path)
    # MastDataset: remove_bad_efit_rating masks psi where ip_rating == 0 (and drops all equilibrium signals if no ip_rating)
    ipr = truth["ip_rating"]
    if np.all(np.isnan(ipr)) and _arr(g, "equilibrium/ip_rating") is None:
        return []
    psi = truth["psi"].copy()
    psi[ipr == 0] = np.nan
    t_psi = truth["t"]
    psi_ok = ~np.all(np.isnan(psi), axis=(1, 2))  # per-slice "has data"
    if not psi_ok.any():
        return []
    sig = []  # (times, has_data[T], dt) per input group; empty groups stay empty (all-NaN windows)
    for grp, name, dt in INPUT_SIGNALS:
        if f"{grp}-{name}" in outliers:
            sig.append((np.array([]), np.array([], bool), dt))
            continue
        tt, vv = _time_of(g, grp, name)
        if tt.size == 0 or vv.size == 0:
            sig.append((np.array([]), np.array([], bool), dt))
            continue
        sig.append((tt, ~np.all(np.isnan(vv), axis=tuple(range(vv.ndim - 1))), dt))
    starts = [tt[ok][0] for tt, ok, _ in sig if tt.size and ok.any()]
    if not starts:
        return []
    t_start = float(np.min(starts))
    t_end = float(t_psi[psi_ok][-1])
    t_cut_start = t_start + INPUT_LENGTH
    t_cut_stop = t_end - DELTA - OUTPUT_LENGTH
    if t_cut_start >= t_cut_stop:
        return []

    def padded(tt, ok, dt):
        if tt.size == 0:
            return tt, ok, 0
        n_l, n_r = _pad_counts(tt[0], tt[-1], dt, t_start, t_end)
        t_all = tt
        if n_l:
            t_all = np.concatenate([tt[0] - dt * np.arange(n_l, 0, -1), t_all])
        if n_r:
            t_all = np.concatenate([t_all, tt[-1] + dt * np.arange(1, n_r + 1)])
        return t_all, np.concatenate([np.zeros(n_l, bool), ok, np.zeros(n_r, bool)]), n_l

    sig = [padded(*s) + (s[2],) for s in sig]  # (t_pad, ok_pad, n_left, dt)
    t_psi_pad, ok_pad, n_left = padded(t_psi, psi_ok, PSI_DT)
    out = []
    for idx_t, t_cut in enumerate(np.arange(t_cut_start, stop=t_cut_stop, step=STRIDE)):
        s0 = int(round((t_cut + DELTA - t_psi_pad[0]) / PSI_DT))
        s0 = int(np.clip(s0, 0, len(t_psi_pad) - 1))
        if not ok_pad[s0]:
            continue  # output all-NaN -> dropped
        any_input = False
        for tt, ok, _, dt in sig:
            if tt.size == 0:
                continue
            end = int(round((t_cut - tt[0]) / dt))
            n_in = int(round(INPUT_LENGTH / dt))
            ids = np.clip(np.arange(end - n_in, end), 0, len(tt) - 1)
            if ok[ids].any():
                any_input = True
                break
        if not any_input:
            continue
        j = s0 - n_left
        out.append(dict(window_index=idx_t, t_cut=float(t_cut), t_psi=float(t_psi[j]), slice_index=j))
    return out


def window_truth(truth, w):
    """Truth dict of one window: the EFIT slice `w['slice_index']`, plus window bookkeeping. Units: metres, raw psi."""
    j = w["slice_index"]
    xp = truth["xpoints"][j]
    xp = xp[np.all(np.isfinite(xp), axis=1)]
    return dict(
        shot_id=truth["shot_id"], window_index=w["window_index"], t_cut=w["t_cut"], t_psi=w["t_psi"], slice_index=j,
        grid=truth["grid"], psi_raw=truth["psi"][j], axis_rz_m=truth["axis_rz"][j], lcfs_rz_m=truth["lcfs"][j],
        xpoints_rz_m=xp, n_xpoints=int(len(xp)), kappa=float(truth["kappa"][j]), kappa_axis=float(truth["kappa_axis"][j]),
        delta_upper=float(truth["delta_upper"][j]), delta_lower=float(truth["delta_lower"][j]),
        delta=float(np.nanmean([truth["delta_upper"][j], truth["delta_lower"][j]])) if np.isfinite(
            [truth["delta_upper"][j], truth["delta_lower"][j]]).any() else float("nan"),
        minor_radius_m=float(truth["minor_radius"][j]), ip_rating=float(truth["ip_rating"][j]),
    )


class TruthIndex:
    """EFIT truth for every window of a directory of `<shot>.zarr` stores.

        ti = TruthIndex(data_dir)                    # data_dir/<shot_id>.zarr
        tr = ti.get(shot_id, window_index)           # dict (see window_truth) or None if the dataset never emits it
        tr = ti.by_tcut(shot_id, t_cut)              # same, keyed by the window's t_cut
        evaluate_window_physics(yt, yp, truth=tr)

    `outlier_yaml` (MAST_tools/metadata/<split>_split_outlier_metadata.yaml, or env TM_OUTLIER_YAML) replaces flagged
    signals by empty ones exactly like MastDataset(remove_outliers=True); without it no signal is treated as an outlier.
    """

    def __init__(self, data_dir, outlier_yaml=None):
        self.data_dir = str(data_dir)
        self._outliers = {}
        path = outlier_yaml or os.environ.get("TM_OUTLIER_YAML")
        if path and os.path.exists(path):
            import yaml
            self._outliers = {int(k): set(v or []) for k, v in (yaml.safe_load(open(path)) or {}).items()}
        self._cache = {}

    def shot_path(self, shot_id):
        return os.path.join(self.data_dir, f"{int(shot_id)}.zarr")

    def shot_ids(self):
        return sorted(int(f[:-5]) for f in os.listdir(self.data_dir) if f.endswith(".zarr") and f[:-5].isdigit())

    def windows(self, shot_id):
        """{window_index: truth dict} of one shot ({} if the shot has no equilibrium or the dataset drops it)."""
        sid = int(shot_id)
        if sid not in self._cache:
            p = self.shot_path(sid)
            tr = load_efit_truth(p) if os.path.isdir(p) else None
            ws = enumerate_windows(p, outliers=self._outliers.get(sid, ()), truth=tr) if tr is not None else []
            self._cache[sid] = {w["window_index"]: window_truth(tr, w) for w in ws}
        return self._cache[sid]

    def get(self, shot_id, window_index, default=None):
        return self.windows(shot_id).get(int(window_index), default)

    def by_tcut(self, shot_id, t_cut, tol=1e-6, default=None):
        for w in self.windows(shot_id).values():
            if abs(w["t_cut"] - float(t_cut)) <= tol:
                return w
        return default


# MastDataset standardises psi per signal with the split's global mean / std (tokamark/metadata/<split>_split_signals_stats.yaml)
PSI_STATS = {"random": (-0.026102678542290144, 0.054930705723548384), "temporal": (-0.02704797707091641, 0.06492857922672353)}


def check_alignment(rows, data_dir, split="random", outlier_yaml=None, atol=1e-9):
    """Compare real TokaMarkDataset windows with this module's index.

    rows: [(shot_id, window_index, y)] where y is the 65x65 target the dataset emitted (standardised psi). For every shot in
    rows the set of window indices must equal TruthIndex's, and y must equal (psi_raw - mean) / std of the slice it names.
    Returns (n_windows_matching, problems) with problems a list of strings (empty when aligned).
    """
    mean, std = PSI_STATS[split]
    ti = TruthIndex(data_dir, outlier_yaml=outlier_yaml)
    by_shot = {}
    for sid, wi, y in rows:
        by_shot.setdefault(int(sid), {})[int(wi)] = np.asarray(y)
    ok, problems = 0, []
    for sid, ds in by_shot.items():
        mine = ti.windows(sid)
        if set(ds) != set(mine):
            problems.append(f"shot {sid}: dataset windows {sorted(set(ds) - set(mine))} not indexed, indexed {sorted(set(mine) - set(ds))} not emitted")
        for wi in sorted(set(ds) & set(mine)):
            ref = (mine[wi]["psi_raw"] - mean) / std
            if np.allclose(ds[wi], ref, atol=atol, rtol=0, equal_nan=True):
                ok += 1
            else:
                problems.append(f"shot {sid} window {wi}: max|dy| = {np.nanmax(np.abs(ds[wi] - ref)):.3g}")
    return ok, problems


In [ ]:
%%writefile /kaggle/working/aggregate_seeds.py
#!/usr/bin/env python
"""Multi-seed aggregation of robust_metrics.json files (numpy only, no torch/pipeline import).

    python aggregate_seeds.py OUT.json RUN_DIR [RUN_DIR ...]      # RUN_DIR holds robust_metrics.json (or is the json itself)

Output: {"n_seeds", "sources", "cells": {cell: {"NRMSE": {mean,std,n}, "boundary_mean_cm": {...}, "RS": ..., ...}}}
cell is "clean" or "family|param|impute" (same ids as robust_metrics.json["corrupted"]). std is the sample std
(ddof=1) across seeds, 0.0 when n == 1. RS = clean NRMSE / cell NRMSE of the SAME run (per seed), then averaged.

Also provides bootstrap_ci(): the cluster (shot-level) bootstrap used for boundary-error CIs inside one run.
"""
import json
import os
import sys

import numpy as np

METRICS = ("NRMSE", "NMAE", "RMSE", "MAE")
PHYS_KEYS = ("boundary_mean_cm", "boundary_hausdorff_cm", "axis_shift_cm", "xpoint_detect_rate_pct", "false_alarm_rate_pct",
             "missed_alarm_rate_pct", "boundary_fail_rate_pct")


def bootstrap_ci(shots, values, n_boot=1000, alpha=0.05, seed=0):
    """Mean of `values` with a percentile CI from resampling whole shots (windows of one shot are correlated).
    NaNs are ignored. Returns dict(mean, lo, hi, n, n_shots)."""
    shots = np.asarray(shots)
    v = np.asarray(values, dtype=float)
    ok = ~np.isnan(v)
    shots, v = shots[ok], v[ok]
    if v.size == 0:
        return dict(mean=float("nan"), lo=float("nan"), hi=float("nan"), n=0, n_shots=0)
    uniq, inv = np.unique(shots, return_inverse=True)
    s = np.bincount(inv, weights=v, minlength=len(uniq))  # per-shot sum and count -> resampled mean without loops over windows
    c = np.bincount(inv, minlength=len(uniq)).astype(float)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(uniq), size=(n_boot, len(uniq)))
    means = s[idx].sum(1) / c[idx].sum(1)
    lo, hi = np.percentile(means, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return dict(mean=float(v.mean()), lo=float(lo), hi=float(hi), n=int(v.size), n_shots=int(len(uniq)))


def _ms(xs):
    xs = np.asarray([x for x in xs if x is not None and not np.isnan(x)], dtype=float)
    if xs.size == 0:
        return None
    return dict(mean=float(xs.mean()), std=float(xs.std(ddof=1)) if xs.size > 1 else 0.0, n=int(xs.size))


def _cell(m, clean_nrmse):
    out = {k: m.get(k) for k in METRICS}
    out["RS"] = clean_nrmse / m["NRMSE"] if m.get("NRMSE") else None
    ph = m.get("PHYS") or {}
    for k in PHYS_KEYS:
        out[k] = ph.get(k)
    out["n_phys_windows"] = ph.get("n_windows")
    return out


def load_run(path):
    p = path if path.endswith(".json") else os.path.join(path, "robust_metrics.json")
    return json.load(open(p))


def aggregate(paths):
    runs = [load_run(p) for p in paths]
    cells = {}
    for r in runs:
        cn = r["clean"]["NRMSE"]
        for cid, m in [("clean", r["clean"])] + list(r["corrupted"].items()):
            cells.setdefault(cid, []).append(_cell(m, cn if cid != "clean" else cn))
    agg = {cid: {k: _ms([c[k] for c in lst]) for k in lst[0]} for cid, lst in cells.items()}
    return dict(n_seeds=len(runs), sources=[str(p) for p in paths], cells=agg)


def main(argv):
    if len(argv) < 3:
        raise SystemExit(__doc__)
    res = aggregate(argv[2:])
    json.dump(res, open(argv[1], "w"), indent=1)
    print(f"{res['n_seeds']} run(s), {len(res['cells'])} cells -> {argv[1]}")


if __name__ == "__main__":
    main(sys.argv)


In [ ]:
%%writefile /kaggle/working/adaptive_mitigation.py
#!/usr/bin/env python
"""Module 4 (v2): Adaptive failure mitigation with calibrated confidence.  See MODULE4_V2.md.

  Step 1  detect     FaultDetector (v2): local rules (NaN / zero gaps, flat-line runs and flat variance, spikes) +
                     cross-sensor residuals (every channel predicted from all the others by a ridge/Gaussian
                     conditional fitted on clean windows; stuck / drift / offset show up as standardised residuals)
                     + flux-loop residual against the model's psi (FluxLoopCheck). Faults are isolated one at a time
                     (largest standardised residual first, then removed from the predictor set and re-scored), so a
                     fault does not smear onto its neighbours. Legacy per-branch RealTimeFaultDetector is kept.
  Step 2  route      RouterV2: by fault duration and sensor criticality tier
                         short transient              -> forward-fill
                         long fault, redundant sensor -> virtual sensor (ridge from healthy neighbours)
                         critical sensor / multi-sensor outage / whole branch -> failure-aware model with the mask
                         beyond what that model was trained for (or no model) -> PHYSICS FALLBACK (linear flux
                                                         reconstruction from the surviving measurements, not EFIT)
                     Oracle mode takes the true fault labels instead of the detector (upper bound of detector+router).
  Step 3  gauge      ConformalGauge (split conformal, per route, kept for physics_study.py) and AdaptiveConformalGauge
                     (difficulty-normalised, locally adaptive), plus a HIGH / LOW / ABSTAIN flag for the controller.

Window format: list with one array per sensor branch, each (T_b, P_b); standardised with the train statistics. The
legacy classes (RealTimeFaultDetector, AdaptiveRouter, ConformalGauge, SmartAdaptiveMitigationBrain(tiers, gauge,
known_missing).process(window, model_fn)) behave exactly as before when no v2 component is passed.
Nothing here is tuned to a target number; thresholds come from clean calibration windows, latencies are measured.
"""
import math
import time
from dataclasses import dataclass, field

import numpy as np

try:
    from physics_evaluation import SAFETY_CLEARANCE_CM, compute_wall_proximity, extract_lcfs_contour
except Exception:  # synthetic tests of detection / routing do not need the physics layer
    SAFETY_CLEARANCE_CM, compute_wall_proximity, extract_lcfs_contour = 5.0, None, None
from fault_models import WINDOW_MS, FaultLabels, FaultSpec, WindowLayout, as_tp
from virtual_sensor import LinearFluxFallback, VirtualSensorBank

SPIKE_SIGMA = 6.5       # |x| above this (in standardised units) is an impulse
SATURATION_SIGMA = 4.5  # a flat probe stuck beyond this is a saturated amplifier, not a dead one
ROUTES = ("healthy", "transient_ffill", "redundant_interp", "module2")  # legacy router, in increasing severity
ROUTES_V2 = ("healthy", "transient_ffill", "virtual_sensor", "module2", "physics_fallback")
# Tier per branch [flux_loop, ccbv, obr, obv, saddle_v, coil_i, solenoid_i, ip] from module 3's last published
# result (AGENTS.md); pass `tiers=` (from criticality.json) to override once module 3 is rebuilt.
DEFAULT_TIERS = {0: 3, 1: 1, 2: 2, 3: 2, 4: 3, 5: 2, 6: 1, 7: 2}


def _as_tp(a):
    tp, restore = as_tp(a)
    return tp, restore


# ====================================================================== legacy (unchanged behaviour)
class RealTimeFaultDetector:
    """Legacy step 1. Per probe: STUCK_ZERO | RAIL_SATURATION | IMPULSE_SPIKE | TEMPORAL_GAP | HEALTHY.

    known_missing: optional list (one (P,) bool array per branch) of probes that are permanently absent from the
    dataset (always zero-filled, so the model was trained without them). They are not faults."""

    def __init__(self, known_missing=None):
        self.known_missing = known_missing

    def inspect(self, window):
        """Returns one dict per branch: dead (P,) bool, transient (T, P) bool samples, diagnoses {probe: label}."""
        report = []
        for i, branch in enumerate(window):
            tp, _ = _as_tp(branch)
            finite = np.isfinite(tp)
            clean = np.where(finite, tp, 0.0)
            dead = (~finite).all(axis=0) | (clean.var(axis=0) < 1e-9)           # flat for the whole window
            saturated = dead & (np.abs(clean.mean(axis=0)) > SATURATION_SIGMA)   # flat and far from the mean
            spikes = np.abs(clean) > SPIKE_SIGMA
            missing = (clean == 0.0) | ~finite
            pair = missing[:-1] & missing[1:]                                    # >= 2 consecutive missing samples
            gap = np.zeros_like(missing)
            gap[:-1] |= pair
            gap[1:] |= pair
            absent = np.asarray(self.known_missing[i], dtype=bool) if self.known_missing is not None else np.zeros_like(dead)
            dead = dead & ~absent
            transient = (spikes | gap | ~finite) & ~(dead | absent)[None, :]
            diagnoses = {int(p): "RAIL_SATURATION" if saturated[p] else "STUCK_ZERO" for p in np.flatnonzero(dead)}
            for p in np.flatnonzero(transient.any(axis=0)):
                diagnoses[int(p)] = "IMPULSE_SPIKE" if spikes[:, p].any() else "TEMPORAL_GAP"
            report.append(dict(dead=dead, transient=transient, diagnoses=diagnoses))
        return report


class AdaptiveRouter:
    """Legacy step 2. `tiers[branch]` = 1 (safety-critical), 2 (shaping) or 3 (redundant); unknown branches count as 2."""

    def __init__(self, tiers=None):
        self.tiers = tiers or {}

    @staticmethod
    def _ffill(tp, bad):
        out = tp.copy()
        for p in np.flatnonzero(bad.any(axis=0)):
            good = np.flatnonzero(~bad[:, p])
            if good.size == 0:
                continue
            last = tp[good[0], p]  # a leading gap takes the first healthy sample
            for t in range(tp.shape[0]):
                if bad[t, p]:
                    out[t, p] = last
                else:
                    last = tp[t, p]
        return out

    @staticmethod
    def _neighbour_interp(tp, dead):
        """Replace each dead probe by the linear interpolation of its nearest healthy probes along the probe axis."""
        out = tp.copy()
        good = np.flatnonzero(~dead)
        for p in np.flatnonzero(dead):
            lo, hi = good[good < p], good[good > p]
            if lo.size and hi.size:
                a, b = lo.max(), hi.min()
                w = (p - a) / (b - a)
                out[:, p] = (1 - w) * tp[:, a] + w * tp[:, b]
            else:
                out[:, p] = tp[:, (lo.max() if lo.size else hi.min())]
        return out

    def route(self, window, report):
        """Returns (repaired window, health mask (n_branches,), route label, per-branch action log)."""
        n = len(window)
        mask = np.ones(n, dtype=np.float32)
        repaired, actions, level = [], {}, 0
        for i, (branch, rep) in enumerate(zip(window, report)):
            tp, restore = _as_tp(branch)
            tp = tp.copy()
            dead, tier = rep["dead"], self.tiers.get(i, 2)
            if rep["transient"].any():
                tp = self._ffill(tp, rep["transient"] & ~dead[None, :])
                actions.setdefault(i, []).append("forward-fill")
                level = max(level, 1)
            if dead.any():
                healthy_left = (~dead).any()
                if tier == 3 and healthy_left:
                    tp = self._neighbour_interp(tp, dead)
                    actions.setdefault(i, []).append(f"neighbour-interp x{int(dead.sum())}")
                    level = max(level, 2)
                else:  # Tier 1/2 probe or a whole branch lost: hand over to the failure-aware network
                    mask[i] = 0.0
                    tp[:] = 0.0
                    actions.setdefault(i, []).append("module2-mask")
                    level = 3
            repaired.append(restore(tp))
        return repaired, mask, ROUTES[level], actions


# ====================================================================== v2 detection
@dataclass
class Detection:
    """Per-channel detector output on one window (canonical channel matrix, T samples)."""
    flagged: np.ndarray            # (C,) bool
    score: np.ndarray              # (C,) float, higher = more suspicious; flagged channels rank above all others
    kind: np.ndarray               # (C,) object: dropout | flatline | rail | spike | drift | offset | flux_residual | ''
    onset_ms: np.ndarray           # (C,) first altered sample (nan if healthy)
    duration_ms: np.ndarray        # (C,) altered span (nan if healthy)
    sample_mask: np.ndarray        # (T, C) bool: samples to repair
    order: list = field(default_factory=list)   # flagged channels in isolation order (most certain first)
    resid_score: float = 0.0       # largest un-flagged-normalised residual ratio (difficulty proxy for the gauge)
    absent: np.ndarray = None      # (C,) known permanently absent channels (never faults)


def detection_from_labels(labels: FaultLabels, window_ms=WINDOW_MS):
    """Oracle: build a Detection from the ground-truth labels of the injector (no detector involved)."""
    C = len(labels.kind)
    fl = labels.faulty
    score = np.where(fl, 1e3, 0.0)
    return Detection(fl, score, labels.kind.copy(), labels.onset_ms.copy(), labels.duration_ms.copy(),
                     labels.mask.copy(), list(np.flatnonzero(fl)), 0.0, np.zeros(C, bool))


def _longest_run(B):
    """B (T, C) bool -> (length (C,), end index (C,)) of the longest run of True per column."""
    T, C = B.shape
    cur = np.zeros(C, int)
    best = np.zeros(C, int)
    end = np.zeros(C, int)
    for t in range(T):
        cur = np.where(B[t], cur + 1, 0)
        up = cur > best
        best = np.where(up, cur, best)
        end = np.where(up, t, end)
    return best, end


def _pot_threshold(v, p, q0=0.95):
    """Threshold exceeded with probability ~p: peaks-over-threshold with an exponential tail above the q0 quantile."""
    v = np.asarray(v, float)
    v = v[np.isfinite(v)]
    if v.size == 0:
        return np.inf
    u = np.quantile(v, q0)
    ex = v[v > u] - u
    if ex.size < 10:
        return float(np.quantile(v, 1 - min(p, 1.0))) if v.size * p >= 1 else float(v.max() * 1.2)
    return float(u + ex.mean() * math.log((1 - q0) / p))


def _stats(R):
    """R (..., T, C) standardised residuals -> peak |r|, rms, |total change across the window| (each (..., C))."""
    T = R.shape[-2]
    t = np.arange(T) - (T - 1) / 2.0
    slope = np.tensordot(R, t, axes=([-2], [0])) / max(np.sum(t * t), 1e-12)
    return np.abs(R).max(axis=-2), np.sqrt((R ** 2).mean(axis=-2)), np.abs(slope) * (T - 1)


def _remove(P, c):
    """Marginalise variable c out of a precision matrix (rank-1 Schur update); row/col c become 0."""
    if P[c, c] > 1e-12:
        P = P - np.outer(P[:, c], P[c, :]) / P[c, c]
    P[c, :] = 0.0
    P[:, c] = 0.0
    return P


class FluxLoopCheck:
    """Flux-loop residual: measured loop flux vs the reconstruction's psi at the loop pixels.

    The two live in different units / offsets (loop flux vs standardised psi), so each loop gets a fitted linear
    map psi_pixel -> flux and a robust residual sigma from clean windows; a loop is faulty when its standardised
    residual exceeds a pooled peaks-over-threshold quantile of clean residuals. Its value is a second opinion that
    does not depend on the cross-sensor model. It sees what the reconstruction contradicts, so it is weaker when
    the same faulty loop fed the reconstruction (the fix is to run it on a model that had the fault masked).

    loop_pixels: list of (row, col) indices into psi. For pipeline.FLUX_LOOP_COORDS (r_idx, z_idx) use
    `loop_pixels_from_coords`. loop_channels: the matching channel indices in the canonical matrix."""

    def __init__(self, loop_pixels, loop_channels, fp_rate=1e-3):
        self.pix = np.asarray(loop_pixels, int)
        self.channels = np.asarray(loop_channels, int)
        self.fp_rate = fp_rate
        self.a = self.b = self.s = None
        self.thr = np.inf

    def psi_at_loops(self, psi):
        return np.asarray(psi)[self.pix[:, 0], self.pix[:, 1]]

    def fit(self, flux_means, psi_maps, calib=None):
        """flux_means (N, L) measured loop flux per window; psi_maps (N, H, W) matching clean reconstructions."""
        Y = np.asarray(flux_means, float)
        Xp = np.stack([self.psi_at_loops(p) for p in psi_maps])
        self.a, self.b, self.s = np.zeros(Y.shape[1]), np.zeros(Y.shape[1]), np.ones(Y.shape[1])
        for l in range(Y.shape[1]):
            ok = np.isfinite(Y[:, l])
            A = np.stack([Xp[ok, l], np.ones(ok.sum())], 1)
            self.a[l], self.b[l] = np.linalg.lstsq(A, Y[ok, l], rcond=None)[0]
            res = Y[ok, l] - A @ np.array([self.a[l], self.b[l]])
            self.s[l] = max(1.4826 * np.median(np.abs(res - np.median(res))), 1e-9)
        cf, cp = (Y, Xp) if calib is None else (np.asarray(calib[0], float), np.stack([self.psi_at_loops(p) for p in calib[1]]))
        z = np.abs((cf - (self.a * cp + self.b)) / self.s)
        self.thr = _pot_threshold(z, self.fp_rate)
        return self

    def residuals(self, flux_mean, psi):
        return (np.asarray(flux_mean, float) - (self.a * self.psi_at_loops(psi) + self.b)) / self.s


def loop_pixels_from_coords(coords):
    """pipeline.FLUX_LOOP_COORDS is (r_idx, z_idx) and psi is indexed [z, r]; returns (row, col) = (z_idx, r_idx)."""
    return [(int(z), int(r)) for r, z in coords]


class FaultDetector:
    """v2 detector on the canonical channel matrix X (T, C) in standardised units (see module doc).

    fit(train, calib) -- train (N, T, C) clean windows fit the statistics and the cross-sensor model; calib (N2, T, C)
    separate clean windows set the thresholds (so the false-positive rate is measured out of sample); if omitted the
    training windows are reused and the thresholds are optimistic.
        fp_rate   target false positive probability per channel per residual statistic
        thr_scale multiplies every residual threshold (operating point)
        demean    cross-check only intra-window dynamics (robust to shot-to-shot level shifts; blind to pure offsets)"""

    def __init__(self, spike_sigma=SPIKE_SIGMA, rail_sigma=SATURATION_SIGMA, fp_rate=1e-4, ridge="auto", thr_scale=1.0,
                 window_ms=WINDOW_MS, max_faults=None, absent=None, flux_check=None, demean=False):
        self.spike_sigma, self.rail_sigma, self.fp_rate, self.ridge = spike_sigma, rail_sigma, fp_rate, ridge
        self.thr_scale, self.window_ms, self.max_faults = thr_scale, window_ms, max_faults
        self.absent_given, self.flux_check, self.demean = absent, flux_check, demean
        self.fitted = False

    # ------------------------------------------------------------------ fit
    def fit(self, train, calib=None):
        tr = np.asarray(train, float)
        N, T, C = tr.shape
        rows = tr.reshape(-1, C)
        self.mu = np.nanmean(rows, axis=0)
        sd = np.nanstd(rows, axis=0)
        zero_frac = np.nanmean(rows == 0.0, axis=0)
        self.absent = (zero_frac > 0.5) | ~(sd > 1e-12)               # always zero / constant: not a fault
        if self.absent_given is not None:
            self.absent |= np.asarray(self.absent_given, bool)
        self.sd = np.where(sd > 1e-12, sd, 1.0)
        Z = self._zr(tr)
        S = Z.reshape(-1, C).T @ Z.reshape(-1, C) / (N * T)
        cal = tr if calib is None else np.asarray(calib, float)
        Zc = self._zr(cal)
        # ridge: pick (out of sample, on the calibration windows) the value that predicts channels best
        grid = [self.ridge] if self.ridge != "auto" else [1e-3, 1e-2, 1e-1, 3e-1, 1.0]
        best = None
        for lam in grid:
            self.Sreg = S + lam * np.eye(C)
            P = self._precision(self.absent)
            e2 = np.mean(((Zc @ P) / np.maximum(np.diag(P), 1e-12)) ** 2, axis=(0, 1))   # raw residual variance per channel
            score = float(np.mean(np.log(e2[~self.absent] + 1e-12)))
            if best is None or score < best[0]:
                best = (score, lam, P)
        _, self.ridge_used, self.P_base = best
        self.Sreg = S + self.ridge_used * np.eye(C)
        diag = np.sqrt(np.maximum(np.diag(self.P_base), 1e-12))
        R = (Zc @ self.P_base) / diag
        std = R.reshape(-1, C).std(axis=0)                            # out-of-sample scale (in-sample would be optimistic)
        self.kappa = np.where(std > 1e-12, 1.0 / np.where(std > 1e-12, std, 1.0), 1.0)
        # local-rule statistics from the clean windows
        allw = np.concatenate([tr, cal], axis=0)
        eq = np.isfinite(allw[:, 1:]) & (allw[:, 1:] == allw[:, :-1])
        runs = np.array([_longest_run(e)[0] for e in eq])             # (N, C) longest equal-step run
        self.min_run = np.maximum(runs.max(axis=0) + 3, 5)            # >= 5 equal-step steps = 6 identical samples
        var = np.nanvar(self._z(allw), axis=1)
        self.flat_var = 0.5 * np.nanmin(var, axis=0)           # below every clean window's variance (0 disables it)
        # per-channel spike level: never below the clean extremes seen in training / calibration
        self.spike_thr = np.maximum(self.spike_sigma, 1.25 * np.abs(self._z(allw)).max(axis=(0, 1)))
        # residual thresholds from clean calibration windows
        Rc = (Zc @ self.P_base) / diag * self.kappa
        peak, rms, slope = _stats(Rc)
        keep = ~self.absent
        self.thr = np.array([_pot_threshold(x[:, keep].ravel(), self.fp_rate) for x in (peak, rms, slope)]) * self.thr_scale
        self.fitted = True
        return self

    def _z(self, X):
        Z = (X - self.mu) / self.sd
        return np.where(np.isfinite(Z), Z, 0.0)

    def _zr(self, X):
        """z-scores as seen by the cross-sensor model: optionally with each window's own mean removed, which makes the
        residuals insensitive to shot-to-shot level shifts (only intra-window dynamics are cross-checked)."""
        Z = self._z(X)
        return Z - Z.mean(axis=-2, keepdims=True) if self.demean else Z

    def _precision(self, removed):
        keep = np.flatnonzero(~removed)
        P = np.zeros_like(self.Sreg)
        if keep.size:
            P[np.ix_(keep, keep)] = np.linalg.inv(self.Sreg[np.ix_(keep, keep)])
        return P

    # ------------------------------------------------------------------ detect
    def detect(self, X, psi_pred=None):
        X = np.asarray(X, float)
        T, C = X.shape
        ms = self.window_ms / T
        finite = np.isfinite(X)
        Z = (X - self.mu) / self.sd
        Zc = np.where(finite, Z, 0.0)
        Zr = Zc - Zc.mean(axis=0, keepdims=True) if self.demean else Zc
        absent = self.absent
        kind = np.full(C, "", dtype=object)
        smask = np.zeros((T, C), bool)
        strength = np.zeros(C)

        # -- local rules ---------------------------------------------------
        isz = (X == 0.0) | ~finite
        pair = isz[:-1] & isz[1:]
        zrun = np.zeros((T, C), bool)
        zrun[:-1] |= pair
        zrun[1:] |= pair
        zrun |= ~finite
        drop = zrun.any(axis=0) & ~absent
        eq = finite[1:] & finite[:-1] & (X[1:] == X[:-1])
        rl, re = _longest_run(eq)
        flat_run = (rl >= self.min_run) & ~absent & ~drop
        flat_var = (np.var(Zc, axis=0) < self.flat_var) & finite.all(axis=0) & ~absent & ~drop & ~flat_run
        spk = (np.abs(Zc) > self.spike_thr) & ~absent[None, :]
        spike = spk.any(axis=0) & ~drop & ~flat_run & ~flat_var
        for c in np.flatnonzero(drop):
            kind[c], smask[:, c], strength[c] = "dropout", zrun[:, c], 30.0 + zrun[:, c].sum()
        for c in np.flatnonzero(flat_run):
            s0, e0 = re[c] - rl[c] + 1, re[c] + 1                     # equal-step run covers samples s0..e0
            m = np.zeros(T, bool)
            m[s0 + 1:e0 + 1] = True
            kind[c] = "rail" if abs(Zc[e0, c]) > self.rail_sigma else "flatline"
            smask[:, c], strength[c] = m, 20.0 + rl[c]
        for c in np.flatnonzero(flat_var):
            kind[c], smask[:, c], strength[c] = "flatline", True, 15.0
        for c in np.flatnonzero(spike):
            kind[c], smask[:, c], strength[c] = "spike", spk[:, c], 10.0 + np.abs(Zc[:, c]).max() / self.spike_thr[c]
        local = kind != ""
        order = [int(c) for c in np.argsort(-strength) if local[c]]

        # -- cross-sensor residuals with isolation by peeling -----------------
        removed = absent | local
        P = self.P_base.copy()
        loc_idx = np.flatnonzero(local)
        if loc_idx.size > 8:
            P = self._precision(removed)
        else:
            for c in loc_idx:
                P = _remove(P, c)
        thr = self.thr
        cap = self.max_faults if self.max_faults is not None else max(3, C // 4)
        ratio = np.zeros(C)
        picks = 0
        res_info = {}
        while True:
            d = np.sqrt(np.maximum(np.diag(P), 1e-12))
            R = (Zr @ P) / d * self.kappa
            R[:, removed] = 0.0
            pk, rm, sl = _stats(R)
            ratio = np.max([pk / thr[0], rm / thr[1], sl / thr[2]], axis=0)
            ratio[removed] = -np.inf
            c = int(np.argmax(ratio))
            if not removed.all() and ratio[c] > 1.0 and picks < cap:
                dom = np.argmax([pk[c] / thr[0], rm[c] / thr[1], sl[c] / thr[2]])
                if dom == 0 and rm[c] / thr[1] < 1.0:
                    kind[c] = "spike"
                    smask[:, c] = np.abs(R[:, c]) > thr[0]
                    smask[np.argmax(np.abs(R[:, c])), c] = True
                else:
                    kind[c] = "drift" if dom == 2 else "offset"
                    smask[:, c] = True
                strength[c] = ratio[c]
                order.append(c)
                removed[c] = True
                P = _remove(P, c)
                picks += 1
            else:
                break
        resid_score = float(np.nanmax(np.where(np.isfinite(ratio), ratio, 0.0))) if ratio.size else 0.0

        # -- flux-loop residual (second opinion against the reconstruction) ----
        fc = self.flux_check
        if fc is not None and psi_pred is not None and fc.a is not None:
            cols = fc.channels
            fm = np.nanmean(np.where(finite[:, cols], X[:, cols], np.nan), axis=0)
            z = np.abs(fc.residuals(fm, psi_pred))
            for l, c in enumerate(cols):
                if kind[c] == "" and not absent[c] and np.isfinite(z[l]) and z[l] > fc.thr:
                    kind[c], smask[:, c], strength[c] = "flux_residual", True, 1.0 + z[l] / fc.thr
                    order.append(int(c))

        flagged = kind != ""
        score = np.where(np.isfinite(ratio), np.minimum(ratio, 0.999), 0.0)
        for pos, c in enumerate(dict.fromkeys(order)):
            score[c] = 1e3 - pos
        onset = np.where(flagged, np.argmax(smask, axis=0) * ms, np.nan)
        dur = np.where(flagged, smask.sum(axis=0) * ms, np.nan)
        return Detection(flagged, score, kind, onset, dur, smask, list(dict.fromkeys(order)), resid_score, absent.copy())

    def detect_window(self, window, layout, psi_pred=None):
        return self.detect(layout.flatten(window), psi_pred)


# ====================================================================== v2 router
@dataclass
class Decision:
    route: str
    branch_mask: np.ndarray        # (n_branches,) float32 for the failure-aware model
    channel_actions: dict          # channel -> 'forward-fill' | 'virtual' | 'masked'
    transient: np.ndarray          # (C,) bool, repaired by forward-fill
    virtual: np.ndarray            # (C,) bool, repaired by the virtual sensor
    lost: np.ndarray               # (C,) bool, unavailable to the reconstruction (masked branch or unrepairable)
    use_fallback: bool = False
    log: dict = field(default_factory=dict)


class RouterV2(AdaptiveRouter):
    """Step 2 (v2). Defaults are policy placeholders (not tuned): set them from module 3 / validation results.

        short_ms         faults of dropout / spike type no longer than this are 'short transients' (forward-fill)
        multi_channels   this many simultaneously lost (non-transient) probes is a multi-sensor outage: the virtual
                         sensors' neighbours are failing too, so the masking threshold drops to multi_branch_frac
        branch_frac      a branch is masked for the failure-aware model when this fraction of its probes is lost
        virtual_r2_min   virtual sensor only if its R^2 given the surviving neighbours reaches this
        max_masked_branches  the failure-aware model was trained with at most this many branches masked
                         (structured augmentation: magnetics_all = 5); beyond it, or with no model, use the fallback
        critical_policy  'mask' (tier-1 loss -> failure-aware model, as specified) or 'virtual' (try virtual first)"""

    def __init__(self, tiers=None, layout=None, bank: VirtualSensorBank = None, fallback: LinearFluxFallback = None,
                 short_ms=1.0, multi_channels=6, branch_frac=0.5, multi_branch_frac=0.25, virtual_r2_min=0.9,
                 max_masked_branches=5, critical_policy="mask"):
        super().__init__(DEFAULT_TIERS if tiers is None else tiers)
        self.layout, self.bank, self.fallback = layout, bank, fallback
        self.short_ms, self.multi_channels, self.branch_frac = short_ms, multi_channels, branch_frac
        self.multi_branch_frac = multi_branch_frac
        self.virtual_r2_min, self.max_masked_branches, self.critical_policy = virtual_r2_min, max_masked_branches, critical_policy

    def decide(self, det: Detection, layout=None, have_model=True):
        L = layout or self.layout
        C = L.n_channels
        absent = det.absent if det.absent is not None else np.zeros(L.n_channels, bool)
        bad = det.flagged & ~absent
        kind = det.kind
        transient = bad & ((kind == "spike") | ((kind == "dropout") & (det.duration_ms <= self.short_ms + 1e-9)))
        long = bad & ~transient
        branch_mask = np.ones(L.n_branches, np.float32)
        if not bad.any():
            return Decision("healthy", branch_mask, {}, transient, long & False, long & False)
        tier = np.array([self.tiers.get(int(b), 2) for b in L.branch_of])
        actions = {int(c): "forward-fill" for c in np.flatnonzero(transient)}
        virtual = np.zeros(C, bool)
        lost = np.zeros(C, bool)
        route = "transient_ffill"
        if long.any():
            n_long = int(long.sum())
            whole_branch = any(long[L.channels_of(b)].mean() >= self.branch_frac for b in range(L.n_branches))
            critical = bool((long & (tier == 1)).any()) and self.critical_policy == "mask"
            eligible = self.bank is not None and not (critical or whole_branch or n_long >= self.multi_channels)
            if eligible:
                r2 = np.array([self.bank.r2_given(int(c), bad | absent) for c in np.flatnonzero(long)])
                eligible = bool((r2 >= self.virtual_r2_min).all())
            if eligible:
                virtual = long.copy()
                actions.update({int(c): "virtual" for c in np.flatnonzero(long)})
                route = "virtual_sensor"
            else:
                route = "module2"
                branch_mask = L.branch_mask(long, self.multi_branch_frac if n_long >= self.multi_channels else self.branch_frac)
                for b in range(L.n_branches):                           # a lost tier-1 probe masks its branch
                    if tier[L.channels_of(b)[0]] == 1 and long[L.channels_of(b)].any():
                        branch_mask[b] = 0.0
                masked_ch = np.isin(L.branch_of, np.flatnonzero(branch_mask == 0))
                rest = long & ~masked_ch                                  # lost probes of branches we keep
                if rest.any() and self.bank is not None:
                    ok = np.array([self.bank.r2_given(int(c), bad | absent | masked_ch) >= self.virtual_r2_min
                                   for c in np.flatnonzero(rest)])
                    unfixable = np.flatnonzero(rest)[~ok]
                else:
                    unfixable = np.flatnonzero(rest)
                for c in unfixable:                                       # cannot be repaired: tell the model, mask the branch
                    branch_mask[L.branch_of[c]] = 0.0
                masked_ch = np.isin(L.branch_of, np.flatnonzero(branch_mask == 0))
                lost = masked_ch.copy()
                virtual = long & ~masked_ch
                actions.update({int(c): "virtual" for c in np.flatnonzero(virtual)})
                actions.update({int(c): "masked" for c in np.flatnonzero(masked_ch & bad)})
                if branch_mask.all():
                    route = "virtual_sensor"
                n_masked = int((branch_mask == 0).sum())
                if self.fallback is not None and (not have_model or n_masked > self.max_masked_branches):
                    route = "physics_fallback"
        use_fb = route == "physics_fallback"
        return Decision(route, branch_mask, actions, transient, virtual, lost, use_fb)

    def apply(self, window, det: Detection, dec: Decision, layout=None):
        """Mitigation: returns the repaired window (native shapes)."""
        L = layout or self.layout
        if dec.route == "healthy":
            return window
        X = L.flatten(window)
        fixed = X.copy()
        touched = np.zeros_like(det.sample_mask)
        for c in np.flatnonzero(dec.transient):
            m = det.sample_mask[:, c]
            good = np.flatnonzero(~m & np.isfinite(X[:, c]))
            if good.size == 0:
                continue
            last = X[good[0], c]
            for t in range(X.shape[0]):
                if m[t]:
                    fixed[t, c] = last
                else:
                    last = X[t, c]
            touched[:, c] = m
        if dec.virtual.any():
            fixed = self.bank.repair(fixed, dec.virtual, sample_mask=det.sample_mask, failed=det.flagged | det.absent)
            touched |= det.sample_mask & dec.virtual[None, :]
        out = L.write_back(window, fixed, touched)
        for b in np.flatnonzero(dec.branch_mask == 0):                      # masked branch: zeros, like the model's training
            out[b] = np.zeros_like(out[b])
        return out

    # legacy-compatible entry point: route_and_mitigate(x_window, mask, diagnoses)
    def route_and_mitigate(self, x_window, mask=None, diagnoses=None, oracle_faults=None, layout=None, have_model=True):
        """x_window: branch-list window. diagnoses: a Detection (from FaultDetector.detect_window) -- ignored in oracle
        mode. oracle_faults: FaultLabels (or list of FaultSpec + `layout`) from the injector, which skips detection and
        routes on the true fault labels (the Oracle row = upper bound of detector + router). `mask` is accepted for
        signature compatibility and unused: the health mask is derived from the detection.
        Returns (repaired window, health mask (n_branches,), route label, per-channel action log)."""
        L = layout or self.layout
        det = diagnoses
        if oracle_faults is not None:
            if not isinstance(oracle_faults, FaultLabels):
                raise TypeError("oracle_faults must be the FaultLabels returned by fault_models.inject / inject_window")
            det = detection_from_labels(oracle_faults)
            if det.absent is None or det.absent.size != L.n_channels:
                det.absent = np.zeros(L.n_channels, bool)
        if not isinstance(det, Detection):
            raise TypeError("pass a Detection as `diagnoses` (FaultDetector.detect_window) or give oracle_faults")
        dec = self.decide(det, L, have_model)
        return self.apply(x_window, det, dec, L), dec.branch_mask, dec.route, dec.channel_actions


# ====================================================================== step 3: confidence
def conformal_quantile(scores, alpha):
    s = np.sort(np.asarray(scores, float))
    k = min(s.size, math.ceil((s.size + 1) * (1 - alpha)))
    return float(s[k - 1])


def confidence_flag(clearance_cm, delta_cm, wall_cm=SAFETY_CLEARANCE_CM, high_frac=0.5, abstain_frac=1.0, valid=True):
    """Controller-facing flag, thresholds derived from the wall margin (default 5 cm) and the conformal radius delta.
        ABSTAIN  delta > abstain_frac * wall (interval as wide as the whole margin) or the output is not valid
        HIGH     the alarm decision is unambiguous (|clearance - wall| > delta) and delta <= high_frac * wall
        LOW      otherwise (decision within the uncertainty band, or a wide interval)
    high_frac / abstain_frac are policy parameters, not tuned."""
    if not valid or not np.isfinite(delta_cm) or delta_cm > abstain_frac * wall_cm:
        return "ABSTAIN"
    return "HIGH" if (abs(clearance_cm - wall_cm) > delta_cm and delta_cm <= high_frac * wall_cm) else "LOW"


class ConformalGauge:
    """Step 3. Split-conformal radius delta (cm) per route (Mondrian), from boundary errors on a calibration set.

    Coverage logic: delta = the ceil((n+1)(1-alpha))-th smallest calibration error, which gives
    P(error <= delta) >= 1 - alpha for a new window exchangeable with the n calibration windows of that route.
    Caveats (see MODULE4_V2.md): physics_study replays the same validation windows under 8 damage scenarios and
    neighbouring windows of one shot are correlated, so the calibration set is not i.i.d.; a single delta per route
    is marginal coverage only (too wide for easy windows, too tight for hard ones) -> AdaptiveConformalGauge."""

    def __init__(self, alpha=0.05, wall_threshold_cm=SAFETY_CLEARANCE_CM, high_frac=0.5, abstain_frac=1.0):
        self.alpha, self.wall = alpha, wall_threshold_cm
        self.high_frac, self.abstain_frac = high_frac, abstain_frac
        self.delta = {}

    def calibrate(self, errors_by_route, min_samples=20):
        """errors_by_route: {route: [boundary error in cm per calibration window]}. Routes with fewer than
        `min_samples` windows fall back to the largest delta of the routes that have enough."""
        for route, errs in errors_by_route.items():
            errs = np.asarray(errs, dtype=np.float64)
            if errs.size >= min_samples:
                self.delta[route] = conformal_quantile(errs, self.alpha)
        fallback = max(self.delta.values()) if self.delta else float("inf")
        for route in errors_by_route:
            self.delta.setdefault(route, fallback)
        return self.delta

    def delta_for(self, route, difficulty=None):
        return self.delta.get(route, max(self.delta.values()) if self.delta else float("inf"))

    def assess_clearance(self, clearance, route, difficulty=None, valid=True):
        delta = self.delta_for(route, difficulty)
        conservative = max(0.0, clearance - delta)
        return dict(clearance_cm=clearance, delta_cm=delta, conservative_clearance_cm=conservative,
                    alarm=bool(conservative < self.wall), point_alarm=bool(clearance < self.wall),
                    confidence=confidence_flag(clearance, delta, self.wall, self.high_frac, self.abstain_frac, valid))

    def assess(self, psi_pred, route, difficulty=None):
        if compute_wall_proximity is None:
            raise ImportError("physics_evaluation is required to compute wall clearance from psi")
        return self.assess_clearance(compute_wall_proximity(extract_lcfs_contour(psi_pred)), route, difficulty)


class AdaptiveConformalGauge(ConformalGauge):
    """Difficulty-normalised (locally adaptive) split conformal.

    A linear difficulty model sigma_hat(f) = max(floor, [1, f_std] . beta) is fitted on one half of the calibration
    errors; nonconformity score = error / sigma_hat on the other half; delta(f) = q * sigma_hat(f), q = the conformal
    quantile of the scores. Marginal coverage stays 1 - alpha but delta shrinks on easy windows and grows on hard ones.
    f = difficulty features, e.g. `difficulty_features(detection, decision)` (largest residual ratio, fraction of lost
    channels, masked branches, fallback)."""

    def calibrate_adaptive(self, errors, feats, fit_frac=0.5, seed=0, floor_frac=0.1, ridge=1e-3):
        e = np.asarray(errors, float)
        F = np.asarray(feats, float).reshape(len(e), -1)
        idx = np.random.default_rng(seed).permutation(len(e))
        nfit = int(round(fit_frac * len(e)))
        a, b = idx[:nfit], idx[nfit:]
        self.f_mu, self.f_sd = F[a].mean(0), np.where(F[a].std(0) > 1e-9, F[a].std(0), 1.0)
        D = np.hstack([np.ones((len(a), 1)), (F[a] - self.f_mu) / self.f_sd])
        self.beta = np.linalg.solve(D.T @ D + ridge * np.eye(D.shape[1]), D.T @ e[a])
        self.floor = floor_frac * float(e[a].mean())
        self.q = conformal_quantile(e[b] / self._sigma(F[b]), self.alpha)
        self.global_delta = conformal_quantile(e[b], self.alpha)
        return self.q

    def _sigma(self, F):
        F = np.atleast_2d(np.asarray(F, float))
        D = np.hstack([np.ones((len(F), 1)), (F - self.f_mu) / self.f_sd])
        return np.maximum(D @ self.beta, self.floor)

    def delta_for(self, route, difficulty=None):
        if difficulty is None or not hasattr(self, "q"):
            return super().delta_for(route)
        return float(self.q * self._sigma(difficulty)[0])


def empirical_coverage(deltas, errors):
    """Fraction of held-out windows whose boundary error is inside the reported radius."""
    return float(np.mean(np.asarray(errors, float) <= np.asarray(deltas, float)))


def difficulty_features(det: Detection, dec: Decision):
    """Per-window features for AdaptiveConformalGauge: [largest residual ratio, lost-channel fraction, masked branches,
    route severity index]."""
    return np.array([min(det.resid_score, 20.0), float(dec.lost.mean()), float((dec.branch_mask == 0).sum()),
                     float(ROUTES_V2.index(dec.route))])


def summarise_latencies(rows, keys=("detector", "router", "mitigation", "model", "gauge", "total")):
    """rows: list of the `latencies_ms` dicts returned by process(); returns {stage: {mean, p50, p95, p99}} in ms."""
    out = {}
    for k in keys:
        v = np.array([r[k] for r in rows if k in r], float)
        if v.size:
            out[k] = dict(mean=float(v.mean()), p50=float(np.percentile(v, 50)), p95=float(np.percentile(v, 95)),
                          p99=float(np.percentile(v, 99)), n=int(v.size))
    return out


# ====================================================================== the brain
class SmartAdaptiveMitigationBrain:
    """Legacy mode (no v2 component): detector -> router -> model -> gauge, exactly as before.
    v2 mode (detector=FaultDetector): detect -> decide -> mitigate -> reconstruct (model or physics fallback) -> gauge.

    known_missing: legacy list of (P,) bool arrays per branch (v2 builds the equivalent channel mask itself).
    oracle=True (or process(..., oracle_faults=labels)) skips detection and uses the true labels."""

    def __init__(self, tiers=None, gauge=None, known_missing=None, detector=None, bank=None, fallback=None,
                 router_kw=None, oracle=False):
        self.gauge = gauge or ConformalGauge()
        self.v2 = detector is not None or oracle
        self.oracle, self.fault_detector, self.known_missing = oracle, detector, known_missing
        if self.v2:
            self.router = RouterV2(tiers, None, bank, fallback, **(router_kw or {}))
            self.fallback = fallback
            self.layout = None
        else:
            self.detector, self.router = RealTimeFaultDetector(known_missing), AdaptiveRouter(tiers)

    def process(self, window, model_fn, oracle_faults=None):
        """window -> dict(route, mask, psi, assessment, actions, latencies_ms). model_fn(window, mask) -> (65, 65)."""
        if not self.v2:
            return self._process_legacy(window, model_fn)
        pc = time.perf_counter
        if self.layout is None:
            self.layout = self.router.layout = WindowLayout(window)
        L = self.layout
        t0 = pc()
        if oracle_faults is not None or self.oracle:
            if oracle_faults is None:
                raise ValueError("oracle brain needs oracle_faults=FaultLabels for every window")
            det = detection_from_labels(oracle_faults)
        else:
            det = self.fault_detector.detect_window(window, L)
        t1 = pc()
        dec = self.router.decide(det, L, have_model=model_fn is not None)
        t2 = pc()
        repaired = self.router.apply(window, det, dec, L)
        t3 = pc()
        if dec.use_fallback or model_fn is None:
            Xr = L.flatten(repaired)
            if self.fallback is None:
                raise RuntimeError("no model_fn and no physics fallback: nothing can reconstruct psi")
            avail = (~det.flagged | dec.transient | dec.virtual) & ~dec.lost & ~det.absent
            psi = self.fallback.predict(self.fallback.features(Xr), avail)
            dec.route = "physics_fallback"
        else:
            psi = model_fn(repaired, dec.branch_mask)
        t4 = pc()
        assessment = self.gauge.assess(psi, dec.route, difficulty_features(det, dec))
        t5 = pc()
        return dict(route=dec.route, mask=dec.branch_mask, psi=psi, assessment=assessment, actions=dec.channel_actions,
                    detection=det, decision=dec,
                    latencies_ms=dict(detector=(t1 - t0) * 1e3, router=(t2 - t1) * 1e3, mitigation=(t3 - t2) * 1e3,
                                      model=(t4 - t3) * 1e3, gauge=(t5 - t4) * 1e3, total=(t5 - t0) * 1e3))

    def _process_legacy(self, window, model_fn):
        t0 = time.perf_counter()
        report = self.detector.inspect(window)
        t1 = time.perf_counter()
        repaired, mask, route, actions = self.router.route(window, report)
        t2 = time.perf_counter()
        psi = model_fn(repaired, mask)
        t3 = time.perf_counter()
        assessment = self.gauge.assess(psi, route)
        t4 = time.perf_counter()
        return dict(route=route, mask=mask, psi=psi, assessment=assessment, actions=actions,
                    latencies_ms=dict(detector=(t1 - t0) * 1e3, router=(t2 - t1) * 1e3, model=(t3 - t2) * 1e3,
                                      gauge=(t4 - t3) * 1e3, total=(t4 - t0) * 1e3))


if __name__ == "__main__":
    # legacy self-check on a synthetic window: a spike, a dead redundant probe, and a dead critical branch
    rng = np.random.default_rng(0)
    win = [rng.normal(size=(25, 15)), rng.normal(size=(25, 8)), rng.normal(size=(20,))]
    win[0][10, 3] = 30.0       # impulse on branch 0 -> forward-fill
    win[1][:, 4] = 0.0         # dead probe on branch 1 (tier 3 below) -> neighbour interpolation
    det = RealTimeFaultDetector().inspect(win)
    assert det[0]["diagnoses"] == {3: "IMPULSE_SPIKE"} and det[1]["diagnoses"] == {4: "STUCK_ZERO"}
    fixed, mask, route, _ = AdaptiveRouter({0: 2, 1: 3, 2: 1}).route(win, det)
    assert route == "redundant_interp" and mask.all() and abs(fixed[0][10, 3]) < 30
    assert np.allclose(fixed[1][:, 4], 0.5 * (win[1][:, 3] + win[1][:, 5]))
    win[2][:] = 0.0            # tier-1 branch dead -> module 2 with that branch masked
    fixed, mask, route, _ = AdaptiveRouter({0: 2, 1: 3, 2: 1}).route(win, RealTimeFaultDetector().inspect(win))
    assert route == "module2" and mask.tolist() == [1, 1, 0]
    known = [np.zeros(15, bool), np.zeros(8, bool), np.zeros(1, bool)]
    known[1][4] = True  # a probe that is always absent is not a fault
    assert RealTimeFaultDetector(known).inspect(win)[1]["diagnoses"] == {}
    g = ConformalGauge(alpha=0.1)
    g.calibrate({"healthy": list(np.linspace(0.1, 1.0, 99))})
    assert abs(g.delta["healthy"] - 0.91) < 0.02  # 90% split-conformal quantile of 99 evenly spaced errors
    print("adaptive_mitigation self-check ok (full v2 tests: python test_mitigation_v2.py)")


In [ ]:
%%writefile /kaggle/working/sensor_criticality.py
#!/usr/bin/env python
"""Module 3 (v2): physics-grounded sensor criticality on a REAL trained model, built on sensor_attribution.py.

value_fn(mask) = mean EFIT-based boundary error (cm) of the frozen network over a fixed window set, with the sensors of ~mask zeroed
(zero = train mean in the standardised input = the pipeline's zero fill). For the failure-aware network the per-branch health input
is set to 0 ONLY when a whole branch is gone (that is all the model's mask can express); single dead sensors reach it as zeros.
Analysis windows come from the VALIDATION shots, the chosen sets are confirmed on disjoint TEST shots.

    run_module3(ctx, out_dir, ...)   -> criticality.json, criticality_report.md, figures/*.png
Legacy tier helpers (TIER1_CM, TIER2_CM, tier_of) are kept for old callers; the v2 tier rule is physics_study.tiers_from_criticality.
"""
import json
import math
import os
import pickle
import sys
import time

import numpy as np

HERE = os.path.dirname(os.path.abspath(__file__))
if HERE not in sys.path:
    sys.path.insert(0, HERE)

import sensor_attribution as SA  # noqa: E402

TIER1_CM, TIER2_CM = 2.0, 1.0


def tier_of(boundary_error_cm):
    """Legacy absolute tier rule (kept for old callers)."""
    return 1 if boundary_error_cm > TIER1_CM else 2 if boundary_error_cm >= TIER2_CM else 3


def _t():
    return time.strftime("%H:%M:%S")


# ======================================================================================================== value function
class ValueFn:
    """mask (bool 115, True = available) -> dict of scalar summaries on a fixed window set; cached by mask and persisted."""

    def __init__(self, ctx, ws, truths, inv, cache_path=None):
        import physics_study as PS
        self.PS, self.ctx, self.ws, self.truths, self.inv = PS, ctx, ws, truths, inv
        self.yt = PS.yt_wb(ctx, ws)
        self.cache_path = cache_path
        self.cache = pickle.load(open(cache_path, "rb")) if cache_path and os.path.exists(cache_path) else {}
        self.n_new, self.seconds, self._last_save = 0, 0.0, 0

    @staticmethod
    def key(mask):
        m = np.asarray(mask, bool)
        return np.packbits(m).tobytes() + bytes([len(m) % 8])

    def _forward(self, mask):
        x = self.inv.apply_mask_to_inputs(self.ws.x, mask)
        h = np.tile(self.inv.branch_health(mask), (len(self.ws), 1))
        return self.ctx.net.predict(x, h)

    def metrics(self, mask):
        k = self.key(mask)
        if k in self.cache:
            return self.cache[k]
        PS = self.PS
        t0 = time.time()
        yp = self._forward(mask)
        a = PS.rec_arrays(self.ctx.scorer.score(self.yt, yp, self.truths))
        a["rmse"] = PS.window_rmse(self.yt, yp)
        xp = a["xpoint_err_cm"]
        d = dict(boundary=float(np.nanmean(a["boundary_mean_cm"])), hausdorff=float(np.nanmean(a["boundary_hausdorff_cm"])),
                 boundary_median=float(np.nanmedian(a["boundary_mean_cm"])), axis=float(np.nanmean(a["axis_shift_cm"])),
                 xpoint=float(np.nanmean(xp)) if np.isfinite(xp).any() else float("nan"), xpoint_n=int(np.isfinite(xp).sum()),
                 fail_rate=float(np.nanmean(a["boundary_failed"])), nrmse=PS.nrmse_official(self.ws.sid, a["rmse"], self.ctx.sd))
        self.cache[k] = d
        self.n_new += 1
        self.seconds += time.time() - t0
        if self.cache_path and self.n_new - self._last_save >= 150:
            self.save()
        return d

    def save(self):
        if self.cache_path:
            tmp = self.cache_path + ".tmp"
            pickle.dump(self.cache, open(tmp, "wb"))
            os.replace(tmp, self.cache_path)
            self._last_save = self.n_new

    def view(self, name="boundary"):
        return lambda mask: self.metrics(mask)[name]

    def pred(self, mask):
        return self._forward(mask)


def _game(fn, inv, kind):
    ev = SA.Evaluator(fn)
    return ev, (SA.Game.groups(ev, inv) if kind == "groups" else SA.Game.sensors(ev, inv))


# ======================================================================================================== searches
def lazy_greedy_backward(game, d1_delta):
    """Greedy backward elimination with lazy (CELF-style) re-evaluation: stale increments are treated as lower bounds on the true
    increment, which is NOT guaranteed (damage can be super- or sub-additive), so this is an approximate greedy. Every value in the
    returned curve is a real evaluation of the failed set listed. Curve covers all levels down to 0 sensors."""
    p = game.p
    failed, cur = [], game.value(np.zeros(p, bool))
    inc = {i: float(d1_delta[i]) for i in range(p)}
    curve = [dict(n_available=p, value=cur, failed=[])]
    while len(failed) < p:
        fresh = {}
        while True:
            pool = [i for i in range(p) if i not in failed and i not in fresh]
            cand = min(pool, key=lambda i: inc[i]) if pool else None
            best = min(fresh, key=fresh.get) if fresh else None
            if cand is None or (best is not None and fresh[best] <= inc[cand]):
                break
            f = np.zeros(p, bool)
            f[failed + [cand]] = True
            inc[cand] = fresh[cand] = game.value(f) - cur
        b = best
        failed = failed + [b]
        cur = cur + fresh[b]
        curve.append(dict(n_available=p - len(failed), value=cur, failed=sorted(failed)))
    return curve


def min_set_at(curve, tol):
    """Smallest n_available whose curve value is <= tol, and that set's failed list (None if the tolerance is never met)."""
    feas = [c for c in curve if c["value"] <= tol]
    if not feas:
        return None
    b = min(feas, key=lambda c: c["n_available"])
    return dict(n_available=b["n_available"], value=b["value"], failed=b["failed"])


def truthful_curve(game, curve):
    """Re-evaluate the failed set of every curve point (cached, so only Shapley/ranked paths cost anything)."""
    out = []
    for c in curve:
        f = np.zeros(game.p, bool)
        f[c["failed"]] = True
        out.append(dict(n_available=c["n_available"], value=game.value(f), failed=c["failed"]))
    return out


def ranked_curve(game, order_least_first, tol_unused=None):
    """Fail players in the given order WITHOUT a tolerance cut: the whole error-vs-size curve of an importance ranking."""
    failed, curve = [], []
    p = game.p
    curve.append(dict(n_available=p, value=game.value(np.zeros(p, bool)), failed=[]))
    for i in order_least_first:
        failed.append(int(i))
        f = np.zeros(p, bool)
        f[failed] = True
        curve.append(dict(n_available=p - len(failed), value=game.value(f), failed=sorted(failed)))
    return curve


# ======================================================================================================== spatial maps
def limiter_region(geo, R_axis, Z_axis):
    from matplotlib.path import Path
    lr, lz = geo.limiter
    RR, ZZ = np.meshgrid(R_axis, Z_axis)
    return Path(np.c_[lr, lz]).contains_points(np.c_[RR.ravel(), ZZ.ravel()]).reshape(RR.shape)


def map_vs_group(err_map, geo, idx, region, R_axis, Z_axis):
    """Spearman(error, distance to the NEAREST sensor of the failed set) and distance error-centroid -> sensor-centroid (m)."""
    from scipy import stats
    pts = [(geo.R[i], geo.Z[i]) for i in idx if np.isfinite(geo.R[i]) and np.isfinite(geo.Z[i])]
    if not pts:
        return dict(spearman_distance=float("nan"), centroid_distance_m=float("nan"), n_positioned=0)
    RR, ZZ = np.meshgrid(R_axis, Z_axis)
    d = np.min([np.hypot(RR - r, ZZ - z) for r, z in pts], axis=0)
    ok = np.isfinite(err_map) & region
    w = err_map[ok]
    cx, cz = (w * RR[ok]).sum() / w.sum(), (w * ZZ[ok]).sum() / w.sum()
    sr, sz = np.mean([p[0] for p in pts]), np.mean([p[1] for p in pts])
    return dict(spearman_distance=float(stats.spearmanr(err_map[ok], d[ok])[0]), centroid_distance_m=float(math.hypot(cx - sr, cz - sz)),
                n_positioned=len(pts))


# ======================================================================================================== driver
def run_module3(ctx, out, n_eval=120, tol_abs=2.0, sensor_budget=3000, max_failed=20, worst_k=8, beam=3, seed=0, pairs=1, **kw):
    import greens_sensitivity as GS
    import physics_study as PS
    import sensor_geometry as SG
    n_eval, sensor_budget, max_failed, worst_k, beam, seed, pairs = (int(v) for v in (n_eval, sensor_budget, max_failed, worst_k, beam, seed, pairs))
    tol_abs = float(tol_abs)
    os.makedirs(out, exist_ok=True)
    figdir = os.path.join(out, "figures")
    os.makedirs(figdir, exist_ok=True)
    log = lambda *a: print(_t(), *a, flush=True)  # noqa: E731
    t_start = time.time()
    inv = SA.default_inventory()
    wsA, trA, _ = PS.pick_set(ctx, "val", n_eval, seed)
    wsB, trB, _ = PS.pick_set(ctx, "test", n_eval, seed + 1)
    log(f"analysis windows: val {len(wsA)} ({len(np.unique(wsA.sid))} shots) | confirmation: test {len(wsB)} ({len(np.unique(wsB.sid))} shots)")
    cdir = ctx.cache_dir
    vfA = ValueFn(ctx, wsA, trA, inv, os.path.join(cdir, f"m3_val_{n_eval}_{seed}.pkl"))
    vfB = ValueFn(ctx, wsB, trB, inv, os.path.join(cdir, f"m3_test_{n_eval}_{seed + 1}.pkl"))
    allm = np.ones(inv.n_sensors, bool)
    t0 = time.time()
    clean = vfA.metrics(allm)
    clean_B = vfB.metrics(allm)
    log(f"clean boundary (val) {clean['boundary']:.2f} cm, hausdorff {clean['hausdorff']:.2f}, xpoint {clean['xpoint']:.2f} ({clean['xpoint_n']} windows); "
        f"test {clean_B['boundary']:.2f} cm; first eval {time.time() - t0:.1f}s")
    t0 = time.time()
    vfA.metrics(~np.isin(inv.group_of, [0]))
    t_eval = time.time() - t0
    log(f"one masked evaluation = {t_eval:.2f}s on {len(wsA)} windows")

    # tolerance policy ------------------------------------------------------------------------------------------------
    floor = None
    tols = {"abs_2cm": tol_abs, "clean_x1.25": 1.25 * clean["boundary"], "clean_x1.5": 1.5 * clean["boundary"], "clean_plus_1cm": clean["boundary"] + 1.0}
    primary = "clean_x1.25" if clean["boundary"] >= 0.8 * tol_abs else "abs_2cm"
    tol = tols[primary]
    log("tolerances", {k: round(v, 2) for k, v in tols.items()}, "-> primary", primary, round(tol, 2))

    res = dict(model=ctx.tag, n_eval_windows=len(wsA), n_eval_shots=int(len(np.unique(wsA.sid))), n_confirm_windows=len(wsB),
               n_confirm_shots=int(len(np.unique(wsB.sid))), clean_val=clean, clean_test=clean_B, tolerances=tols, primary_tolerance=primary,
               tolerance_note=("The model's own clean boundary error already exceeds the 2 cm design tolerance, so 'abs_2cm' is infeasible even with "
                               "every sensor; the headline set uses a tolerance RELATIVE to the clean error (primary_tolerance)."
                               if clean["boundary"] >= tol_abs else "clean error is below the absolute tolerance"),
               value="mean over windows of boundary_mean_cm (EFIT-mode, 50 cm when no closed contour); X-point = mean xpoint_err_cm over windows where defined",
               failure_aware_note="health input = 0 only for fully lost branches; single dead sensors enter as zeros (unmasked)",
               device=PS.CPU_NOTE % (ctx.torch_version, ctx.torch_threads), seconds_per_eval=t_eval)

    # 1. groups (exact, 256 coalitions) on val, and again on test -----------------------------------------------------
    evG, gg = _game(vfA.view("boundary"), inv, "groups")
    vals = SA.full_table(gg)
    phi_g = SA.exact_shapley(vals, inv.n_groups)
    I_g = SA.exact_interaction(vals, inv.n_groups)
    d1g = SA.drop_one(gg)
    alone_g = SA.alone_value(gg)
    pop = np.array([bin(b).count("1") for b in range(256)])
    xp_vals = np.array([vfA.metrics(inv_mask(inv, b))["xpoint"] for b in range(256)])
    xp_phi = SA.exact_shapley(np.nan_to_num(xp_vals, nan=np.nanmax(xp_vals)), inv.n_groups)
    evGB, ggB = _game(vfB.view("boundary"), inv, "groups")
    valsB = SA.full_table(ggB)
    phi_gB = SA.exact_shapley(valsB, inv.n_groups)
    from scipy import stats
    res["group"] = dict(names=inv.group_names, counts=inv.counts, shapley=phi_g, shapley_test=phi_gB, shapley_xpoint=xp_phi,
                        drop_one=d1g["delta"], drop_one_test=SA.drop_one(ggB)["delta"], alone=alone_g, interaction=I_g,
                        ranking_shapley=SA.ranking(phi_g, inv.group_names), table_boundary_val=vals, table_boundary_test=valsB,
                        table_xpoint_val=xp_vals, val_test_shapley_spearman=float(stats.spearmanr(phi_g, phi_gB)[0]),
                        efficiency_check=dict(sum_shapley=float(phi_g.sum()), f_all_minus_f_none=float(vals[-1] - vals[0])),
                        worst_k_exhaustive=SA.exhaustive_group_curve(vals, inv.n_groups), worst_k_greedy=SA.worst_k_beam(gg, 8, 1),
                        worst_k_beam=SA.worst_k_beam(gg, 8, beam))
    log("groups done; Shapley (cm):", {n: round(float(v), 2) for n, v in zip(inv.group_names, phi_g)}, f"| {vfA.n_new + vfB.n_new} evals, {vfA.seconds + vfB.seconds:.0f}s")
    vfA.save(); vfB.save()

    # 2. sensors: drop-one, sampled Shapley with CI -------------------------------------------------------------------
    evS, gs = _game(vfA.view("boundary"), inv, "sensors")
    d1 = SA.drop_one(gs)
    xp_d1 = np.array([vfA.metrics(inv.mask_from_failed_sensors([i]))["xpoint"] for i in range(inv.n_sensors)]) - clean["xpoint"]
    log(f"sensor drop-one done ({vfA.n_new} new evals so far); max delta {np.nanmax(d1['delta']):.2f} cm")
    rng = np.random.default_rng(seed)
    sh = SA.sampled_shapley(gs, rng, max_pairs=10 ** 6, budget_evals=evS.n_evals + sensor_budget, max_failed=max_failed)
    vfA.save()
    headroom = max(tol - clean["boundary"], 0.25)
    cls = SA.classify_sensors(d1["delta"], sh["phi"], 0.5 * headroom, 0.05 * headroom)
    z = 1.96 * sh["se"]
    res["sensor"] = dict(names=inv.sensor_names, group_of=inv.group_of, drop_one=d1["delta"], drop_one_xpoint=xp_d1, shapley=sh["phi"],
                         shapley_se=sh["se"], shapley_ci95=z, shapley_pairs=sh["n_pairs"], shapley_max_failed=sh["max_failed"],
                         shapley_stop=sh["stop"], classes=cls, class_thresholds=dict(critical=0.5 * headroom, redundant=0.05 * headroom),
                         ranking_drop_one=SA.ranking(d1["delta"], inv.sensor_names), ranking_shapley=SA.ranking(sh["phi"], inv.sensor_names),
                         shapley_group_sum=[float(sh["phi"][inv.indices(g)].sum()) for g in range(inv.n_groups)],
                         n_distinguishable_top=int(np.sum(np.sort(np.abs(sh['phi']))[::-1][:10] > z[np.argsort(-np.abs(sh['phi']))[:10]])))
    log(f"sensor Shapley: {sh['n_pairs']} antithetic pairs, K={sh['max_failed']}, stop={sh['stop']}, {evS.n_evals} unique evals; classes",
        {c: cls.count(c) for c in set(cls)})

    # interactions among the top sensors (sampled)
    if pairs:
        top = list(np.argsort(-np.abs(sh["phi"]))[:6])
        inter = []
        for a_ in range(len(top)):
            for b_ in range(a_ + 1, len(top)):
                m, se = SA.sampled_interaction(gs, int(top[a_]), int(top[b_]), rng, n_samples=8, max_failed=max_failed)
                inter.append(dict(i=int(top[a_]), j=int(top[b_]), names=[inv.sensor_names[top[a_]], inv.sensor_names[top[b_]]], mean=m, se=se,
                                  kind="backup/redundant" if m - 1.96 * se > 0 else "overlap" if m + 1.96 * se < 0 else "not distinguishable"))
        res["sensor"]["interactions_top6"] = inter
        vfA.save()

    # 3. worst-k failure sets --------------------------------------------------------------------------------------------
    wg = SA.worst_k_beam(gs, worst_k, 1)
    log("sensor worst-k greedy:", {k: round(v["value"], 2) for k, v in wg.items()})
    wb = SA.worst_k_beam(gs, worst_k, beam)
    log(f"sensor worst-k beam{beam}:", {k: round(v["value"], 2) for k, v in wb.items()}, f"| {vfA.n_new} new evals, {vfA.seconds:.0f}s")
    rnd = {}
    for k in range(1, worst_k + 1):   # random-k baseline: 20 random k-sets
        v = gs.many([np.isin(np.arange(inv.n_sensors), rng.choice(inv.n_sensors, k, replace=False)) for _ in range(12)])
        rnd[k] = dict(mean=float(v.mean()), p90=float(np.percentile(v, 90)))
    res["sensor"]["worst_k_greedy"], res["sensor"]["worst_k_beam"], res["sensor"]["random_k"] = wg, wb, rnd
    res["sensor"]["worst_k_beam_width"] = beam
    for w_ in (wg, wb):
        for k, v in w_.items():
            v["names"] = [inv.sensor_names[i] for i in v["set"]]
            v["value_test"] = float(vfB.view("boundary")(inv.mask_from_failed_sensors(v["set"])))
    vfA.save(); vfB.save()

    # 4. minimum sensor set -----------------------------------------------------------------------------------------------
    gcurve = SA.greedy_backward(gg, tol)["curve"]
    lazy = lazy_greedy_backward(gs, d1["delta"])
    ranked_sh = ranked_curve(gs, np.argsort(sh["phi"]))
    ranked_d1 = ranked_curve(gs, np.argsort(np.nan_to_num(d1["delta"])))
    log(f"minimum-set curves done ({vfA.n_new} new evals, {vfA.seconds:.0f}s)")
    ms = dict(groups_greedy=gcurve, groups_exhaustive=res["group"]["worst_k_exhaustive"], sensors_lazy_greedy=lazy,
              sensors_shapley_ranked=ranked_sh, sensors_dropone_ranked=ranked_d1, per_tolerance={})
    for name, tv in tols.items():
        entry = {}
        for cname, curve in (("lazy_greedy", lazy), ("shapley_ranked", ranked_sh), ("dropone_ranked", ranked_d1), ("groups_greedy", gcurve)):
            m = min_set_at(curve, tv)
            if m is not None and cname != "groups_greedy":   # confirm on the disjoint test shots
                m["value_test"] = float(vfB.view("boundary")(inv.mask_from_failed_sensors(m["failed"])))
                m["kept"] = [inv.sensor_names[i] for i in range(inv.n_sensors) if i not in m["failed"]]
                m["kept_per_group"] = {inv.group_names[g]: int(sum(1 for i in inv.indices(g) if i not in m["failed"])) for g in range(inv.n_groups)}
                m["failed"] = [int(i) for i in m["failed"]]
            entry[cname] = m
        ex = SA.exhaustive_group_curve(vals, inv.n_groups)
        feas = [c for c in ex if c["best_value"] <= tv]
        entry["groups_exhaustive_best"] = min(feas, key=lambda c: c["n_available"]) if feas else None
        ms["per_tolerance"][name] = dict(tol=tv, **entry)
    ms["clean_test_reference"] = clean_B["boundary"]
    res["min_set"] = ms
    for name, e in ms["per_tolerance"].items():
        lz = e["lazy_greedy"]
        log(f"min set @ {name} ({e['tol']:.2f} cm): lazy-greedy {lz['n_available'] if lz else None} sensors"
            f"{'' if not lz else f' (val {lz['value']:.2f}, test {lz['value_test']:.2f} cm)'}; shapley-ranked "
            f"{e['shapley_ranked']['n_available'] if e['shapley_ranked'] else None}")
    vfA.save(); vfB.save()

    # 5. Green's-function sensitivity (vacuum linear-Gaussian model) vs the data-driven rankings ---------------------------
    geo = SG.load_geometry()
    gm = GS.build_model(geo)
    gsc = gm.scores()
    evg = SA.Evaluator(gm.value_fn)
    ggg, gsg = SA.Game.groups(evg, inv), SA.Game.sensors(evg, inv)
    gvals = SA.full_table(ggg)
    gphi_g = SA.exact_shapley(gvals, inv.n_groups)
    gsh = SA.sampled_shapley(gsg, np.random.default_rng(seed + 5), max_pairs=10 ** 6, budget_evals=4000, max_failed=max_failed)
    glabels = [inv.group_names[g] for g in inv.group_of]
    res["greens"] = dict(
        model="vacuum Green's functions, linear-Gaussian information model (greens_sensitivity.build_model defaults); saddle loops unmodelled "
              "(voltage sensors), Ip = Rogowski sum; modelling choices (prior, noise, kernel) are parameters, see MODULE3_V2.md",
        has_model=gm.has_model, loo_loss=gsc["loo_loss"], alone_gain=gsc["alone_gain"], noise_prop=gsc["noise_prop"], shapley=gsh["phi"],
        group_shapley=gphi_g, group_drop_one=SA.drop_one(ggg)["delta"],
        agreement=dict(
            dropone_vs_loo=GS.rank_agreement_by_group(d1["delta"], gsc["loo_loss"], glabels, n_perm=2000),
            shapley_vs_loo=GS.rank_agreement_by_group(sh["phi"], gsc["loo_loss"], glabels, n_perm=2000),
            shapley_vs_green_shapley=GS.rank_agreement_by_group(sh["phi"], gsh["phi"], glabels, n_perm=2000),
            group_shapley=GS.rank_agreement(phi_g, gphi_g, n_perm=5000),
            group_dropone=GS.rank_agreement(d1g["delta"], SA.drop_one(ggg)["delta"], n_perm=5000)))
    ag = res["greens"]["agreement"]
    log("Green vs data: sensor Shapley~loo rho", round(ag["shapley_vs_loo"]["all"]["spearman"], 3), "tau", round(ag["shapley_vs_loo"]["all"]["kendall"], 3),
        "| groups rho", round(ag["group_shapley"]["spearman"], 3))
    # Green-guided minimal set evaluated on the REAL model (least informative sensors dropped first)
    order_g = np.argsort(np.nan_to_num(gsc["loo_loss"], nan=-1.0))
    gcurve_real = ranked_curve(gs, order_g)
    res["min_set"]["sensors_green_ranked"] = gcurve_real
    res["min_set"]["per_tolerance_green"] = {name: min_set_at(gcurve_real, tv) for name, tv in tols.items()}
    vfA.save()

    # 6. spatial error maps vs sensor location ------------------------------------------------------------------------------
    R_axis, Z_axis = np.linspace(0.06, 1.98, 65), np.linspace(-2.0, 2.0, 65)
    region = limiter_region(geo, R_axis, Z_axis)
    yt = vfA.yt
    yp0 = vfA.pred(allm)
    base_err = np.abs(yp0 - yt).mean(0)
    cases = []
    gorder = np.argsort(-d1g["delta"])
    for g in gorder[:3]:
        cases.append(("group", f"{inv.group_names[g]} lost", list(inv.indices(g))))
    sorder = [int(i) for i in np.argsort(-np.nan_to_num(d1["delta"]))[:4]]
    for i in sorder:
        cases.append(("sensor", f"{inv.sensor_names[i]} lost", [i]))
    k3 = wb[min(3, worst_k)]["set"]
    cases.append(("worst3", f"worst-3 sensors ({', '.join(inv.sensor_names[i] for i in k3)})", list(k3)))
    maps = []
    for kind, label, idx in cases:
        m = ~np.isin(np.arange(inv.n_sensors), idx)
        yp = vfA.pred(m)
        e_fail = np.abs(yp - yt).mean(0)
        delta = np.abs(yp - yp0).mean(0)
        st = map_vs_group(delta, geo, idx, region, R_axis, Z_axis)
        if len(idx) == 1:
            i = idx[0]
            st.update(GS.error_map_vs_location(delta, geo.R[i], geo.Z[i], R_axis, Z_axis, region, GS.sensor_response_map(geo, i)))
        st_err = map_vs_group(np.maximum(e_fail - base_err, 0), geo, idx, region, R_axis, Z_axis)
        maps.append(dict(kind=kind, label=label, sensors=[int(i) for i in idx], delta_map=delta, err_map=e_fail, stats_delta_vs_clean_prediction=st,
                         stats_excess_error_vs_clean=st_err, boundary_cm=vfA.metrics(m)["boundary"]))
    res["spatial"] = dict(R_axis=R_axis, Z_axis=Z_axis, base_err_map=base_err, cases=maps,
                          note="delta_map = mean |psi_pred(lost) - psi_pred(clean)| over the analysis windows (Wb/rad); stats are Spearman(map, distance "
                               "to the nearest lost sensor) over pixels inside the limiter (negative = effect concentrated near the sensor) and the distance "
                               "between the map's centroid and the sensor(s).")
    res["cost"] = dict(unique_evals_val=len(vfA.cache), unique_evals_test=len(vfB.cache), seconds=time.time() - t_start,
                       evals_new_this_run=vfA.n_new + vfB.n_new)
    vfA.save(); vfB.save()
    _figures(res, inv, geo, figdir, tol, primary, region)
    path = os.path.join(out, "criticality.json")
    np.savez_compressed(os.path.join(out, "spatial_maps.npz"), base_err=base_err,
                        **{f"case{i}_{k}": c.pop(k).astype(np.float32) for i, c in enumerate(res["spatial"]["cases"]) for k in ("delta_map", "err_map")})
    res["spatial"].pop("base_err_map")
    json.dump(SA.jsonable(res), open(path, "w"), indent=1)
    _report(res, inv, os.path.join(out, "criticality_report.md"))
    log("wrote", path)
    return res


def inv_mask(inv, bits):
    """Availability mask of the group coalition whose failed groups are the set bits of `bits`."""
    return ~np.isin(inv.group_of, [g for g in range(inv.n_groups) if (bits >> g) & 1])


# ======================================================================================================== outputs
def _figures(res, inv, geo, figdir, tol, primary, region):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    cm = plt.get_cmap("tab10")
    # groups
    g = res["group"]
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
    x = np.arange(inv.n_groups)
    ax[0].bar(x - 0.2, g["drop_one"], 0.4, label="drop-one")
    ax[0].bar(x + 0.2, g["shapley"], 0.4, label="Shapley (val)")
    ax[0].plot(x + 0.2, g["shapley_test"], "k_", ms=14, label="Shapley (test)")
    ax[0].set_xticks(x); ax[0].set_xticklabels(inv.group_names, rotation=40, ha="right"); ax[0].set_ylabel("boundary error added (cm)")
    ax[0].legend(); ax[0].grid(alpha=.3); ax[0].set_title("branch (group) importance, exact over 256 coalitions")
    im = ax[1].imshow(np.array(g["interaction"]), cmap="coolwarm", vmin=-np.abs(g["interaction"]).max(), vmax=np.abs(g["interaction"]).max())
    ax[1].set_xticks(x); ax[1].set_yticks(x); ax[1].set_xticklabels(inv.group_names, rotation=60, ha="right", fontsize=8); ax[1].set_yticklabels(inv.group_names, fontsize=8)
    ax[1].set_title("pair interaction (+ backup, - overlap), cm"); fig.colorbar(im, ax=ax[1]); fig.tight_layout()
    fig.savefig(os.path.join(figdir, "group_shapley.png"), dpi=140); plt.close(fig)
    # sensor shapley top 25 with CI
    s = res["sensor"]
    order = np.argsort(-np.abs(s["shapley"]))[:25][::-1]
    fig, ax = plt.subplots(figsize=(7, 6.5))
    ax.barh(range(len(order)), np.array(s["shapley"])[order], xerr=np.array(s["shapley_ci95"])[order], color=[cm(int(inv.group_of[i])) for i in order])
    ax.set_yticks(range(len(order))); ax.set_yticklabels([inv.sensor_names[i] for i in order], fontsize=7)
    ax.set_xlabel(f"sampled Shapley damage (cm), K<={s['shapley_max_failed']}, {s['shapley_pairs']} antithetic pairs, 95% CI"); ax.grid(alpha=.3)
    fig.tight_layout(); fig.savefig(os.path.join(figdir, "sensor_shapley_top25.png"), dpi=140); plt.close(fig)
    # worst-k
    fig, ax = plt.subplots(figsize=(6.5, 4))
    ks = list(range(1, len(s["worst_k_greedy"]) + 1))
    ax.plot(ks, [s["worst_k_greedy"][str(k) if str(k) in s["worst_k_greedy"] else k]["value"] for k in ks], "o-", label="sensors, greedy")
    ax.plot(ks, [s["worst_k_beam"][str(k) if str(k) in s["worst_k_beam"] else k]["value"] for k in ks], "s-", label=f"sensors, beam {s['worst_k_beam_width']}")
    ax.plot(ks, [s["random_k"][str(k) if str(k) in s["random_k"] else k]["mean"] for k in ks], "k--", label="random k sensors (mean)")
    gx = res["group"]["worst_k_exhaustive"]
    ax.plot([c["n_failed"] for c in gx[1:9]], [c["worst_value"] for c in gx[1:9]], "^-", label="GROUPS, exhaustive")
    ax.axhline(res["clean_val"]["boundary"], c="gray", ls=":", label="clean")
    ax.set_xlabel("k simultaneous failures"); ax.set_ylabel("mean boundary error (cm)"); ax.set_yscale("log"); ax.legend(fontsize=8); ax.grid(alpha=.3)
    fig.tight_layout(); fig.savefig(os.path.join(figdir, "worst_k.png"), dpi=140); plt.close(fig)
    # min-set curves
    ms = res["min_set"]
    fig, ax = plt.subplots(figsize=(7, 4.2))
    for key, lab in (("sensors_lazy_greedy", "lazy greedy"), ("sensors_shapley_ranked", "Shapley-ranked"), ("sensors_dropone_ranked", "drop-one-ranked"),
                     ("sensors_green_ranked", "Green's-ranked")):
        c = ms[key]
        ax.plot([p["n_available"] for p in c], [p["value"] for p in c], label=lab)
    for name, tv in res["tolerances"].items():
        ax.axhline(tv, ls="--", lw=.8, c="r" if name == primary else "gray")
        ax.text(1, tv, name, fontsize=7, va="bottom")
    ax.set_xlabel("sensors kept (of 115)"); ax.set_ylabel("mean boundary error (cm)"); ax.set_yscale("log"); ax.invert_xaxis(); ax.legend(fontsize=8); ax.grid(alpha=.3)
    fig.tight_layout(); fig.savefig(os.path.join(figdir, "minimal_sensor_set_curve.png"), dpi=140); plt.close(fig)
    # data vs Green
    gr = res["greens"]
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    for a_, (xv, yv, xl) in zip(ax, ((s["drop_one"], gr["loo_loss"], "data drop-one (cm)"), (s["shapley"], gr["loo_loss"], "data Shapley (cm)"))):
        for gi in range(inv.n_groups):
            m = inv.group_of == gi
            a_.scatter(np.array(xv)[m], np.array(yv, float)[m], s=14, color=cm(gi), label=inv.group_names[gi])
        a_.set_xlabel(xl); a_.set_ylabel("Green's leave-one-out loss"); a_.set_yscale("symlog", linthresh=1e-6); a_.set_xscale("symlog", linthresh=1e-2); a_.grid(alpha=.3)
    ax[0].legend(fontsize=6); fig.tight_layout(); fig.savefig(os.path.join(figdir, "data_vs_greens.png"), dpi=140); plt.close(fig)
    # spatial maps with sensor positions
    sp = res["spatial"]
    cases = sp["cases"]
    fig, axes = plt.subplots(2, 4, figsize=(15, 8.2))
    lr, lz = geo.limiter
    for a_, c in zip(axes.ravel(), cases):
        a_.imshow(np.array(c["delta_map"]), origin="lower", extent=[0.06, 1.98, -2, 2], cmap="magma", aspect="auto")
        a_.plot(lr, lz, "w-", lw=.6)
        idx = c["sensors"]
        a_.scatter(geo.R[idx], geo.Z[idx], s=26 if len(idx) < 20 else 10, c="cyan", edgecolors="k", linewidths=.4)
        st = c["stats_delta_vs_clean_prediction"]
        a_.set_title(f"{c['label'][:44]}\nbnd {c['boundary_cm']:.1f} cm, rho(dist) {st['spearman_distance']:.2f}", fontsize=8)
    for a_ in axes.ravel()[len(cases):]:
        a_.axis("off")
    fig.tight_layout(); fig.savefig(os.path.join(figdir, "spatial_error_vs_sensor_location.png"), dpi=130); plt.close(fig)


def _report(res, inv, path):
    g, s, ms = res["group"], res["sensor"], res["min_set"]
    L = ["# Module 3 (v2) on the real model: " + res["model"], "",
         f"Value = mean EFIT-mode boundary error (cm). Analysis: {res['n_eval_windows']} val windows / {res['n_eval_shots']} shots; confirmation: "
         f"{res['n_confirm_windows']} test windows / {res['n_confirm_shots']} shots. Device: {res['device']}.",
         f"Clean boundary error: val {res['clean_val']['boundary']:.2f} cm, test {res['clean_test']['boundary']:.2f} cm. {res['tolerance_note']}", "",
         "## Groups (exact Shapley, 256 coalitions)", "", "| group | drop-one (cm) | Shapley val (cm) | Shapley test (cm) | Shapley X-point (cm) |", "|---|---|---|---|---|"]
    for i in np.argsort(-np.array(g["shapley"])):
        L.append(f"| {inv.group_names[i]} | {g['drop_one'][i]:.2f} | {g['shapley'][i]:.2f} | {g['shapley_test'][i]:.2f} | {g['shapley_xpoint'][i]:.2f} |")
    L += ["", f"Val/test Shapley Spearman: {g['val_test_shapley_spearman']:.2f}.", "", "## Top sensors (sampled Shapley, 95% CI)", "",
          "| sensor | class | drop-one (cm) | Shapley (cm) | +-CI |", "|---|---|---|---|---|"]
    for i in np.argsort(-np.abs(s["shapley"]))[:15]:
        L.append(f"| {inv.sensor_names[i]} | {s['classes'][i]} | {s['drop_one'][i]:.2f} | {s['shapley'][i]:.2f} | {s['shapley_ci95'][i]:.2f} |")
    L += ["", "## Minimum sensor sets (val-selected, test-confirmed)", "", "| tolerance | cm | lazy greedy | Shapley-ranked | drop-one-ranked | groups best |", "|---|---|---|---|---|---|"]
    for name, e in ms["per_tolerance"].items():
        f = lambda m: "infeasible" if m is None else f"{m['n_available']} (val {m['value']:.2f} / test {m.get('value_test', float('nan')):.2f})"  # noqa: E731
        gb = e["groups_exhaustive_best"]
        L.append(f"| {name} | {e['tol']:.2f} | {f(e['lazy_greedy'])} | {f(e['shapley_ranked'])} | {f(e['dropone_ranked'])} | "
                 f"{'infeasible' if gb is None else str(gb['n_available']) + ' groups'} |")
    gr = res["greens"]["agreement"]
    L += ["", "## Green's-function sensitivity vs data-driven ranking", "",
          f"sensors, Shapley vs Green's LOO: Spearman {gr['shapley_vs_loo']['all']['spearman']:.2f} (p {gr['shapley_vs_loo']['all']['p_spearman']:.3f}), "
          f"Kendall {gr['shapley_vs_loo']['all']['kendall']:.2f}; drop-one vs LOO: Spearman {gr['dropone_vs_loo']['all']['spearman']:.2f}; "
          f"groups (n=8): Spearman {gr['group_shapley']['spearman']:.2f}.", "", "Figures: figures/*.png. Full numbers: criticality.json."]
    open(path, "w", encoding="utf8").write("\n".join(L) + "\n")


if __name__ == "__main__":
    print(__doc__)


In [ ]:
%%writefile /kaggle/working/physics_study.py
#!/usr/bin/env python
"""Model-driven studies for Module 3 (sensor criticality) and Module 4 (adaptive mitigation) on a REAL trained model (v2).

Library + the Module 4 study. Module 3 lives in sensor_criticality.py (built on sensor_attribution.py).
Driver: run_module34_local.py (window cache -> stages m3 / m4); see STUDY_V2.md.

Scoring is Module 1 v2: EFIT-truth physics (evaluate_window_physics(..., truth=...), efit_truth.TruthIndex), always in "efit" mode.
Every window keeps its (shot_id, window_index). Windows without an EFIT LCFS (about 13 %) are dropped from the sets up front
(and counted), so every reported boundary error is against the EFIT separatrix.
The wall alarm uses the documented PROXY wall (the EFIT grid box; the data holds no vessel geometry): alarm rates are a protocol
demonstration, not a safety result.
"""
import json
import math
import os
import sys
import time

import numpy as np

HERE = os.path.dirname(os.path.abspath(__file__))
if HERE not in sys.path:
    sys.path.insert(0, HERE)

from physics_evaluation import BOUNDARY_FAIL_CM, SAFETY_CLEARANCE_CM, evaluate_window_physics  # noqa: E402
from fault_models import FaultSpec, WindowLayout, as_tp, group_loss_specs, inject_window, random_specs  # noqa: E402
from adaptive_mitigation import (AdaptiveConformalGauge, ConformalGauge, FaultDetector, RouterV2,  # noqa: E402
                                 SmartAdaptiveMitigationBrain, confidence_flag, detection_from_labels,
                                 difficulty_features, empirical_coverage, summarise_latencies)
from virtual_sensor import LinearFluxFallback, fit_physics_fallback, fit_virtual_sensors  # noqa: E402

REC_KEYS = ("boundary_mean_cm", "boundary_hausdorff_cm", "axis_shift_cm", "xpoint_err_cm", "d_wall_true_cm", "d_wall_pred_cm",
            "is_false_alarm", "is_missed_alarm", "boundary_failed", "xpoint_true_found", "xpoint_pred_found", "kappa_err_field")
SIGNAL_SHORT = ("flux_loop", "ccbv", "obr", "obv", "saddle_v", "coil_i", "solenoid_i", "ip")
ALPHA = 0.1                      # conformal miscoverage (90 % intervals)
CPU_NOTE = "CPU (torch %s, %s threads); a GPU number comes from Kaggle / Modal"


def t():
    return time.strftime("%H:%M:%S")


# ======================================================================================================== windows
class WindowSet:
    """Cached model inputs/targets. x[b]: (N,1,1,T,n) for the six profile branches, (N,1,1,T) for solenoid and ip (standardised
    with the TokaMark train statistics, i.e. zero = the train mean = the pipeline's zero fill). y: (N,65,65) standardised psi."""

    def __init__(self, sid, wid, x, y):
        self.sid = np.asarray(sid, np.int64)
        self.wid = np.asarray(wid, np.int64)
        self.x = [np.asarray(v, np.float32) for v in x]
        self.y = np.asarray(y, np.float32).reshape(len(self.sid), 65, 65)

    def __len__(self):
        return len(self.sid)

    def subset(self, idx):
        idx = np.asarray(idx)
        return WindowSet(self.sid[idx], self.wid[idx], [v[idx] for v in self.x], self.y[idx])

    def window(self, i):
        """List of 8 (T_b, P_b) float64 arrays (the format of fault_models / adaptive_mitigation)."""
        return [as_tp(v[i])[0] for v in self.x]

    def put(self, windows):
        """List of N windows (as returned by window()) -> list of 8 arrays shaped like self.x."""
        return [np.stack([np.asarray(w[b], np.float32).reshape(self.x[b].shape[1:]) for w in windows]) for b in range(8)]

    def save(self, path):
        tmp = path + ".tmp.npz"
        np.savez(tmp, sid=self.sid, wid=self.wid, y=self.y, **{f"x{b}": v for b, v in enumerate(self.x)})
        os.replace(tmp, path)

    @staticmethod
    def load(path):
        z = np.load(path)
        return WindowSet(z["sid"], z["wid"], [z[f"x{b}"] for b in range(8)], z["y"])


class Net:
    """Frozen network on CPU: predict(x_list, health) -> psi in Wb/rad (N,65,65) float64."""

    def __init__(self, model, mu, sd, failure_aware=True, chunk=64):
        self.model, self.mu, self.sd, self.fa, self.chunk = model, mu, sd, failure_aware, chunk
        self.n_calls = 0

    def predict(self, x, health=None):
        import torch
        N = len(x[0])
        out = []
        with torch.no_grad():
            for s in range(0, N, self.chunk):
                xin = [torch.from_numpy(np.ascontiguousarray(v[s:s + self.chunk], dtype=np.float32)) for v in x]
                n = xin[0].shape[0]
                if self.fa:
                    h = torch.ones(n, 8) if health is None else torch.from_numpy(np.asarray(health[s:s + n], np.float32))
                    y = self.model(*xin, h)
                else:
                    y = self.model(*xin)
                if not isinstance(y, (list, tuple)):
                    y = [y]
                out.append(y[0].detach().cpu().numpy().reshape(n, 65, 65))
        self.n_calls += N
        return np.concatenate(out).astype(np.float64) * self.sd + self.mu


# ======================================================================================================== scoring
def _reduce(rec):
    return {k: (bool(rec[k]) if isinstance(rec[k], (bool, np.bool_)) else float(rec[k])) for k in REC_KEYS if k in rec}


def _score_one(args):
    yt, yp, tr = args
    try:
        return _reduce(evaluate_window_physics(yt, yp, truth=tr))
    except Exception as e:  # a failed score is recorded as such, never replaced by a number
        return dict(error=repr(e)[:80])


class Scorer:
    """EFIT-truth physics for batches of windows; optional process pool (workers are stateless: truth is passed along)."""

    def __init__(self, procs=0):
        self.pool = None
        self.procs = procs
        self.n_scored = 0
        self.n_errors = 0
        if procs and procs > 1:
            import multiprocessing as mp
            self.pool = mp.get_context("spawn").Pool(procs)

    def score(self, yt, yp, truths):
        jobs = list(zip(yt, yp, truths))
        recs = self.pool.map(_score_one, jobs, chunksize=max(1, len(jobs) // (4 * self.procs))) if self.pool else [_score_one(j) for j in jobs]
        self.n_scored += len(recs)
        self.n_errors += sum("error" in r for r in recs)
        return recs

    def close(self):
        if self.pool:
            self.pool.close()
            self.pool.join()


def rec_arrays(recs):
    """list of reduced records -> dict of float arrays (bool -> 0/1, missing/failed -> nan)."""
    out = {}
    for k in REC_KEYS:
        out[k] = np.array([float(r.get(k, np.nan)) for r in recs], float)
    return out


def window_rmse(yt, yp):
    return np.sqrt(np.mean((np.asarray(yp) - np.asarray(yt)) ** 2, axis=(1, 2)))


def nrmse_official(shots, rmse, sd):
    """TokaMark: per-window RMSE -> per-shot sqrt(mean(rmse^2)) -> / output std -> mean over shots."""
    shots, rmse = np.asarray(shots), np.asarray(rmse, float)
    u = np.unique(shots)
    return float(np.mean([np.sqrt(np.mean(rmse[shots == s] ** 2)) / sd for s in u]))


def shot_bootstrap(shots, boundary, rmse, sd, n_boot=1000, seed=0, extra=None):
    """Point estimate + 95 % shot-level bootstrap CI of the pooled-window mean boundary error and the official NRMSE.
    `extra` optionally maps name -> per-window array whose pooled mean (ignoring nan) is bootstrapped too."""
    shots, boundary, rmse = np.asarray(shots), np.asarray(boundary, float), np.asarray(rmse, float)
    u = np.unique(shots)
    ok = np.isfinite(boundary)
    sb = np.array([boundary[(shots == s) & ok].sum() for s in u])
    cb = np.array([((shots == s) & ok).sum() for s in u], float)
    sr = np.array([np.sqrt(np.mean(rmse[shots == s] ** 2)) for s in u]) / sd
    ex = {k: (np.array([np.nansum(np.asarray(v, float)[shots == s]) for s in u]),
              np.array([np.isfinite(np.asarray(v, float)[shots == s]).sum() for s in u], float)) for k, v in (extra or {}).items()}
    rng = np.random.default_rng(seed)
    bs, ns, exs = [], [], {k: [] for k in ex}
    for _ in range(n_boot):
        i = rng.integers(0, len(u), len(u))
        bs.append(sb[i].sum() / max(cb[i].sum(), 1.0))
        ns.append(sr[i].mean())
        for k, (a, c) in ex.items():
            exs[k].append(a[i].sum() / max(c[i].sum(), 1.0))
    q = lambda v: [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))]  # noqa: E731
    out = dict(boundary_cm=float(sb.sum() / max(cb.sum(), 1.0)), boundary_ci95=q(bs), nrmse=float(sr.mean()), nrmse_ci95=q(ns),
               n_windows=int(len(shots)), n_shots=int(len(u)), n_boundary_valid=int(ok.sum()))
    for k, (a, c) in ex.items():
        out[k] = float(a.sum() / max(c.sum(), 1.0))
        out[k + "_ci95"] = q(exs[k])
    return out


# ======================================================================================================== context
class Ctx:
    pass


def make_context(P, run_dir, data_dir, cache_dir, tag, procs="0", **kw):
    """Load the frozen checkpoint and the cached window sets (train / val / test); keep only windows with an EFIT LCFS."""
    import torch
    from efit_truth import TruthIndex
    c = Ctx()
    c.tag, c.data_dir, c.cache_dir = tag, data_dir, cache_dir
    torch.set_num_threads(int(kw.get("threads", min(8, os.cpu_count() or 4))))
    raw = {s: WindowSet.load(os.path.join(cache_dir, f"{s}.npz")) for s in ("train", "val", "test")
           if os.path.exists(os.path.join(cache_dir, f"{s}.npz"))}
    first = next(iter(raw))
    loaders, meta = P.build_data("random", [first if first in ("val", "test", "train") else "test"], shuffle_windows=False)
    m = meta["output"]["equilibrium-psi"]
    c.mu, c.sd = float(m["mean"]), float(m["std"])
    shapes = ([tuple(v.shape[1:]) for v in raw[first].x], [], [(1, 65, 65)])
    model = P.make_model(None, meta, shapes=shapes)
    model.load_state_dict(torch.load(os.path.join(run_dir, "out", "random", "model", "best_model.pt"), map_location="cpu", weights_only=False))
    model.eval()
    c.net = Net(model, c.mu, c.sd, failure_aware=P.CFG["failure_aware"])
    c.n_params = int(sum(p.numel() for p in model.parameters()))
    c.truth = TruthIndex(data_dir, outlier_yaml=P.split_files("random")[2])
    c.sets, c.truths, c.dropped = {}, {}, {}
    for name, ws in raw.items():
        keep, trs = [], []
        for i in range(len(ws)):
            tr = c.truth.get(ws.sid[i], ws.wid[i])
            if tr is not None and tr.get("lcfs_rz_m") is not None:
                keep.append(i)
                trs.append(tr)
        c.dropped[name] = dict(n_cached=len(ws), n_efit_valid=len(keep), n_dropped_no_efit_lcfs=len(ws) - len(keep))
        c.sets[name] = ws.subset(keep)
        c.truths[name] = trs
    c.scorer = Scorer(int(procs))
    c.torch_threads = torch.get_num_threads()
    c.torch_version = torch.__version__
    return c


def yt_wb(ctx, ws):
    return ws.y.astype(np.float64) * ctx.sd + ctx.mu


def score_pred(ctx, ws, truths, yp):
    """Physics records + official-NRMSE ingredients for predictions yp (N,65,65) in Wb against the window set's targets."""
    yt = yt_wb(ctx, ws)
    recs = ctx.scorer.score(yt, yp, truths)
    a = rec_arrays(recs)
    a["rmse"] = window_rmse(yt, yp)
    return a


# ======================================================================================================== Module 3
def run_criticality(ctx, out, **kw):
    import sensor_criticality as SC
    return SC.run_module3(ctx, out, **kw)


# ======================================================================================================== Module 4
SCENARIOS = ("clean", "channel_loss", "channel_loss_6", "group_loss", "temporal_gap", "pre_disruption", "sensor_loss",
             "stuck", "drift", "spike")
TABLE_COLUMNS = {"Clean": "clean", "Channel loss": "channel_loss", "Group loss": "group_loss", "Temporal gap": "temporal_gap",
                 "Pre-disruption": "pre_disruption"}
FAULT_SCENARIOS = tuple(s for s in SCENARIOS if s != "clean")
MAGNETICS_ALL = (0, 1, 2, 3, 4)   # the harness group_failure "magnetics_all"


def make_faults(scen, L, rng, usable, channel_k=3):
    """Fault specs of one scenario for one window. Definitions follow the robustness harness (pipeline_robustness.py):
    channel_loss      channel_k whole input signals (branches) zeroed for the whole window  (Gupta channel_ablation)
    group_loss        magnetics_all = flux loops, CCBV, OBR, OBV, saddle loops zeroed       (group_failure)
    temporal_gap      a contiguous 40 % of the window zeroed on every probe, random position (temporal_gap 0.4)
    pre_disruption    the LAST 25 % of the window zeroed on every probe                      (disruption_proximate 0.25)
    sensor_loss       6 random individual probes zeroed for the whole window                  (sensor level, extra)
    stuck / drift / spike  3 random probes, random onset (fault_models); stuck is half rails, half frozen values."""
    ms = 5.0
    if scen == "clean":
        return []
    if scen.startswith("channel_loss"):
        k = 6 if scen.endswith("_6") else channel_k
        return group_loss_specs(L, sorted(rng.choice(8, size=k, replace=False).tolist()))
    if scen == "group_loss":
        return group_loss_specs(L, MAGNETICS_ALL)
    if scen in ("temporal_gap", "pre_disruption"):
        if scen == "temporal_gap":
            on = float(rng.uniform(0, ms * 0.6))
            return [FaultSpec(int(c), "dropout", on, ms * 0.4) for c in range(L.n_channels)]
        return [FaultSpec(int(c), "dropout", ms * 0.75, None) for c in range(L.n_channels)]
    if scen == "sensor_loss":
        return [FaultSpec(int(c), "dropout", 0.0, None) for c in rng.choice(usable, size=6, replace=False)]
    if scen == "stuck":
        return [s for kind in (rng.choice(["stuck_rail", "stuck_value"]) for _ in range(3)) for s in random_specs(rng, 1, str(kind), 1, channels=usable)]
    if scen in ("drift", "spike"):
        return random_specs(rng, 1, scen, 3, channels=usable)
    raise ValueError(scen)


def impute_window(win, L, labels, mode, mean_c):
    """Gupta-style fixed imputation of the KNOWN holes (labels.mask): zero | mean (train channel mean) | ffill (last good sample)."""
    M = labels.mask
    if not M.any():
        return win
    X = L.flatten(win)
    Y = X.copy()
    if mode == "zero":
        Y[M] = 0.0
    elif mode == "mean":
        Y = np.where(M, mean_c[None, :], X)
    elif mode == "ffill":
        for c in np.flatnonzero(M.any(axis=0)):
            last = 0.0
            for i in range(X.shape[0]):
                if M[i, c]:
                    Y[i, c] = last
                else:
                    last = X[i, c]
    else:
        raise ValueError(mode)
    return L.write_back(win, Y, M)


class Mitigators:
    """Everything fitted on TRAIN windows (detector statistics, virtual sensors, physics fallback) + val_A (detector thresholds)."""

    def __init__(self, ctx, train, cal, tiers=None, router_kw=None):
        self.ctx = ctx
        w0 = train.window(0)
        self.L = L = WindowLayout(w0)
        t0 = time.time()
        self.Xtr = np.stack([L.flatten(train.window(i)) for i in range(len(train))])
        Xcal = np.stack([L.flatten(cal.window(i)) for i in range(len(cal))])
        self.det = FaultDetector().fit(self.Xtr, Xcal)
        self.bank = fit_virtual_sensors(self.Xtr)
        feats = np.stack([LinearFluxFallback.features(x) for x in self.Xtr])
        self.fb = fit_physics_fallback(feats, yt_wb(ctx, train))
        self.mean_c = np.nanmean(self.Xtr.reshape(-1, L.n_channels), axis=0)
        self.usable = np.flatnonzero(~self.det.absent)
        self.tiers = tiers
        self.router_kw = router_kw or {}
        self.router = RouterV2(tiers, L, self.bank, self.fb, **self.router_kw)
        self.fit_seconds = time.time() - t0

    def brain(self, gauge, oracle=False):
        return SmartAdaptiveMitigationBrain(self.tiers, gauge, detector=None if oracle else self.det, bank=self.bank,
                                            fallback=self.fb, router_kw=self.router_kw, oracle=oracle)

    def prepare(self, win, labels, oracle):
        """detect (or oracle) -> decide -> repair. Returns (repaired window, health (8,), route, Detection, Decision, psi_fallback|None)."""
        det = detection_from_labels(labels) if oracle else self.det.detect_window(win, self.L)
        if oracle and (det.absent is None or det.absent.size != self.L.n_channels):
            det.absent = np.zeros(self.L.n_channels, bool)
        dec = self.router.decide(det, self.L)
        rep = self.router.apply(win, det, dec, self.L)
        psi = None
        if dec.use_fallback:
            avail = (~det.flagged | dec.transient | dec.virtual) & ~dec.lost & ~det.absent
            psi = self.fb.predict(self.fb.features(self.L.flatten(rep)), avail)
            dec.route = "physics_fallback"
        return rep, dec.branch_mask, dec.route, det, dec, psi


def detection_stats(det, labels):
    f, g = det.flagged & ~det.absent, labels.faulty
    return dict(tp=int((f & g).sum()), fn=int((~f & g).sum()), fp=int((f & ~g).sum()),
                iso_top1=bool(g.any() and g[int(np.argmax(det.score))]), has_fault=bool(g.any()), any_flag=bool(f.any()))


def run_variants(ctx, mit, ws, truths, scen, seed, variants, channel_k=3, tag=""):
    """Damage every window of `ws` with scenario `scen` and score the requested variants.
    variants subset of: none known_mask imp_zero imp_mean imp_ffill adaptive oracle.
    Returns (per-variant arrays, per-window meta: route/difficulty/detection stats for adaptive & oracle)."""
    L = mit.L
    rng = np.random.default_rng(seed)
    N = len(ws)
    wins_by_v = {v: [] for v in variants}
    health = {v: np.ones((N, 8), np.float32) for v in variants}
    fb_psi = {v: {} for v in ("adaptive", "oracle") if v in variants}
    meta = {v: dict(route=[], feats=[], det=[]) for v in ("adaptive", "oracle") if v in variants}
    for i in range(N):
        win = ws.window(i)
        specs = make_faults(scen, L, rng, mit.usable, channel_k)
        dwin, lab = inject_window(win, L, specs) if specs else (win, inject_window(win, L, [])[1])
        for v in variants:
            if v == "none":
                wins_by_v[v].append(dwin)
            elif v == "known_mask":
                wins_by_v[v].append(dwin)
                health[v][i] = L.branch_mask(lab.faulty, 0.5)
            elif v.startswith("imp_"):
                wins_by_v[v].append(impute_window(dwin, L, lab, v[4:], mit.mean_c))
            else:
                rep, h, route, det, dec, psi = mit.prepare(dwin, lab, oracle=(v == "oracle"))
                wins_by_v[v].append(rep)
                health[v][i] = h
                if psi is not None:
                    fb_psi[v][i] = psi
                meta[v]["route"].append(route)
                meta[v]["feats"].append(difficulty_features(det, dec))
                meta[v]["det"].append(detection_stats(det, lab))
    res = {}
    for v in variants:
        yp = ctx.net.predict(ws.put(wins_by_v[v]), health[v])
        for i, psi in fb_psi.get(v, {}).items():
            yp[i] = psi
        res[v] = score_pred(ctx, ws, truths, yp)
        res[v]["shot"] = ws.sid
    return res, meta


def pick_set(ctx, name, n, seed=0):
    """n windows spread evenly over the shots of a window set (round-robin by shot, fixed order)."""
    ws, trs = ctx.sets[name], ctx.truths[name]
    rng = np.random.default_rng(seed)
    by = {}
    for i, s in enumerate(ws.sid):
        by.setdefault(int(s), []).append(i)
    for s in by:
        rng.shuffle(by[s])
    idx, k = [], 0
    while len(idx) < min(n, len(ws)):
        for s in sorted(by):
            if k < len(by[s]) and len(idx) < n:
                idx.append(by[s][k])
        k += 1
    idx = np.array(sorted(idx))
    return ws.subset(idx), [trs[i] for i in idx], idx


def split_val_by_shot(ctx):
    """val_A (detector thresholds, router / imputation choice) and val_B (conformal calibration): disjoint SHOTS."""
    ws, trs = ctx.sets["val"], ctx.truths["val"]
    shots = np.unique(ws.sid)
    A = set(shots[::2].tolist())
    ia = [i for i, s in enumerate(ws.sid) if int(s) in A]
    ib = [i for i, s in enumerate(ws.sid) if int(s) not in A]
    return (ws.subset(ia), [trs[i] for i in ia]), (ws.subset(ib), [trs[i] for i in ib])


def tiers_from_criticality(path, cuts=(1.0, 0.25)):
    """Module 3 -> Module 4: tier 1 if losing the branch (group drop-one, failure-aware model told) costs >= cuts[0] cm of boundary
    error, tier 2 if >= cuts[1], else tier 3. None (-> DEFAULT_TIERS) when criticality.json is absent."""
    if not os.path.exists(path):
        return None
    d = json.load(open(path))
    dl = d["group"]["drop_one"]
    return {b: (1 if v is not None and v >= cuts[0] else 2 if v is not None and v >= cuts[1] else 3) for b, v in enumerate(dl)}


def clearance_alarm(a, delta=None):
    """Alarm decisions from the EFIT-mode clearances: true = EFIT clearance < 5 cm (PROXY wall); predicted = clearance - delta < 5 cm."""
    d_true, d_pred = a["d_wall_true_cm"], a["d_wall_pred_cm"]
    dl = 0.0 if delta is None else np.asarray(delta, float)
    pred = (d_pred - dl) < SAFETY_CLEARANCE_CM
    true = d_true < SAFETY_CLEARANCE_CM
    ok = np.isfinite(d_true) & np.isfinite(d_pred)
    return dict(n=int(ok.sum()), n_true_alarm=int((true & ok).sum()), false_alarm_rate=float((pred & ~true & ok).sum() / max(1, (~true & ok).sum())),
                missed_alarm_rate=float((~pred & true & ok).sum() / max(1, (true & ok).sum())),
                false_alarms=int((pred & ~true & ok).sum()), missed=int((~pred & true & ok).sum()))


def _jf(o):
    if isinstance(o, dict):
        return {str(k): _jf(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)):
        return [_jf(v) for v in o]
    if isinstance(o, np.ndarray):
        return _jf(o.tolist())
    if isinstance(o, (np.floating, float)):
        return None if not math.isfinite(float(o)) else float(o)
    if isinstance(o, np.integer):
        return int(o)
    if isinstance(o, np.bool_):
        return bool(o)
    return o


def run_mitigation(ctx, out, n_val_a=150, n_val_b=300, n_test=300, n_train=800, n_lat=120, channel_k=3, seed=0, tiers_json="",
                   n_boot=1000, **kw):
    """Module 4 study on the real network. Fit on TRAIN shots, choose the imputation / calibrate the detector on val_A shots,
    calibrate the conformal gauge on val_B shots, report on TEST shots (disjoint shots at every step)."""
    n_val_a, n_val_b, n_test, n_train, n_lat, channel_k, seed = (int(v) for v in (n_val_a, n_val_b, n_test, n_train, n_lat, channel_k, seed))
    t0 = time.time()
    log = lambda *a: print(t(), *a, flush=True)  # noqa: E731
    tr_n = min(n_train, len(ctx.sets["train"]))
    train = ctx.sets["train"].subset(np.linspace(0, len(ctx.sets["train"]) - 1, tr_n).astype(int))
    (vaA, trA), (vaB, trB) = split_val_by_shot(ctx)
    cal, cal_t, _ = _sub(vaA, trA, n_val_a, seed)
    vb, vb_t, _ = _sub(vaB, trB, n_val_b, seed + 1)
    te, te_t, _ = pick_set(ctx, "test", n_test, seed + 2)
    sets_info = dict(train=dict(n=len(train), shots=int(len(np.unique(train.sid)))), val_A=dict(n=len(cal), shots=int(len(np.unique(cal.sid)))),
                     val_B=dict(n=len(vb), shots=int(len(np.unique(vb.sid)))), test=dict(n=len(te), shots=int(len(np.unique(te.sid)))))
    assert not (set(np.unique(train.sid)) & set(np.unique(te.sid))) and not (set(np.unique(cal.sid)) & set(np.unique(vb.sid)))
    log("sets", sets_info)
    tiers = tiers_from_criticality(tiers_json) if tiers_json else None
    mit = Mitigators(ctx, train, cal, tiers=({int(k): v for k, v in tiers.items()} if tiers else None))
    log(f"mitigators fitted in {mit.fit_seconds:.1f}s; tiers={tiers or 'DEFAULT_TIERS'}; usable channels {len(mit.usable)}/{mit.L.n_channels}")
    imp = ["imp_zero", "imp_mean", "imp_ffill"]

    # ---- 1. imputation choice on val_A (the "best fixed imputation" row never sees the test shots)
    best_imp, val_imp = {}, {}
    for si, scen in enumerate(SCENARIOS):
        r, _ = run_variants(ctx, mit, cal, trA, scen, seed + 100 + si, imp, channel_k)
        val_imp[scen] = {v: float(np.nanmean(r[v]["boundary_mean_cm"])) for v in imp}
        best_imp[scen] = min(imp, key=lambda v: val_imp[scen][v])
        log(f"val_A {scen}: imputation boundary cm {val_imp[scen]} -> {best_imp[scen]}")

    # ---- 2. conformal calibration on val_B: adaptive pipeline, all scenarios
    cal_err, cal_feat, cal_route, cal_scen = [], [], [], []
    for si, scen in enumerate(SCENARIOS):
        r, m = run_variants(ctx, mit, vb, trB, scen, seed + 200 + si, ["adaptive"], channel_k)
        e = r["adaptive"]["boundary_hausdorff_cm"]
        ok = np.isfinite(e)
        cal_err += e[ok].tolist()
        cal_feat += [f for f, o in zip(m["adaptive"]["feats"], ok) if o]
        cal_route += [x for x, o in zip(m["adaptive"]["route"], ok) if o]
        cal_scen += [scen] * int(ok.sum())
        log(f"val_B {scen}: n={int(ok.sum())} mean hausdorff {np.nanmean(e):.2f} cm, routes {dict(zip(*np.unique(m['adaptive']['route'], return_counts=True)))}")
    g_global = ConformalGauge(alpha=ALPHA)
    by_route = {}
    for e, r_ in zip(cal_err, cal_route):
        by_route.setdefault(r_, []).append(e)
    g_global.calibrate(by_route)
    g_global.global_delta_all = float(np.sort(cal_err)[min(len(cal_err), math.ceil((len(cal_err) + 1) * (1 - ALPHA))) - 1])
    g_adapt = AdaptiveConformalGauge(alpha=ALPHA)
    g_adapt.calibrate(by_route)
    g_adapt.calibrate_adaptive(np.array(cal_err), np.array(cal_feat))
    log("conformal: per-route delta (cm)", {k: round(v, 2) for k, v in g_global.delta.items()}, "| marginal", round(g_global.global_delta_all, 2),
        "| adaptive q", round(g_adapt.q, 2))

    # ---- 3. test: every scenario x variant
    variants = ["none", "known_mask"] + imp + ["adaptive", "oracle"]
    cells, det_rows, route_use, conf_rows, flag_rows = {}, {}, {}, {}, {}
    test_arrays = {}
    for si, scen in enumerate(SCENARIOS):
        r, m = run_variants(ctx, mit, te, te_t, scen, seed + 300 + si, variants, channel_k)
        r["best_imp"] = r[best_imp[scen]]
        cells[scen] = {}
        for v in ["none", "known_mask", "best_imp", "adaptive", "oracle"] + imp:
            a = r[v]
            cells[scen][v] = shot_bootstrap(te.sid, a["boundary_mean_cm"], a["rmse"], ctx.sd, n_boot, seed,
                                            extra=dict(hausdorff_cm=a["boundary_hausdorff_cm"], axis_shift_cm=a["axis_shift_cm"],
                                                       xpoint_err_cm=a["xpoint_err_cm"]))
            cells[scen][v]["alarms_point"] = clearance_alarm(a)
        cells[scen]["best_imp"]["chosen"] = best_imp[scen]
        for v in ("adaptive", "oracle"):
            ds = m[v]["det"]
            tp, fn, fp = (sum(d[k] for d in ds) for k in ("tp", "fn", "fp"))
            hf = [d for d in ds if d["has_fault"]]
            det_rows.setdefault(scen, {})[v] = dict(
                recall=tp / max(1, tp + fn), precision=tp / max(1, tp + fp), tp=tp, fn=fn, fp=fp,
                isolation_top1=float(np.mean([d["iso_top1"] for d in hf])) if hf else None,
                window_flag_rate=float(np.mean([d["any_flag"] for d in ds])))
            route_use.setdefault(scen, {})[v] = {k: int(c) for k, c in zip(*np.unique(m[v]["route"], return_counts=True))}
        # confidence for the adaptive output
        a = r["adaptive"]
        feats = np.array(m["adaptive"]["feats"])
        routes = m["adaptive"]["route"]
        d_glob = np.array([g_global.delta_for(x) for x in routes])
        d_marg = np.full(len(routes), g_global.global_delta_all)
        d_adap = np.array([g_adapt.delta_for(x, f) for x, f in zip(routes, feats)])
        err = a["boundary_hausdorff_cm"]
        ok = np.isfinite(err)
        flags = [confidence_flag(c, d, valid=bool(o)) for c, d, o in zip(a["d_wall_pred_cm"], d_adap, ok)]
        conf_rows[scen] = dict(
            coverage_marginal=empirical_coverage(d_marg[ok], err[ok]), coverage_route_global=empirical_coverage(d_glob[ok], err[ok]),
            coverage_adaptive=empirical_coverage(d_adap[ok], err[ok]), mean_delta_marginal=float(d_marg.mean()),
            mean_delta_route_global=float(d_glob.mean()), mean_delta_adaptive=float(d_adap.mean()),
            n=int(ok.sum()), alarms_with_delta=clearance_alarm(a, d_adap), alarms_point=clearance_alarm(a))
        flag_rows[scen] = {k: int(sum(f == k for f in flags)) for k in ("HIGH", "LOW", "ABSTAIN")}
        test_arrays[scen] = dict(err=err, d_glob=d_glob, d_adap=d_adap, d_marg=d_marg, routes=np.array(routes))
        log(f"test {scen}: " + " | ".join(f"{v} {cells[scen][v]['boundary_cm']:.2f}cm/{cells[scen][v]['nrmse']:.3f}" for v in
                                         ("none", "best_imp", "adaptive", "oracle")) + f" | cov(adapt) {conf_rows[scen]['coverage_adaptive']:.2f}")
    allerr = np.concatenate([x["err"] for x in test_arrays.values()])
    alld = {k: np.concatenate([x[k] for x in test_arrays.values()]) for k in ("d_glob", "d_adap", "d_marg")}
    ok = np.isfinite(allerr)
    cov_all = {k: empirical_coverage(v[ok], allerr[ok]) for k, v in alld.items()}
    # coverage by route / by difficulty tercile (the adaptive gauge should narrow easy windows)
    routes_all = np.concatenate([x["routes"] for x in test_arrays.values()])
    cov_route = {r_: dict(n=int(((routes_all == r_) & ok).sum()), global_=empirical_coverage(alld["d_glob"][(routes_all == r_) & ok], allerr[(routes_all == r_) & ok]),
                          adaptive=empirical_coverage(alld["d_adap"][(routes_all == r_) & ok], allerr[(routes_all == r_) & ok]))
                 for r_ in np.unique(routes_all) if ((routes_all == r_) & ok).sum() > 0}

    # ---- 4. latency with the real network (CPU, batch 1), per-window brain.process
    lat = latency_study(ctx, mit, g_adapt, te, n_lat, seed)
    log("latency", {k: round(v["p50"], 2) for k, v in lat["summary"].items()})

    res = dict(model=ctx.tag, device=CPU_NOTE % (ctx.torch_version, ctx.torch_threads), sets=sets_info, dropped_no_efit_lcfs=ctx.dropped,
               alpha=ALPHA, scenarios=list(SCENARIOS), table_columns=TABLE_COLUMNS, channel_k=channel_k,
               scenario_definitions=make_faults.__doc__, imputation_choice_val_A=dict(val_boundary_cm=val_imp, chosen=best_imp),
               tiers=tiers or "DEFAULT_TIERS", cells=cells, detection=det_rows, routes=route_use, confidence=conf_rows,
               confidence_flags=flag_rows, coverage_all_scenarios=cov_all, coverage_by_route=cov_route,
               conformal=dict(per_route_delta_cm=g_global.delta, marginal_delta_cm=g_global.global_delta_all, adaptive_q=g_adapt.q,
                              n_calibration=len(cal_err), calibration_error="boundary_hausdorff_cm (EFIT mode), adaptive pipeline outputs",
                              calibration_scenarios={s: cal_scen.count(s) for s in SCENARIOS}),
               latency=lat, wall_note="wall = PROXY (EFIT grid box, no vessel data): alarm rates are a protocol demonstration",
               runtime_s=time.time() - t0, fit_seconds=mit.fit_seconds)
    path = os.path.join(out, "mitigation.json")
    json.dump(_jf(res), open(path, "w"), indent=1)
    np.savez_compressed(os.path.join(ctx.cache_dir, "m4_test_arrays.npz"), **{f"{s}__{k}": v for s, d in test_arrays.items() for k, v in d.items()})
    log("wrote", path, f"({res['runtime_s']:.0f}s)")
    return res


def _sub(ws, trs, n, seed):
    rng = np.random.default_rng(seed)
    idx = np.sort(rng.permutation(len(ws))[:n]) if n < len(ws) else np.arange(len(ws))
    return ws.subset(idx), [trs[i] for i in idx], idx


def latency_study(ctx, mit, gauge, ws, n, seed):
    """Per-window brain.process (detector, router, mitigation, REAL network forward at batch 1 on CPU, gauge) over a scenario mix."""
    rng = np.random.default_rng(seed + 7)
    brain = mit.brain(gauge)
    L = mit.L
    rows, net_only, per_scen = [], [], {}

    def model_fn(window, mask):
        x = ws.put([window])
        t0 = time.perf_counter()
        y = ctx.net.predict(x, np.asarray(mask, np.float32)[None])[0]
        net_only.append((time.perf_counter() - t0) * 1e3)
        return y
    scens = list(SCENARIOS)
    for k in range(min(n, len(ws))):
        scen = scens[k % len(scens)]
        win = ws.window(k)
        specs = make_faults(scen, L, rng, mit.usable)
        dwin, lab = inject_window(win, L, specs) if specs else (win, inject_window(win, L, [])[1])
        r = brain.process(dwin, model_fn)
        if k < 3:
            continue   # first calls warm the allocator
        rows.append(r["latencies_ms"])
        per_scen.setdefault(scen, []).append(r["latencies_ms"])
    return dict(summary=summarise_latencies(rows), per_scenario={s: summarise_latencies(v) for s, v in per_scen.items()},
                network_only_batch1=dict(mean=float(np.mean(net_only)), p50=float(np.percentile(net_only, 50)),
                                         p95=float(np.percentile(net_only, 95)), p99=float(np.percentile(net_only, 99)), n=len(net_only)),
                device=CPU_NOTE % (ctx.torch_version, ctx.torch_threads),
                note="'model' = real network forward on one window (batch 1) including numpy->torch conversion; the gauge stage uses the "
                     "35 % heuristic contour of physics_evaluation (cheap, NOT the EFIT-mode score used for the accuracy tables); "
                     "the physics fallback route replaces the network forward by a linear solve.")


def main():
    print(__doc__)
    print("Run: python run_module34_local.py --model mlp_fa_v3 --data-root <dir with random/*.zarr> --work-dir <scratch> "
          "--stage all --pylib <numpy<2 + zarr2 dirs>")


if __name__ == "__main__":
    main()


## 1. Data Preparation (Downloads the exact test shots)
Uses existing `out/random/selection.json` and `out/tar_index.json` to download the identical test data without re-indexing.


In [ ]:
run(f"cd /kaggle/working && TM_PREP_SPLITS=val,test python -u pipeline.py prep --split {DATA_SPLIT}")


## 2. Clean Gate Sanity Check
Runs inference on the clean test data to confirm the model reproduces the baseline NRMSE (~0.1419) before any corruption is tested.


In [ ]:
run(f"cd /kaggle/working && TM_ROB_CLEAN_ONLY=1 python -u robustness.py eval-robust --split {DATA_SPLIT}")


## 3. Generate Corruption Sweep Manifest
Builds the 63 corruption tasks across the 6 failure scenarios and 3 imputation modes.


In [ ]:
run(f"cd /kaggle/working && python -u robustness.py corrupt --split {DATA_SPLIT}")


## 4. Run the Full Robustness Sweep (~15-20 min)
Iterates through all 63 tasks on the test split. Metrics are saved incrementally to `out/random/robustness/robust_metrics.json`.


In [ ]:
run(f"cd /kaggle/working && python -u robustness.py eval-robust --split {DATA_SPLIT}")


## 5. Generate Report & Degradation Curves
Computes Robustness Scores (RS), degradation curves, signal importance bars, and spatial error maps.


In [ ]:
run(f"cd /kaggle/working && python -u robustness.py report --split {DATA_SPLIT}")


## 5b. Module 3 and 4 studies on the trained model
Ablates each sensor branch (criticality ranking + minimal sensor set) and runs the adaptive mitigation brain with a conformal gauge calibrated on validation windows. Writes `physics/criticality.json`, `physics/mitigation.json` and their reports next to the sweep results (~10-15 min on a T4).

In [ ]:
run(f"cd /kaggle/working && python -u physics_study.py all --split {DATA_SPLIT}")
run(f"cd /kaggle/working && python sensor_criticality.py --run out/{DATA_SPLIT}/robustness")


## 6. View Results Inline & Package Artifacts
Prints `robustness.md`, renders the figures, and creates `robustness_results.zip` in `/kaggle/working` for one-click download.


In [ ]:
from IPython.display import Image, display, Markdown

rdir = f"/kaggle/working/out/{DATA_SPLIT}/robustness"
rep_file = os.path.join(rdir, "robustness.md")

if os.path.exists(rep_file):
    print("=" * 60)
    print("ROBUSTNESS STUDY REPORT")
    print("=" * 60)
    display(Markdown(open(rep_file, encoding="utf-8").read()))

fig_dir = os.path.join(rdir, "figures")
if os.path.isdir(fig_dir):
    all_figs = [
        "robustness_psi_reconstruction.png",
        "robustness_criticality.png",
        "robustness_mitigation.png",
        "degradation_curves.png",
        "signal_importance.png",
        "spatial_error_clean.png",
    ]
    for f in all_figs:
        p = os.path.join(fig_dir, f)
        if os.path.exists(p):
            print(f"\nFigure: {f}")
            display(Image(filename=p))

# Zip existing results safely for easy download from Kaggle output section
cands = [f"out/{DATA_SPLIT}/robustness", f"out/{DATA_SPLIT}/figures", f"out/{DATA_SPLIT}/results", f"out/{DATA_SPLIT}/selection.json", f"out/{DATA_SPLIT}/model"]
targets = [p for p in cands if os.path.exists(f"/kaggle/working/{p}")]
run(f"cd /kaggle/working && zip -q -r robustness_results.zip {' '.join(targets)}")
print("\n✓ All robustness artifacts packaged to /kaggle/working/robustness_results.zip")
